# Performance Benchmarking: HEIF/GeoHEIF vs COG/GeoTIFF

## Overview
Pure C++ implementation for benchmarking HEIF/GeoHEIF against COG/GeoTIFF formats using xeus-cpp.

## Metrics Evaluated:
- **Speed**: Read/write time, metadata retrieval time
- **Data Size**: File size, metadata byte size
- **I/O**: Actual bytes read from disk (measured via system calls)
- **Resource Usage**: CPU usage, memory usage (via /proc filesystem)
- **Cloud Optimization**: First-byte retrieval efficiency, header size
- **Spatial Operations**: Bounding box subset performance

## Requirements:
- xeus-cpp (C++ Jupyter kernel)
- GDAL >= 3.9 with COG driver
- libheif >= 1.21 (compiled from source with experimental features)
- gnuplot (for visualization)

## Version: 2.0 Enhanced
- Added comprehensive validation
- Enhanced compression handling
- Progress indicators
- Detailed results summary
- First-byte-time measurements


## Environment Setup


### Mamba Environment Creation (simple example)


```bash
# Create dedicated environment (genenral steps)
mamba create -n heif_benchmark_cpp python=3.14 -y
mamba activate heif_benchmark_cpp

# Install base packages
mamba install -c conda-forge xeus-cpp jupyterlab gdal libgdal cmake ninja pkg-config -y
mamba install -c conda-forge gnuplot -y

# Compile and install libheif from source (for latest experimental features)
cd /tmp
git clone https://github.com/strukturag/libheif.git
cd libheif
git checkout main  # or specific version >= 1.21

# Build with experimental features enabled
mkdir build && cd build
cmake .. \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_INSTALL_PREFIX=$CONDA_PREFIX \
  -DWITH_EXAMPLES=ON \
  -DWITH_LIBDE265=ON \
  -DWITH_X265=ON \
  -DWITH_AOM=ON \
  -DENABLE_PLUGIN_LOADING=ON

make -j$(nproc)
make install

# Verify installations
gdalinfo --version
gdalinfo --formats | grep -E "COG|HEIF"
pkg-config --modversion libheif

# Start Jupyter
jupyter lab
```

### Mamba Environment Creation Example (macOS, intel, zsh)

```zsh

# Create a new environment with all dependencies
mamba create -n env4cppgdal python=3.14 -y

mamba activate env4cppgdal

mamba install -n env4cppgdal -c conda-forge jupyterlab clangdev clang_osx-64 clangxx_osx-64 xeus-cpp cmake cxx-compiler cmake make gxx_osx-64 gcc_osx-64 -y

# The following is for libheif compiling from source
mamba install pkg-config make libtool automake  yasm openjph x264 openh264 dav1d-dev gdk-pixbuf -y

# downgrade svt-av1 to version 3 due to a compiling error with libheif
# SvtAv1PredStructure enum removed from API in 4.0.1
mamba install dav1d svt-av1=3  rav1e openjpeg openjph x264 openh264 dav1d-dev -y

mamba install gdk-pixbuf zlib glib

# compile and install kvazaar
# under ~/wksp/cpp/src
$ git clone https://github.com/ultravideo/kvazaar.git
$ cd kvazaar
$ ./autogen.sh
$ ./configure --prefix=$CONDA_PREFIX
$ make clean
$ make
$ make install
$ kvazaar --version

# compile and install uvg266
# under ~/wksp/cpp/src
$ git clone https://github.com/ultravideo/uvg266.git
$ cd uvg266
$ rm -r build
$ mkdir build 
$ cd build
$ cmake .. -DCMAKE_INSTALL_PREFIX=$CONDA_PREFIX
$ make -j$(sysctl -n hw.ncpu)
$ make install
$ uvg266 --version

# compile and install vvdec
# under ~/wksp/cpp/src
$ git clone https://github.com/fraunhoferhhi/vvdec.git
$ cd vvdec
$ mkdir build
$ cd build
## edit ../CMakeLists.txt by adding the following on CXXFLAGS and CFLAGS
# Use nano to edit
# Search for Wno-error=pedantic
# IN 
# add_compile_options( "$<$<OR:$<CXX_COMPILER_ID:Clang>,$<CXX_COMPILER_ID:AppleClang>>:-Wall;-$
# add_compile_options( "$<$<CXX_COMPILER_ID:GNU>:-Wall;-Wno-unused-function;-Wno-sign-compare;$
# add_compile_options( "$<$<CXX_COMPILER_ID:MSVC>:/W4;/wd4100;/wd4127;/wd4244;/wd4245;/wd4389;$
 
# Add ;-Wno-error=maybe-uninitialized

cmake .. -DCMAKE_INSTALL_PREFIX=$CONDA_PREFIX -DVVDEC_INSTALL_VVDECAPP=ON
make -j$(sysctl -n hw.ncpu)
make install 
vvdecapp --version


# compile and install vvenc
# under ~/wksp/cpp/src
$ git clone https://github.com/fraunhoferhhi/vvenc.git
$ cd vvenc
$ mkdir build
$ cd build
## edit ../CMakeLists.txt by adding the following on CXXFLAGS and CFLAGS
# Use nano to edit
# Search for add_compile_options
# IN 
# add_compile_options( "$<$<OR:$<CXX_COMPILER_ID:Clang>,$<CXX_COMPILER_ID:AppleClang>>:-Wall;-$
# add_compile_options( "$<$<CXX_COMPILER_ID:GNU>:-Wall;-Wno-unused-function;-Wno-sign-compare;$
# add_compile_options( "$<$<CXX_COMPILER_ID:MSVC>:/W4;/wd4100;/wd4127;/d4244;/wd4245;/wd4389;$
 
# Add ;-Wno-error=maybe-uninitialized

# Add ;-Wno-error=array-bounds=
 
$ cmake .. -DCMAKE_INSTALL_PREFIX=$CONDA_PREFIX -DVVENC_INSTALL_VVENCAPP=ON
$ make -j$(sysctl -n hw.ncpu)
$ make install 
$ vvencapp --version

# add brotli for unci
mamba install brotli

mamba install doxygen sdl2 

# to support geotiff
# makdir $CONDA_PREFIX/include/geotiff
# cp $CONDA_PREFIX/include/geotiff.h $CONDA_PREFIX/include/geotiff/
# cp $CONDA_PREFIX/include/xtiffio.h $CONDA_PREFIX/include/geotiff/
# cp $CONDA_PREFIX/include/geotiffio.h $CONDA_PREFIX/include/geotiff/
# cp $CONDA_PREFIX/include/geo_normalize.h $CONDA_PREFIX/include/geotiff/

# compile libheif from source
# under ~/wksp/cpp/src
git clone https://github.com/strukturag/libheif.git
cd libheif
mkdir build
cd build

cmake  --preset=release .. -DCMAKE_PREFIX_PATH=$CONDA_PREFIX -DCMAKE_INSTALL_PREFIX=$CONDA_PREFIX -DWITH_UNCOMPRESSED_CODEC=ON -DWITH_HEADER_COMPRESSION=ON -DENABLE_PLUGIN_LOADING=ON -DWITH_LIBDE265=ON -DWITH_X265=ON -DWITH_AOM_DECODER=ON -DWITH_JPEG_ENCODER=ON -DWITH_JPEG_DECODER=ON -DWITH_AOM_ENCODER=ON -DWITH_KVAZAAR=ON -DWITH_UVG266=ON -DWITH_OpenJPEG_DECODER=ON -DWITH_OPENJPH_ENCODER=ON -DWITH_OpenJPEG_ENCODER=ON  -DWITH_RAV1E_PLUGIN=ON -DWITH_DAV1D_PLUGIN=ON -DWITH_SvtEnc=ON -DWITH_OpenH264_DECODER_PLUGIN=ON -DWITH_GEOTIFF=ON -DWITH_VVDEC_PLUGIN=ON -DWITH_VVENC_PLUGIN=ON -DENABLE_EXPERIMENTAL_FEATURES=ON

make
make install
heif-enc --version
heif-enc --list-encoders

```

# Setup - all

## Dynamic Library Loading with dlopen

In [ ]:
// Test and check c++ compiler and build system

#include <iostream>
#include <string>

std::cout << "xeus-cpp kernel is working!" << std::endl;
std::cout << "C++ standard: " << __cplusplus << std::endl;

In [ ]:
#include <dlfcn.h>
#include <iostream>
#include <string>
#include <vector>
#include <cstdlib>
#include <sys/utsname.h>

// Platform detection and library extension configuration
namespace Platform {

enum class OS {
    Linux,
    MacOS,
    Unknown
};

enum class Architecture {
    x86_64,
    ARM64,
    ARM,
    Unknown
};

struct SystemInfo {
    OS os;
    Architecture arch;
    std::string os_name;
    std::string arch_name;
    std::string lib_extension;
    std::string lib_prefix;
};

// Detect operating system
OS detectOS() {
#ifdef __linux__
    return OS::Linux;
#elif defined(__APPLE__) && defined(__MACH__)
    return OS::MacOS;
#else
    return OS::Unknown;
#endif
}

// Detect architecture
Architecture detectArchitecture() {
#if defined(__x86_64__) || defined(_M_X64)
    return Architecture::x86_64;
#elif defined(__aarch64__) || defined(_M_ARM64)
    return Architecture::ARM64;
#elif defined(__arm__) || defined(_M_ARM)
    return Architecture::ARM;
#else
    return Architecture::Unknown;
#endif
}

// Get detailed system information using uname
SystemInfo getSystemInfo() {
    SystemInfo info;
    info.os = detectOS();
    info.arch = detectArchitecture();
    
    // Use uname for additional system information
    struct utsname uts;
    if (uname(&uts) == 0) {
        info.os_name = uts.sysname;
        info.arch_name = uts.machine;
    }
    
    // Set library extension and prefix based on OS
    switch (info.os) {
        case OS::Linux:
            info.lib_extension = ".so";
            info.lib_prefix = "lib";
            break;
        case OS::MacOS:
            info.lib_extension = ".dylib";
            info.lib_prefix = "lib";
            break;
        default:
            info.lib_extension = ".so";  // Default to Linux
            info.lib_prefix = "lib";
            break;
    }
    
    return info;
}

// Print system information
void printSystemInfo(const SystemInfo& info) {
    std::cout << "System Information:" << std::endl;
    std::cout << "  OS: " << info.os_name << std::endl;
    std::cout << "  Architecture: " << info.arch_name << std::endl;
    std::cout << "  Library Extension: " << info.lib_extension << std::endl;
    std::cout << "  Library Prefix: " << info.lib_prefix << std::endl;
}

} // namespace Platform

// Detect system and display information
Platform::SystemInfo g_system_info = Platform::getSystemInfo();
Platform::printSystemInfo(g_system_info);


In [ ]:
// Global handles for dynamic libraries
void* gdal_handle = nullptr;
void* heif_handle = nullptr;

// Library loading utilities
namespace LibraryLoader {

// Get standard library search paths based on platform
std::vector<std::string> getLibrarySearchPaths() {
    std::vector<std::string> search_paths;
    
    // Add conda environment path if available
    const char* conda_prefix = std::getenv("CONDA_PREFIX");
    if (conda_prefix) {
        search_paths.push_back(std::string(conda_prefix) + "/lib/");
    }
    
    // Add platform-specific paths
    if (g_system_info.os == Platform::OS::Linux) {
        search_paths.push_back("/usr/lib/");
        search_paths.push_back("/usr/local/lib/");
        search_paths.push_back("/usr/lib/x86_64-linux-gnu/");
        search_paths.push_back("/usr/lib/aarch64-linux-gnu/");
        
        // Add LD_LIBRARY_PATH entries
        const char* ld_path = std::getenv("LD_LIBRARY_PATH");
        if (ld_path) {
            std::string ld_str(ld_path);
            size_t pos = 0;
            while ((pos = ld_str.find(':')) != std::string::npos) {
                std::string path = ld_str.substr(0, pos);
                if (!path.empty()) {
                    search_paths.push_back(path + "/");
                }
                ld_str.erase(0, pos + 1);
            }
            if (!ld_str.empty()) {
                search_paths.push_back(ld_str + "/");
            }
        }
    } else if (g_system_info.os == Platform::OS::MacOS) {
        search_paths.push_back("/usr/lib/");
        search_paths.push_back("/usr/local/lib/");
        search_paths.push_back("/opt/homebrew/lib/");  // Apple Silicon Homebrew
        search_paths.push_back("/usr/local/opt/");      // Intel Homebrew
        
        // Add DYLD_LIBRARY_PATH entries
        const char* dyld_path = std::getenv("DYLD_LIBRARY_PATH");
        if (dyld_path) {
            std::string dyld_str(dyld_path);
            size_t pos = 0;
            while ((pos = dyld_str.find(':')) != std::string::npos) {
                std::string path = dyld_str.substr(0, pos);
                if (!path.empty()) {
                    search_paths.push_back(path + "/");
                }
                dyld_str.erase(0, pos + 1);
            }
            if (!dyld_str.empty()) {
                search_paths.push_back(dyld_str + "/");
            }
        }
    }
    
    // Add current directory
    search_paths.push_back("./");
    
    return search_paths;
}

// Construct full library name with proper prefix and extension
std::string getLibraryName(const std::string& base_name) {
    // If it already has an extension, return as-is
    if (base_name.find(g_system_info.lib_extension) != std::string::npos) {
        return base_name;
    }
    
    // If it already has the prefix, just add extension
    if (base_name.find(g_system_info.lib_prefix) == 0) {
        return base_name + g_system_info.lib_extension;
    }
    
    // Add both prefix and extension
    return g_system_info.lib_prefix + base_name + g_system_info.lib_extension;
}

// Find library in search paths
std::string findLibraryPath(const std::string& libname) {
    std::string full_libname = getLibraryName(libname);
    std::vector<std::string> search_paths = getLibrarySearchPaths();
    
    // Try each search path
    for (const auto& path : search_paths) {
        std::string full_path = path + full_libname;
        
        // Try to open with RTLD_NOLOAD to check if it exists
        void* test_handle = dlopen(full_path.c_str(), RTLD_LAZY | RTLD_NOLOAD);
        if (test_handle) {
            dlclose(test_handle);
            return full_path;
        }
        
        // Try without RTLD_NOLOAD (actually check file existence)
        test_handle = dlopen(full_path.c_str(), RTLD_LAZY);
        if (test_handle) {
            dlclose(test_handle);
            return full_path;
        }
    }
    
    // If not found in paths, return the library name and let dlopen search system paths
    return full_libname;
}

// Load a library with proper error handling
void* loadLibrary(const std::string& libname, bool verbose = true) {
    std::string lib_path = findLibraryPath(libname);
    
    if (verbose) {
        std::cout << "Attempting to load: " << lib_path << std::endl;
    }
    
    // Clear any existing error
    dlerror();
    
    // Try to load the library
    void* handle = dlopen(lib_path.c_str(), RTLD_LAZY | RTLD_GLOBAL);
    
    if (!handle) {
        const char* error = dlerror();
        std::cerr << "Failed to load " << libname << ": " << (error ? error : "Unknown error") << std::endl;
        
        // Try alternative names
        if (g_system_info.os == Platform::OS::MacOS) {
            // On macOS, try versioned dylib names
            std::vector<std::string> alternatives = {
                libname + ".1" + g_system_info.lib_extension,
                libname + ".0" + g_system_info.lib_extension
            };
            
            for (const auto& alt : alternatives) {
                std::string alt_path = findLibraryPath(alt);
                if (verbose) {
                    std::cout << "Trying alternative: " << alt_path << std::endl;
                }
                handle = dlopen(alt_path.c_str(), RTLD_LAZY | RTLD_GLOBAL);
                if (handle) {
                    if (verbose) {
                        std::cout << "Successfully loaded: " << alt_path << std::endl;
                    }
                    return handle;
                }
            }
        }
        
        return nullptr;
    }
    
    if (verbose) {
        std::cout << "Successfully loaded: " << lib_path << std::endl;
    }
    
    return handle;
}

} // namespace LibraryLoader

std::cout << "\nLibrary loader utilities initialized." << std::endl;

In [ ]:
// Load GDAL library
bool loadGDAL() {
    if (gdal_handle) {
        std::cout << "GDAL already loaded." << std::endl;
        return true;
    }
    
    std::cout << "\nLoading GDAL library..." << std::endl;
    gdal_handle = LibraryLoader::loadLibrary("gdal", true);
    
    if (!gdal_handle) {
        std::cerr << "\n** ERROR: Failed to load GDAL library **" << std::endl;
        std::cerr << "Please ensure GDAL is installed:" << std::endl;
        if (g_system_info.os == Platform::OS::Linux) {
            std::cerr << "  mamba install -c conda-forge gdal libgdal" << std::endl;
        } else if (g_system_info.os == Platform::OS::MacOS) {
            std::cerr << "  mamba install -c conda-forge gdal libgdal" << std::endl;
            std::cerr << "  or: brew install gdal" << std::endl;
        }
        return false;
    }
    
    std::cout << "✓ GDAL library loaded successfully." << std::endl;
    return true;
}

// Load libheif library
bool loadHEIF() {
    if (heif_handle) {
        std::cout << "libheif already loaded." << std::endl;
        return true;
    }
    
    std::cout << "\nLoading libheif library..." << std::endl;
    heif_handle = LibraryLoader::loadLibrary("heif", true);
    
    if (!heif_handle) {
        std::cerr << "\n** ERROR: Failed to load libheif library **" << std::endl;
        std::cerr << "Please ensure libheif >= 1.21 is installed from source." << std::endl;
        std::cerr << "See the environment setup instructions above." << std::endl;
        return false;
    }
    
    std::cout << "✓ libheif library loaded successfully." << std::endl;
    return true;
}

// Load all required libraries
bool loadAllLibraries() {
    std::cout << "\n========================================" << std::endl;
    std::cout << "Loading Required Libraries" << std::endl;
    std::cout << "========================================" << std::endl;
    
    bool success = true;
    success &= loadGDAL();
    success &= loadHEIF();
    
    if (success) {
        std::cout << "\n✓ All libraries loaded successfully!" << std::endl;
    } else {
        std::cout << "\n✗ Some libraries failed to load!" << std::endl;
    }
    std::cout << "========================================\n" << std::endl;
    
    return success;
}

// Execute library loading
if (!loadAllLibraries()) {
    std::cerr << "CRITICAL ERROR: Required libraries could not be loaded!" << std::endl;
    std::cerr << "Please check your installation and try again." << std::endl;
}

## Include Headers (After Library Loading)

In [ ]:
// In your "Include Headers" cell - ADD THIS AT THE TOP

// Enable experimental features (libheif was compiled with these enabled)
#ifndef HEIF_ENABLE_EXPERIMENTAL_FEATURES
#define HEIF_ENABLE_EXPERIMENTAL_FEATURES 1
#endif

#include <gdal.h>
#include <gdal_priv.h>
#include <cpl_conv.h>
#include <cpl_string.h>
#include <cpl_vsi.h>
#include <ogr_spatialref.h>
#include <libheif/heif.h>
#include <libheif/heif_experimental.h>  // ADD THIS LINE
#include <libheif/heif_uncompressed.h>  // ADD THIS LINE
#include <iostream>
#include <fstream>
#include <sstream>
#include <iomanip>
#include <vector>
#include <string>
#include <cstring>
#include <chrono>
#include <map>
#include <cmath>
#include <algorithm>
#include <memory>
#include <functional>
#include <limits>
#include <thread>
#include <filesystem>
#include <sys/stat.h>
#include <sys/types.h>
#include <unistd.h>
#include <sys/resource.h>
#include <sys/time.h>
#include <fcntl.h>

// Initialize GDAL
GDALAllRegister();

std::cout << "\n========================================" << std::endl;
std::cout << "Library Version Information" << std::endl;
std::cout << "========================================" << std::endl;
std::cout << "GDAL Version: " << GDALVersionInfo("RELEASE_NAME") << std::endl;
std::cout << "libheif Version: " << heif_get_version() << std::endl;

// Verify experimental features are enabled
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
std::cout << "libheif experimental features: ✅ ENABLED" << std::endl;
std::cout << "  - heif_context_add_tiled_image: Available" << std::endl;
std::cout << "  - Tili support: Ready" << std::endl;
#else
std::cout << "libheif experimental features: ❌ NOT AVAILABLE" << std::endl;
std::cout << "  - Will use grid/unci tiling only" << std::endl;
#endif

std::cout << "========================================" << std::endl;
std::cout << "\n✓ Headers loaded and GDAL initialized successfully." << std::endl;

## Overview & Start Guide

In [ ]:
// Quick Start Guide
std::cout << R"(
╔══════════════════════════════════════════════════════════════╗
║         HEIF/GeoHEIF vs COG/GeoTIFF Benchmark Tool          ║
║                     Quick Start Guide                        ║
╚══════════════════════════════════════════════════════════════╝

STEP 1: Verify Environment
   Run all cells in order up to "Run Configuration Dialog"
   Check that all libraries loaded successfully

STEP 2: Configure Benchmark
   Execute the configuration cell
   Choose a preset or use custom configuration

STEP 3: Create Datasets
   Run the "Create Dataset Pairs" cell
   Wait for all datasets to be created

STEP 4: Run Benchmarks
   Execute the "Run Complete Benchmarks" cell
   This may take significant time depending on preset

STEP 5: View Results
   Check the CSV file in output directory
   Run gnuplot to generate visualizations
   Review the summary cell output

TROUBLESHOOTING:
   • Libraries not loading? Check conda environment
   • HEIF creation failing? Ensure libheif >= 1.21
   • COG creation failing? Verify GDAL >= 3.9
   • Slow performance? Try "Quick Test" preset first

REQUIRED FILES:
   • Source GeoTIFF: srcdata/ACT2017.tiff (or specify path)
   • Output directory will be created automatically

═══════════════════════════════════════════════════════════════
)";

## Platform-Specific Utilities

In [ ]:
namespace PlatformUtils {

// Get memory usage (cross-platform)
long getMemoryUsageKB() {
    if (g_system_info.os == Platform::OS::Linux) {
        std::ifstream status("/proc/self/status");
        std::string line;
        long vmrss = 0;
        
        while (std::getline(status, line)) {
            if (line.find("VmRSS:") == 0) {
                std::istringstream iss(line);
                std::string label;
                iss >> label >> vmrss;
                break;
            }
        }
        return vmrss;
    } else if (g_system_info.os == Platform::OS::MacOS) {
        // macOS uses rusage
        struct rusage usage;
        getrusage(RUSAGE_SELF, &usage);
        // On macOS, ru_maxrss is in bytes, convert to KB
        return usage.ru_maxrss / 1024;
    }
    return 0;
}

// Get peak memory (cross-platform)
long getPeakMemoryKB() {
    if (g_system_info.os == Platform::OS::Linux) {
        std::ifstream status("/proc/self/status");
        std::string line;
        long vmpeak = 0;
        
        while (std::getline(status, line)) {
            if (line.find("VmPeak:") == 0) {
                std::istringstream iss(line);
                std::string label;
                iss >> label >> vmpeak;
                break;
            }
        }
        return vmpeak;
    } else if (g_system_info.os == Platform::OS::MacOS) {
        struct rusage usage;
        getrusage(RUSAGE_SELF, &usage);
        return usage.ru_maxrss / 1024;
    }
    return 0;
}

// Get bytes read (Linux-specific, approximation for macOS)
long getBytesRead() {
    if (g_system_info.os == Platform::OS::Linux) {
        std::ifstream io("/proc/self/io");
        std::string line;
        long bytes_read = 0;
        
        while (std::getline(io, line)) {
            if (line.find("read_bytes:") == 0) {
                std::istringstream iss(line);
                std::string label;
                iss >> label >> bytes_read;
                break;
            }
        }
        return bytes_read;
    } else if (g_system_info.os == Platform::OS::MacOS) {
        // macOS doesn't have /proc/self/io
        // Return 0 as placeholder - actual I/O tracking requires different approach
        struct rusage usage;
        getrusage(RUSAGE_SELF, &usage);
        // ru_inblock gives block input operations
        return usage.ru_inblock * 512;  // Approximate: assume 512 bytes per block
    }
    return 0;
}

// Get number of CPU cores
int getNumCores() {
    if (g_system_info.os == Platform::OS::Linux) {
        return static_cast<int>(sysconf(_SC_NPROCESSORS_ONLN));
    } else if (g_system_info.os == Platform::OS::MacOS) {
        FILE* fp = popen("sysctl -n hw.ncpu", "r");
        if (fp) {
            int ncpu = 0;
            fscanf(fp, "%d", &ncpu);
            pclose(fp);
            return ncpu;
        }
    }
    return 1;  // Default
}

} // namespace PlatformUtils

std::cout << "Platform-specific utilities loaded." << std::endl;
std::cout << "Available CPU cores: " << PlatformUtils::getNumCores() << std::endl;

## Core Structures and Utilities

In [ ]:
// Core Structures and Utilities - CORRECTED with proper namespace
namespace BenchmarkUtils {

// Performance metrics structure
struct PerformanceMetrics {
    std::string format;
    std::string operation;  // "full_read", "subset_read", "write", "metadata_retrieval", "first_byte"
    std::string compression;
    int tile_size;
    int pyramid_levels;
    
    // File metrics
    long file_size_bytes;
    long metadata_size_bytes;
    long header_size_bytes;
    
    // Performance metrics (averaged over iterations)
    double read_time_ms;
    double write_time_ms;
    double metadata_retrieval_time_ms;
    double first_byte_time_ms;
    
    // Resource metrics
    long bytes_read_from_disk;
    double cpu_usage_percent;
    long memory_usage_kb;
    long peak_memory_kb;
    
    // Image properties
    int width;
    int height;
    int bands;
    
    // Statistics
    double time_std_dev;
    int num_iterations;
    
    PerformanceMetrics() : format(""), operation(""), compression(""),
        tile_size(0), pyramid_levels(0), file_size_bytes(0), metadata_size_bytes(0),
        header_size_bytes(0), read_time_ms(0), write_time_ms(0),
        metadata_retrieval_time_ms(0), first_byte_time_ms(0),
        bytes_read_from_disk(0), cpu_usage_percent(0),
        memory_usage_kb(0), peak_memory_kb(0),
        width(0), height(0), bands(0),
        time_std_dev(0), num_iterations(0) {}
};

// Configuration structure
struct BenchmarkConfig {
    std::vector<int> tile_sizes = {128, 256, 512};
    std::vector<int> pyramid_levels = {0, 3, 5};
    std::vector<std::string> compressions = {"NONE", "LZW", "DEFLATE", "ZSTD"};
    bool include_no_tiling = true;      // Include untiled versions
    bool include_no_pyramids = true;    // Include versions without pyramids
    int benchmark_iterations = 5;
    bool enable_cloud_optimization = true;
    bool enable_validation = true;
    bool show_progress = true;
};

// Dataset pair structure
struct DatasetPair {
    std::string cog_path;
    std::string geoheif_path;
    std::string compression;
    int tile_size;          // 0 means no tiling
    int pyramid_levels;     // 0 means no pyramids
    bool is_tiled;          // Flag to distinguish tiled vs untiled
};

} // namespace BenchmarkUtils

std::cout << "Core structures defined with no-tiling support." << std::endl;

## System Utility Functions (Cross-Platform)

In [ ]:
namespace BenchmarkUtils {

// Get file size
long getFileSize(const std::string& filename) {
    struct stat stat_buf;
    int rc = stat(filename.c_str(), &stat_buf);
    return rc == 0 ? stat_buf.st_size : -1;
}

// Use platform-specific memory functions
long getMemoryUsageKB() {
    return PlatformUtils::getMemoryUsageKB();
}

long getPeakMemoryKB() {
    return PlatformUtils::getPeakMemoryKB();
}

long getBytesRead() {
    return PlatformUtils::getBytesRead();
}

// Get CPU time
double getCPUTime() {
    struct rusage usage;
    getrusage(RUSAGE_SELF, &usage);
    return (usage.ru_utime.tv_sec + usage.ru_stime.tv_sec) + 
           (usage.ru_utime.tv_usec + usage.ru_stime.tv_usec) / 1e6;
}

// Timer class for high-precision timing
class Timer {
private:
    std::chrono::high_resolution_clock::time_point start_time;
public:
    Timer() : start_time(std::chrono::high_resolution_clock::now()) {}
    
    void reset() {
        start_time = std::chrono::high_resolution_clock::now();
    }
    
    double elapsed_ms() {
        auto end_time = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::milli>(end_time - start_time).count();
    }
};

// Calculate standard deviation
double calcStdDev(const std::vector<double>& values) {
    if (values.size() <= 1) return 0.0;
    
    double sum = 0.0;
    for (double v : values) sum += v;
    double mean = sum / values.size();
    
    double sq_sum = 0.0;
    for (double v : values) {
        sq_sum += (v - mean) * (v - mean);
    }
    
    return std::sqrt(sq_sum / (values.size() - 1));
}

// Progress bar class
class ProgressBar {
private:
    int total;
    int current;
    int bar_width;
    std::string description;
    bool enabled;
    
public:
    ProgressBar(int total, const std::string& desc = "", int width = 50, bool enabled = true) 
        : total(total), current(0), bar_width(width), description(desc), enabled(enabled) {
        if (enabled) display();
    }
    
    void update(int value = -1) {
        if (!enabled) return;
        if (value >= 0) {
            current = value;
        } else {
            current++;
        }
        display();
    }
    
    void display() {
        if (!enabled) return;
        float progress = static_cast<float>(current) / total;
        int pos = static_cast<int>(bar_width * progress);
        
        std::cout << "\r" << description << " [";
        for (int i = 0; i < bar_width; i++) {
            if (i < pos) std::cout << "=";
            else if (i == pos) std::cout << ">";
            else std::cout << " ";
        }
        std::cout << "] " << int(progress * 100.0) << "% (" 
                  << current << "/" << total << ")" << std::flush;
        
        if (current >= total) {
            std::cout << std::endl;
        }
    }
    
    void finish() {
        if (!enabled) return;
        current = total;
        display();
    }
};

} // namespace BenchmarkUtils

std::cout << "Cross-platform utility functions loaded." << std::endl;

## HEIF RAII Wrappers for Memory Safety

In [ ]:
namespace HEIFWrappers {

// RAII wrapper for heif_context
class HeifContext {
private:
    heif_context* ctx;
public:
    HeifContext() : ctx(heif_context_alloc()) {}
    ~HeifContext() { 
        if (ctx) {
            heif_context_free(ctx);
            ctx = nullptr;
        }
    }
    
    // Delete copy constructor and assignment
    HeifContext(const HeifContext&) = delete;
    HeifContext& operator=(const HeifContext&) = delete;
    
    heif_context* get() { return ctx; }
    operator heif_context*() { return ctx; }
};

// RAII wrapper for heif_image
class HeifImage {
private:
    heif_image* img;
public:
    HeifImage() : img(nullptr) {}
    explicit HeifImage(heif_image* i) : img(i) {}
    ~HeifImage() { 
        if (img) {
            heif_image_release(img);
            img = nullptr;
        }
    }
    
    // Delete copy constructor and assignment
    HeifImage(const HeifImage&) = delete;
    HeifImage& operator=(const HeifImage&) = delete;
    
    void set(heif_image* i) {
        if (img) heif_image_release(img);
        img = i;
    }
    heif_image* get() { return img; }
    heif_image** ptr() { return &img; }
    operator heif_image*() { return img; }
};

// RAII wrapper for heif_encoder
class HeifEncoder {
private:
    heif_encoder* enc;
public:
    HeifEncoder() : enc(nullptr) {}
    explicit HeifEncoder(heif_encoder* e) : enc(e) {}
    ~HeifEncoder() {
        if (enc) {
            heif_encoder_release(enc);
            enc = nullptr;
        }
    }
    
    // Delete copy constructor and assignment
    HeifEncoder(const HeifEncoder&) = delete;
    HeifEncoder& operator=(const HeifEncoder&) = delete;
    
    void set(heif_encoder* e) {
        if (enc) heif_encoder_release(enc);
        enc = e;
    }
    heif_encoder* get() { return enc; }
    heif_encoder** ptr() { return &enc; }
    operator heif_encoder*() { return enc; }
};

// RAII wrapper for heif_image_handle
class HeifImageHandle {
private:
    heif_image_handle* handle;
public:
    HeifImageHandle() : handle(nullptr) {}
    explicit HeifImageHandle(heif_image_handle* h) : handle(h) {}
    ~HeifImageHandle() {
        if (handle) {
            heif_image_handle_release(handle);
            handle = nullptr;
        }
    }
    
    // Delete copy constructor and assignment
    HeifImageHandle(const HeifImageHandle&) = delete;
    HeifImageHandle& operator=(const HeifImageHandle&) = delete;
    
    void set(heif_image_handle* h) {
        if (handle) heif_image_handle_release(handle);
        handle = h;
    }
    heif_image_handle* get() { return handle; }
    heif_image_handle** ptr() { return &handle; }
    operator heif_image_handle*() { return handle; }
};

// RAII wrapper for encoding options
class HeifEncodingOptions {
private:
    heif_encoding_options* opts;
public:
    HeifEncodingOptions() : opts(heif_encoding_options_alloc()) {}
    ~HeifEncodingOptions() {
        if (opts) {
            heif_encoding_options_free(opts);
            opts = nullptr;
        }
    }
    
    // Delete copy constructor and assignment
    HeifEncodingOptions(const HeifEncodingOptions&) = delete;
    HeifEncodingOptions& operator=(const HeifEncodingOptions&) = delete;
    
    heif_encoding_options* get() { return opts; }
    operator heif_encoding_options*() { return opts; }
};

} // namespace HEIFWrappers

std::cout << "HEIF RAII wrappers loaded." << std::endl;

## Comprehensive Testing on Setup

In [ ]:
// Component Testing and Verification
namespace Testing {

bool testPlatformDetection() {
    std::cout << "Testing platform detection...\n";
    Platform::SystemInfo info = Platform::getSystemInfo();
    bool passed = !info.os_name.empty() && !info.arch_name.empty();
    std::cout << (passed ? "✓" : "✗") << " Platform detection\n";
    return passed;
}

bool testLibraryLoading() {
    std::cout << "\nTesting library loading...\n";
    bool passed = (gdal_handle != nullptr) && (heif_handle != nullptr);
    std::cout << (passed ? "✓" : "✗") << " Libraries loaded\n";
    return passed;
}

bool testGDALFunctionality() {
    std::cout << "\nTesting GDAL functionality...\n";
    int driver_count = GDALGetDriverCount();
    bool passed = driver_count > 0;
    std::cout << (passed ? "✓" : "✗") << " GDAL drivers: " << driver_count << "\n";
    return passed;
}

bool testHEIFFunctionality() {
    std::cout << "\nTesting HEIF functionality...\n";
    HEIFWrappers::HeifContext ctx;
    bool passed = (ctx.get() != nullptr);
    std::cout << (passed ? "✓" : "✗") << " HEIF context creation\n";
    return passed;
}

bool runAllTests() {
    std::cout << "\n" << std::string(60, '=') << "\n";
    std::cout << "RUNNING COMPONENT TESTS\n";
    std::cout << std::string(60, '=') << "\n\n";
    
    bool all_passed = true;
    all_passed &= testPlatformDetection();
    all_passed &= testLibraryLoading();
    all_passed &= testGDALFunctionality();
    all_passed &= testHEIFFunctionality();
    
    std::cout << "\n" << std::string(60, '=') << "\n";
    if (all_passed) {
        std::cout << "✅ ALL TESTS PASSED\n";
    } else {
        std::cout << "❌ SOME TESTS FAILED\n";
    }
    std::cout << std::string(60, '=') << "\n";
    
    return all_passed;
}

} // namespace Testing

// Run tests
Testing::runAllTests();

## COG Creation with Cloud Optimization

In [ ]:
namespace COGCreator {

// Create Cloud-Optimized GeoTIFF - CORRECTED for untiled support
bool createCOG(const std::string& input_tiff, 
               const std::string& output_cog,
               const std::string& compression = "LZW",
               int tile_size = 256,
               int num_overview_levels = 3,
               bool verbose = true) {
    
    if (verbose) {
        std::cout << "\n[COG] Creating: " << output_cog << std::endl;
        std::cout << "  Compression: " << compression << std::endl;
        if (tile_size > 0) {
            std::cout << "  Tile size: " << tile_size << std::endl;
        } else {
            std::cout << "  Tiling: NONE (scanline/strip)" << std::endl;
        }
        std::cout << "  Overview levels: " << num_overview_levels << std::endl;
    }
    
    // Validate compression type
    std::string comp_upper = compression;
    std::transform(comp_upper.begin(), comp_upper.end(), comp_upper.begin(), ::toupper);
    
    // Check if ZSTD is available
    if (comp_upper == "ZSTD") {
        GDALDriver* gtiff_driver = GetGDALDriverManager()->GetDriverByName("GTiff");
        if (gtiff_driver) {
            const char* metadata = gtiff_driver->GetMetadataItem(GDAL_DMD_CREATIONOPTIONLIST);
            if (metadata) {
                std::string meta_str(metadata);
                if (meta_str.find("ZSTD") == std::string::npos) {
                    std::cerr << "[COG] Warning: ZSTD compression not supported by this GDAL build" << std::endl;
                    std::cerr << "[COG] Falling back to DEFLATE" << std::endl;
                    comp_upper = "DEFLATE";
                }
            }
        }
    }
    
    // Open source dataset
    GDALDataset* src_ds = static_cast<GDALDataset*>(GDALOpen(input_tiff.c_str(), GA_ReadOnly));
    if (!src_ds) {
        std::cerr << "[COG] Failed to open source: " << input_tiff << std::endl;
        return false;
    }
    
    // Choose driver based on tiling
    GDALDriver* driver;
    bool use_cog_driver = false;
    
    if (tile_size > 0 && num_overview_levels >= 0) {
        // Use COG driver for tiled versions
        driver = GetGDALDriverManager()->GetDriverByName("COG");
        if (driver) {
            use_cog_driver = true;
            if (verbose) {
                std::cout << "[COG] Using COG driver" << std::endl;
            }
        } else {
            if (verbose) {
                std::cout << "[COG] COG driver not available, using GTiff" << std::endl;
            }
            driver = GetGDALDriverManager()->GetDriverByName("GTiff");
        }
    } else {
        // Use GTiff driver for untiled versions
        driver = GetGDALDriverManager()->GetDriverByName("GTiff");
        if (verbose) {
            std::cout << "[COG] Using GTiff driver for untiled" << std::endl;
        }
    }
    
    if (!driver) {
        std::cerr << "[COG] No suitable driver available" << std::endl;
        GDALClose(src_ds);
        return false;
    }
    
    // Setup creation options
    char** papszOptions = nullptr;
    
    // Main compression
    papszOptions = CSLSetNameValue(papszOptions, "COMPRESS", comp_upper.c_str());
    
    if (use_cog_driver && tile_size > 0) {
        // ===== COG DRIVER OPTIONS =====
        papszOptions = CSLSetNameValue(papszOptions, "BLOCKSIZE", std::to_string(tile_size).c_str());
        
        // Overview behavior
        if (num_overview_levels > 0) {
            papszOptions = CSLSetNameValue(papszOptions, "OVERVIEWS", "IGNORE_EXISTING");
            papszOptions = CSLSetNameValue(papszOptions, "OVERVIEW_COUNT", std::to_string(num_overview_levels).c_str());
            papszOptions = CSLSetNameValue(papszOptions, "OVERVIEW_RESAMPLING", "NEAREST");
            papszOptions = CSLSetNameValue(papszOptions, "OVERVIEW_COMPRESS", comp_upper.c_str());
        } else {
            papszOptions = CSLSetNameValue(papszOptions, "OVERVIEWS", "NONE");
        }
        
        // Multi-threading
        papszOptions = CSLSetNameValue(papszOptions, "NUM_THREADS", "ALL_CPUS");
        
    } else {
        // ===== GTIFF DRIVER OPTIONS =====
        if (tile_size > 0) {
            // Tiled GTiff
            papszOptions = CSLSetNameValue(papszOptions, "TILED", "YES");
            papszOptions = CSLSetNameValue(papszOptions, "BLOCKXSIZE", std::to_string(tile_size).c_str());
            papszOptions = CSLSetNameValue(papszOptions, "BLOCKYSIZE", std::to_string(tile_size).c_str());
        } else {
            // Untiled GTiff (strip/scanline)
            papszOptions = CSLSetNameValue(papszOptions, "TILED", "NO");
            // Use default strip size (one scanline typically)
        }
        
        // No overviews for untiled or when using GTiff directly
        // (Overviews would be added separately via gdaladdo if needed)
    }
    
    // Compression-specific settings (common to both drivers)
    if (comp_upper == "DEFLATE") {
        papszOptions = CSLSetNameValue(papszOptions, "ZLEVEL", "6");
        papszOptions = CSLSetNameValue(papszOptions, "PREDICTOR", "2");  // Horizontal differencing
    } else if (comp_upper == "LZW") {
        papszOptions = CSLSetNameValue(papszOptions, "PREDICTOR", "2");
    } else if (comp_upper == "ZSTD") {
        papszOptions = CSLSetNameValue(papszOptions, "ZSTD_LEVEL", "9");
        papszOptions = CSLSetNameValue(papszOptions, "PREDICTOR", "2");
    } else if (comp_upper == "JPEG") {
        papszOptions = CSLSetNameValue(papszOptions, "JPEG_QUALITY", "90");
    } else if (comp_upper == "WEBP") {
        papszOptions = CSLSetNameValue(papszOptions, "WEBP_LEVEL", "90");
    }
    
    // BigTIFF handling (common)
    papszOptions = CSLSetNameValue(papszOptions, "BIGTIFF", "IF_SAFER");
    
    // Interleaving (common)
    papszOptions = CSLSetNameValue(papszOptions, "INTERLEAVE", "PIXEL");
    
    if (verbose) {
        std::cout << "[COG] Creation options:" << std::endl;
        for (int i = 0; papszOptions && papszOptions[i] != nullptr; i++) {
            std::cout << "  " << papszOptions[i] << std::endl;
        }
    }
    
    // Create output using CreateCopy
    GDALDataset* dst_ds = driver->CreateCopy(output_cog.c_str(), src_ds, FALSE, 
                                              papszOptions, nullptr, nullptr);
    
    CSLDestroy(papszOptions);
    GDALClose(src_ds);
    
    if (!dst_ds) {
        std::cerr << "[COG] Failed to create output" << std::endl;
        const char* err = CPLGetLastErrorMsg();
        if (err && strlen(err) > 0) {
            std::cerr << "[COG] Error: " << err << std::endl;
        }
        return false;
    }
    
    GDALClose(dst_ds);
    
    if (verbose) {
        long file_size = BenchmarkUtils::getFileSize(output_cog);
        std::cout << "[COG] Created successfully" << std::endl;
        std::cout << "[COG] File size: " << std::fixed << std::setprecision(2) 
                  << (file_size / 1024.0 / 1024.0) << " MB" << std::endl;
    }
    
    return true;
}

} // namespace COGCreator

std::cout << "COG creator loaded with corrected untiled support (uses GTiff driver for untiled)." << std::endl;

## Combined HEIF namespace

In [ ]:
// Complete HEIF Creator with grid, unci, and tili support (FINAL CORRECTED VERSION)
// Based on official libheif heif-enc.cc example
// NOTE: heif_experimental.h must be included in the headers cell BEFORE this namespace

namespace HEIFCreator {

#ifndef HEIF_ENABLE_EXPERIMENTAL_FEATURES
#error "HEIF_ENABLE_EXPERIMENTAL_FEATURES must be defined! Add it before including heif_experimental.h"
#endif

// Helper function to check heif_error and print message
bool checkHeifError(const heif_error& error, const std::string& operation) {
    if (error.code != heif_error_Ok) {
        std::cerr << "HEIF Error in " << operation << ": " << error.message << std::endl;
        std::cerr << "  Error code: " << error.code << std::endl;
        std::cerr << "  Subcode: " << error.subcode << std::endl;
        return false;
    }
    return true;
}

// Tiling scheme enum
enum class TilingScheme {
    Grid,      // grid images - standard HEIF tiling
    Unci,      // unci images - ISO 23001-17 uncompressed tiling
    Tili       // tili images - efficient tiling with compression
};

// Compression configuration structure
struct CompressionConfig {
    std::string name;
    TilingScheme tiling_scheme;
    heif_compression_format format;
    std::string encoder_name;
    bool lossless;
    int quality;
    heif_unci_compression unci_compression;
    uint32_t tili_fourcc;
    std::map<std::string, std::string> parameters;
};


// Get compression configuration with smart COG-to-HEIF mapping
CompressionConfig getCompressionConfig(const std::string& compression_choice) {
    CompressionConfig config;
    
    std::string comp_upper = compression_choice;
    std::transform(comp_upper.begin(), comp_upper.end(), comp_upper.begin(), ::toupper);
    
    std::cout << "  [CONFIG] Processing compression: " << comp_upper << std::endl;
    
    if (comp_upper == "NONE" || comp_upper == "UNCOMPRESSED") {
        config.name = "UNCOMPRESSED";
        config.tiling_scheme = TilingScheme::Unci;
        config.format = heif_compression_uncompressed;
        config.encoder_name = "uncompressed";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_off;
        std::cout << "  [CONFIG] Using unci with no compression" << std::endl;
        
    } else if (comp_upper == "DEFLATE") {
        config.name = "DEFLATE";
        config.tiling_scheme = TilingScheme::Unci;
        config.format = heif_compression_uncompressed;
        config.encoder_name = "uncompressed";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_deflate;
        std::cout << "  [CONFIG] Using unci with DEFLATE compression" << std::endl;
        
    } else if (comp_upper == "LZW") {
        // LZW not supported in HEIF, map to ZLIB(DEFLATE)
        config.name = "ZLIB";
        config.tiling_scheme = TilingScheme::Unci;
        config.format = heif_compression_uncompressed;
        config.encoder_name = "uncompressed";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_zlib;
        std::cout << "  [CONFIG] LZW not supported, using unci with ZLIB(DEFLATE)" << std::endl;
        
    } else if (comp_upper == "ZSTD") {
        // ZSTD in COG maps to BROTLI in HEIF (both modern, high-efficiency)
        config.name = "BROTLI";
        config.tiling_scheme = TilingScheme::Unci;
        config.format = heif_compression_uncompressed;
        config.encoder_name = "uncompressed";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_brotli;
        std::cout << "  [CONFIG] ZSTD->BROTLI: Using unci with BROTLI compression (modern equivalent)" << std::endl;
        
    } else if (comp_upper == "BROTLI") {
        config.name = "BROTLI";
        config.tiling_scheme = TilingScheme::Unci;
        config.format = heif_compression_uncompressed;
        config.encoder_name = "uncompressed";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_brotli;
        std::cout << "  [CONFIG] Using unci with BROTLI compression" << std::endl;
        
    } else if (comp_upper == "HEVC" || comp_upper == "H265" || comp_upper == "X265") {
        config.name = "HEVC-LOSSLESS";
#if HEIF_ENABLE_EXPERIMENTAL_FEATURES
        config.tiling_scheme = TilingScheme::Tili;
        std::cout << "  [CONFIG] Using tili with lossless HEVC" << std::endl;
#else
        config.tiling_scheme = TilingScheme::Grid;
        std::cout << "  [CONFIG] Using grid with lossless HEVC (tili not available)" << std::endl;
#endif
        config.format = heif_compression_HEVC;
        config.encoder_name = "x265";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_off;
        config.parameters["preset"] = "medium";
        
    } else if (comp_upper == "AV1" || comp_upper == "AOM") {
        config.name = "AV1-LOSSLESS";
#if HEIF_ENABLE_EXPERIMENTAL_FEATURES
        config.tiling_scheme = TilingScheme::Tili;
        std::cout << "  [CONFIG] Using tili with lossless AV1" << std::endl;
#else
        config.tiling_scheme = TilingScheme::Grid;
        std::cout << "  [CONFIG] Using grid with lossless AV1 (tili not available)" << std::endl;
#endif
        config.format = heif_compression_AV1;
        config.encoder_name = "aom";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_off;
        config.parameters["cpu-used"] = "4";
        
    } else if (comp_upper == "JPEG") {
        config.name = "JPEG";
        config.tiling_scheme = TilingScheme::Grid;
        config.format = heif_compression_JPEG;
        config.encoder_name = "jpeg";
        config.lossless = false;
        config.quality = 90;
        config.unci_compression = heif_unci_compression_off;
        std::cout << "  [CONFIG] Using grid with JPEG" << std::endl;
        
    } else {
        config.name = "UNCOMPRESSED";
        config.tiling_scheme = TilingScheme::Unci;
        config.format = heif_compression_uncompressed;
        config.encoder_name = "uncompressed";
        config.lossless = true;
        config.quality = 100;
        config.unci_compression = heif_unci_compression_off;
        std::cout << "  [CONFIG] Unknown compression, defaulting to uncompressed unci" << std::endl;
    }
    
    return config;
}

// Create prototype image for unci
heif_image* createPrototypeImageRaw(int num_channels = 3) {
    std::cout << "  [PROTOTYPE] Creating prototype image with " << num_channels << " channels" << std::endl;
    
    heif_image* prototype = nullptr;
    heif_error error = heif_image_create(1, 1, heif_colorspace_RGB, heif_chroma_444, &prototype);
    
    if (error.code != heif_error_Ok) {
        std::cerr << "  [PROTOTYPE] Failed to create prototype: " << error.message << std::endl;
        return nullptr;
    }
    
    error = heif_image_add_plane(prototype, heif_channel_R, 1, 1, 8);
    if (error.code != heif_error_Ok) {
        std::cerr << "  [PROTOTYPE] Failed to add R plane" << std::endl;
        heif_image_release(prototype);
        return nullptr;
    }
    
    if (num_channels >= 2) {
        error = heif_image_add_plane(prototype, heif_channel_G, 1, 1, 8);
        if (error.code != heif_error_Ok) {
            std::cerr << "  [PROTOTYPE] Failed to add G plane" << std::endl;
            heif_image_release(prototype);
            return nullptr;
        }
    }
    
    if (num_channels >= 3) {
        error = heif_image_add_plane(prototype, heif_channel_B, 1, 1, 8);
        if (error.code != heif_error_Ok) {
            std::cerr << "  [PROTOTYPE] Failed to add B plane" << std::endl;
            heif_image_release(prototype);
            return nullptr;
        }
    }
    
    std::cout << "  [PROTOTYPE] Prototype created successfully" << std::endl;
    return prototype;
}

// Read tile data from GDAL and create properly sized tile image
// Returns a heif_image* that must be released by caller
heif_image* createAndReadTile(GDALDataset* ds, int tile_x, int tile_y, 
                              int tile_width, int tile_height,
                              int img_width, int img_height) {
    
    int x_offset = tile_x * tile_width;
    int y_offset = tile_y * tile_height;
    int actual_width = std::min(tile_width, img_width - x_offset);
    int actual_height = std::min(tile_height, img_height - y_offset);
    
    int bands = std::min(3, ds->GetRasterCount());
    
    // Create tile image with actual size (may be smaller than tile_width x tile_height at edges)
    heif_image* tile_img = nullptr;
    heif_error error = heif_image_create(actual_width, actual_height,
                                         heif_colorspace_RGB, heif_chroma_444,
                                         &tile_img);
    
    if (error.code != heif_error_Ok) {
        std::cerr << "    [TILE " << tile_x << "," << tile_y << "] Failed to create: " 
                  << error.message << std::endl;
        return nullptr;
    }
    
    // Add planes
    heif_image_add_plane(tile_img, heif_channel_R, actual_width, actual_height, 8);
    heif_image_add_plane(tile_img, heif_channel_G, actual_width, actual_height, 8);
    heif_image_add_plane(tile_img, heif_channel_B, actual_width, actual_height, 8);
    
    // Get plane pointers
    int stride_r, stride_g, stride_b;
    uint8_t* plane_r = heif_image_get_plane(tile_img, heif_channel_R, &stride_r);
    uint8_t* plane_g = heif_image_get_plane(tile_img, heif_channel_G, &stride_g);
    uint8_t* plane_b = heif_image_get_plane(tile_img, heif_channel_B, &stride_b);
    
    if (!plane_r || !plane_g || !plane_b) {
        std::cerr << "    [TILE " << tile_x << "," << tile_y << "] Failed to get planes" << std::endl;
        heif_image_release(tile_img);
        return nullptr;
    }
    
    // Read each band from GDAL
    for (int band_idx = 1; band_idx <= bands; band_idx++) {
        GDALRasterBand* band = ds->GetRasterBand(band_idx);
        if (!band) continue;
        
        uint8_t* target_plane = (band_idx == 1) ? plane_r : (band_idx == 2) ? plane_g : plane_b;
        int target_stride = (band_idx == 1) ? stride_r : (band_idx == 2) ? stride_g : stride_b;
        
        CPLErr err = band->RasterIO(GF_Read, x_offset, y_offset,
                                   actual_width, actual_height,
                                   target_plane, actual_width, actual_height,
                                   GDT_Byte, 0, target_stride, nullptr);
        
        if (err != CE_None) {
            std::cerr << "    [TILE " << tile_x << "," << tile_y << "] Failed to read band " 
                      << band_idx << std::endl;
            heif_image_release(tile_img);
            return nullptr;
        }
    }
    
    // If grayscale, duplicate R channel to G and B
    if (bands == 1) {
        for (int y = 0; y < actual_height; y++) {
            memcpy(plane_g + y * stride_g, plane_r + y * stride_r, actual_width);
            memcpy(plane_b + y * stride_b, plane_r + y * stride_r, actual_width);
        }
    }
    
    // Extend to full tile size with zeros (for edge tiles)
    // This is CRITICAL - matches the heif-enc.cc example
    if (actual_width < tile_width || actual_height < tile_height) {
        error = heif_image_extend_to_size_fill_with_zero(tile_img, tile_width, tile_height);
        if (error.code != heif_error_Ok) {
            std::cerr << "    [TILE " << tile_x << "," << tile_y << "] Failed to extend: " 
                      << error.message << std::endl;
            heif_image_release(tile_img);
            return nullptr;
        }
    }
    
    return tile_img;
}

// Unified function to add tiles to any tiling scheme (grid, tili, or unci)
bool addTilesToImage(heif_context* ctx, 
                     heif_image_handle* tiled_image_handle,
                     heif_encoder* encoder,
                     GDALDataset* src_ds,
                     int tile_width, int tile_height,
                     int img_width, int img_height,
                     const std::string& scheme_name) {
    
    int tile_cols = (img_width + tile_width - 1) / tile_width;
    int tile_rows = (img_height + tile_height - 1) / tile_height;
    
    std::cout << "[" << scheme_name << "] Encoding tiled image, tile size: " 
              << tile_width << "x" << tile_height << std::endl;
    std::cout << "[" << scheme_name << "] Image size: " 
              << img_width << "x" << img_height << std::endl;
    std::cout << "[" << scheme_name << "] Grid: " << tile_cols << "x" << tile_rows 
              << " tiles" << std::endl;
    
    // Add all tiles
    for (int tile_y = 0; tile_y < tile_rows; tile_y++) {
        for (int tile_x = 0; tile_x < tile_cols; tile_x++) {
            
            // Progress indicator (matching heif-enc.cc format)
            std::cout << "encoding tile " << (tile_y + 1) << " " << (tile_x + 1)
                      << " (of " << tile_rows << "x" << tile_cols << ")  \r";
            std::cout.flush();
            
            // Create and read tile (with proper zero-padding for edge tiles)
            heif_image* tile_img = createAndReadTile(src_ds, tile_x, tile_y,
                                                     tile_width, tile_height,
                                                     img_width, img_height);
            
            if (!tile_img) {
                std::cerr << "\n[" << scheme_name << "] Failed to create tile at (" 
                          << tile_x << "," << tile_y << ")" << std::endl;
                return false;
            }
            
            // Add tile to tiled image
            // NOTE: This same function works for grid, tili, and unci!
            heif_error error = heif_context_add_image_tile(ctx,
                                                           tiled_image_handle,
                                                           tile_x, tile_y,
                                                           tile_img,
                                                           encoder);
            
            // Release tile image immediately after adding
            heif_image_release(tile_img);
            
            if (!checkHeifError(error, "add_image_tile")) {
                std::cerr << "\n[" << scheme_name << "] Failed to add tile at (" 
                          << tile_x << "," << tile_y << ")" << std::endl;
                return false;
            }
        }
    }
    
    std::cout << "\n[" << scheme_name << "] All tiles encoded successfully" << std::endl;
    return true;
}

// Create GeoHEIF with Grid tiling
bool createGeoHEIFGrid(GDALDataset* src_ds, const std::string& output_heif,
                       const CompressionConfig& comp_config, int tile_size,
                       int width, int height) {
    
    std::cout << "[GRID] Creating grid-based HEIF image..." << std::endl;
    
    HEIFWrappers::HeifContext ctx;
    if (!ctx.get()) {
        std::cerr << "[GRID] Failed to create context" << std::endl;
        return false;
    }
    
    int tile_cols = (width + tile_size - 1) / tile_size;
    int tile_rows = (height + tile_size - 1) / tile_size;
    
    // Create grid structure
    HEIFWrappers::HeifImageHandle grid_handle;
    heif_error error = heif_context_add_grid_image(ctx.get(),
                                                   width, height,
                                                   tile_cols, tile_rows,
                                                   nullptr,  // options
                                                   grid_handle.ptr());
    
    if (!checkHeifError(error, "add_grid_image")) {
        return false;
    }
    
    std::cout << "[GRID] Grid structure created" << std::endl;
    
    // Get and configure encoder
    HEIFWrappers::HeifEncoder encoder;
    error = heif_context_get_encoder_for_format(ctx.get(), 
                                                comp_config.format, 
                                                encoder.ptr());
    
    if (!checkHeifError(error, "get encoder") || !encoder.get()) {
        return false;
    }
    
    if (comp_config.lossless) {
        heif_encoder_set_lossless(encoder.get(), 1);
    }
    heif_encoder_set_parameter_integer(encoder.get(), "quality", comp_config.quality);
    
    for (const auto& param : comp_config.parameters) {
        heif_encoder_set_parameter_string(encoder.get(), 
                                          param.first.c_str(), 
                                          param.second.c_str());
    }
    
    // Add tiles using unified function
    bool success = addTilesToImage(ctx.get(), grid_handle.get(), encoder.get(),
                                   src_ds, tile_size, tile_size, width, height, "GRID");
    
    if (!success) {
        return false;
    }
    
    // Set as primary and write
    heif_context_set_primary_image(ctx.get(), grid_handle.get());
    
    error = heif_context_write_to_file(ctx.get(), output_heif.c_str());
    if (!checkHeifError(error, "write to file")) {
        return false;
    }
    
    std::cout << "[GRID] Successfully created" << std::endl;
    return true;
}

// Create GeoHEIF with Tili tiling
bool createGeoHEIFTili(GDALDataset* src_ds, const std::string& output_heif,
                       const CompressionConfig& comp_config, int tile_size,
                       int width, int height) {
    
    std::cout << "[TILI] Creating tili-based HEIF image..." << std::endl;
    
    HEIFWrappers::HeifContext ctx;
    if (!ctx.get()) {
        std::cerr << "[TILI] Failed to create context" << std::endl;
        return false;
    }
    
    // Get and configure encoder FIRST (needed for heif_context_add_tiled_image)
    HEIFWrappers::HeifEncoder encoder;
    heif_error error = heif_context_get_encoder_for_format(ctx.get(), 
                                                           comp_config.format, 
                                                           encoder.ptr());
    
    if (!checkHeifError(error, "get encoder") || !encoder.get()) {
        return false;
    }
    
    if (comp_config.lossless) {
        heif_encoder_set_lossless(encoder.get(), 1);
    }
    heif_encoder_set_parameter_integer(encoder.get(), "quality", comp_config.quality);
    
    for (const auto& param : comp_config.parameters) {
        heif_encoder_set_parameter_string(encoder.get(), 
                                          param.first.c_str(), 
                                          param.second.c_str());
    }
    
    // Setup tili parameters (based on heif-enc.cc)
    heif_tiled_image_parameters tili_params{};
    tili_params.version = 1;
    tili_params.image_width = width;
    tili_params.image_height = height;
    tili_params.tile_width = tile_size;
    tili_params.tile_height = tile_size;
    tili_params.offset_field_length = 32;
    tili_params.size_field_length = 24;
    tili_params.tiles_are_sequential = 1;
    
    // Create tili structure
    HEIFWrappers::HeifImageHandle tili_handle;
    error = heif_context_add_tiled_image(ctx.get(),
                                         &tili_params,
                                         nullptr,      // options
                                         encoder.get(),
                                         tili_handle.ptr());
    
    if (!checkHeifError(error, "add_tiled_image")) {
        return false;
    }
    
    std::cout << "[TILI] Tili structure created" << std::endl;
    
    // Add tiles using unified function
    bool success = addTilesToImage(ctx.get(), tili_handle.get(), encoder.get(),
                                   src_ds, tile_size, tile_size, width, height, "TILI");
    
    if (!success) {
        return false;
    }
    
    // Set as primary and write
    heif_context_set_primary_image(ctx.get(), tili_handle.get());
    
    error = heif_context_write_to_file(ctx.get(), output_heif.c_str());
    if (!checkHeifError(error, "write to file")) {
        return false;
    }
    
    std::cout << "[TILI] Successfully created" << std::endl;
    return true;
}

// Create GeoHEIF with Unci tiling
bool createGeoHEIFUnci(GDALDataset* src_ds, const std::string& output_heif,
                       const CompressionConfig& comp_config, int tile_size,
                       int width, int height) {
    
    std::cout << "[UNCI] Creating unci-based HEIF image..." << std::endl;
    
    HEIFWrappers::HeifContext ctx;
    if (!ctx.get()) {
        std::cerr << "[UNCI] Failed to create context" << std::endl;
        return false;
    }
    
    // Unci requires image dimensions to be multiple of tile size
    int tile_cols = (width + tile_size - 1) / tile_size;
    int tile_rows = (height + tile_size - 1) / tile_size;
    int padded_width = tile_cols * tile_size;
    int padded_height = tile_rows * tile_size;
    
    std::cout << "[UNCI] Original: " << width << "x" << height << std::endl;
    std::cout << "[UNCI] Padded: " << padded_width << "x" << padded_height << std::endl;
    
    // Create prototype
    heif_image* prototype_raw = createPrototypeImageRaw(3);
    if (!prototype_raw) {
        std::cerr << "[UNCI] Failed to create prototype" << std::endl;
        return false;
    }
    HEIFWrappers::HeifImage prototype(prototype_raw);
    
    // Setup unci parameters
    heif_unci_image_parameters unci_params{};
    unci_params.version = 1;
    unci_params.image_width = padded_width;
    unci_params.image_height = padded_height;
    unci_params.tile_width = tile_size;
    unci_params.tile_height = tile_size;
    unci_params.compression = comp_config.unci_compression;
    
    // Create unci structure
    HEIFWrappers::HeifImageHandle unci_handle;
    heif_error error = heif_context_add_empty_unci_image(ctx.get(),
                                                         &unci_params,
                                                         nullptr,
                                                         prototype.get(),
                                                         unci_handle.ptr());
    
    if (!checkHeifError(error, "add_unci_image")) {
        return false;
    }
    
    std::cout << "[UNCI] Unci structure created" << std::endl;
    
    // Get encoder
    HEIFWrappers::HeifEncoder encoder;
    error = heif_context_get_encoder_for_format(ctx.get(), 
                                                comp_config.format, 
                                                encoder.ptr());
    
    if (!checkHeifError(error, "get encoder") || !encoder.get()) {
        return false;
    }
    
    // Add tiles using unified function
    // NOTE: For unci, tiles outside image bounds will be all zeros (handled in createAndReadTile)
    bool success = addTilesToImage(ctx.get(), unci_handle.get(), encoder.get(),
                                   src_ds, tile_size, tile_size, width, height, "UNCI");
    
    if (!success) {
        return false;
    }
    
    // Set as primary and write
    heif_context_set_primary_image(ctx.get(), unci_handle.get());
    
    error = heif_context_write_to_file(ctx.get(), output_heif.c_str());
    if (!checkHeifError(error, "write to file")) {
        return false;
    }
    
    std::cout << "[UNCI] Successfully created" << std::endl;
    return true;
}

// Main GeoHEIF creation function
// Create overview level as a properly tiled image from GDAL dataset
// Encodes it directly as a thumbnail to the main image
bool createOverviewLevel(heif_context* ctx,
                        GDALDataset* src_ds,
                        heif_image_handle* main_handle,
                        heif_encoder* encoder,
                        const CompressionConfig& comp_config,
                        TilingScheme scheme,
                        int tile_size,
                        int level,
                        int base_width,
                        int base_height) {
    
    int scale_factor = 1 << level; // 2^level
    int ov_width = base_width / scale_factor;
    int ov_height = base_height / scale_factor;
    
    if (ov_width < 1) ov_width = 1;
    if (ov_height < 1) ov_height = 1;
    
    std::cout << "  [OVERVIEW " << level << "] Creating " << ov_width << "x" << ov_height 
              << " (scale 1:" << scale_factor << "), tile size: " << tile_size << std::endl;
    
    heif_image_handle* ov_tiled_handle = nullptr;
    heif_error error;
    
    // Create tiled structure for overview based on scheme
    if (scheme == TilingScheme::Grid) {
        int tile_cols = (ov_width + tile_size - 1) / tile_size;
        int tile_rows = (ov_height + tile_size - 1) / tile_size;
        
        std::cout << "    [GRID] " << tile_cols << "x" << tile_rows << " tiles" << std::endl;
        
        error = heif_context_add_grid_image(ctx, ov_width, ov_height,
                                           tile_cols, tile_rows,
                                           nullptr, &ov_tiled_handle);
        
    } else if (scheme == TilingScheme::Tili) {
        heif_tiled_image_parameters tili_params{};
        tili_params.version = 1;
        tili_params.image_width = ov_width;
        tili_params.image_height = ov_height;
        tili_params.tile_width = tile_size;
        tili_params.tile_height = tile_size;
        tili_params.offset_field_length = 32;
        tili_params.size_field_length = 24;
        tili_params.tiles_are_sequential = 1;
        
        std::cout << "    [TILI] " << ((ov_width + tile_size - 1) / tile_size) << "x" 
                  << ((ov_height + tile_size - 1) / tile_size) << " tiles" << std::endl;
        
        error = heif_context_add_tiled_image(ctx, &tili_params,
                                            nullptr, encoder,
                                            &ov_tiled_handle);
        
    } else { // Unci
        int tile_cols = (ov_width + tile_size - 1) / tile_size;
        int tile_rows = (ov_height + tile_size - 1) / tile_size;
        int padded_width = tile_cols * tile_size;
        int padded_height = tile_rows * tile_size;
        
        std::cout << "    [UNCI] " << tile_cols << "x" << tile_rows << " tiles, "
                  << "padded to " << padded_width << "x" << padded_height << std::endl;
        
        heif_image* prototype_raw = createPrototypeImageRaw(3);
        if (!prototype_raw) return false;
        HEIFWrappers::HeifImage prototype(prototype_raw);
        
        heif_unci_image_parameters unci_params{};
        unci_params.version = 1;
        unci_params.image_width = padded_width;
        unci_params.image_height = padded_height;
        unci_params.tile_width = tile_size;
        unci_params.tile_height = tile_size;
        unci_params.compression = comp_config.unci_compression;
        
        error = heif_context_add_empty_unci_image(ctx, &unci_params,
                                                  nullptr, prototype.get(),
                                                  &ov_tiled_handle);
    }
    
    if (!checkHeifError(error, "create overview tiled structure")) {
        return false;
    }
    
    // Now add tiles to the overview image by reading from GDAL at reduced resolution
    std::cout << "    Encoding overview tiles..." << std::endl;
    
    int tile_cols = (ov_width + tile_size - 1) / tile_size;
    int tile_rows = (ov_height + tile_size - 1) / tile_size;
    
    int bands = std::min(3, src_ds->GetRasterCount());
    
    for (int tile_y = 0; tile_y < tile_rows; tile_y++) {
        for (int tile_x = 0; tile_x < tile_cols; tile_x++) {
            
            // Calculate overview tile position
            int ov_x_offset = tile_x * tile_size;
            int ov_y_offset = tile_y * tile_size;
            int ov_actual_width = std::min(tile_size, ov_width - ov_x_offset);
            int ov_actual_height = std::min(tile_size, ov_height - ov_y_offset);
            
            // Calculate corresponding position in source dataset
            int src_x_offset = ov_x_offset * scale_factor;
            int src_y_offset = ov_y_offset * scale_factor;
            int src_width = ov_actual_width * scale_factor;
            int src_height = ov_actual_height * scale_factor;
            
            // Clamp to source bounds
            if (src_x_offset + src_width > base_width) {
                src_width = base_width - src_x_offset;
            }
            if (src_y_offset + src_height > base_height) {
                src_height = base_height - src_y_offset;
            }
            
            // Create tile image (full tile size, will be padded if needed)
            heif_image* tile_img = nullptr;
            error = heif_image_create(tile_size, tile_size,
                                     heif_colorspace_RGB, heif_chroma_444,
                                     &tile_img);
            
            if (error.code != heif_error_Ok) {
                std::cerr << "      [TILE " << tile_x << "," << tile_y << "] Failed to create" << std::endl;
                if (ov_tiled_handle) heif_image_handle_release(ov_tiled_handle);
                return false;
            }
            
            // Add planes
            heif_image_add_plane(tile_img, heif_channel_R, tile_size, tile_size, 8);
            heif_image_add_plane(tile_img, heif_channel_G, tile_size, tile_size, 8);
            heif_image_add_plane(tile_img, heif_channel_B, tile_size, tile_size, 8);
            
            // Get plane pointers
            int stride_r, stride_g, stride_b;
            uint8_t* plane_r = heif_image_get_plane(tile_img, heif_channel_R, &stride_r);
            uint8_t* plane_g = heif_image_get_plane(tile_img, heif_channel_G, &stride_g);
            uint8_t* plane_b = heif_image_get_plane(tile_img, heif_channel_B, &stride_b);
            
            if (!plane_r || !plane_g || !plane_b) {
                heif_image_release(tile_img);
                if (ov_tiled_handle) heif_image_handle_release(ov_tiled_handle);
                return false;
            }
            
            // Initialize with zeros (for padding)
            memset(plane_r, 0, tile_size * stride_r);
            memset(plane_g, 0, tile_size * stride_g);
            memset(plane_b, 0, tile_size * stride_b);
            
            // Read from GDAL with resampling (downsampling)
            for (int band_idx = 1; band_idx <= bands; band_idx++) {
                GDALRasterBand* band = src_ds->GetRasterBand(band_idx);
                if (!band) continue;
                
                uint8_t* target_plane = (band_idx == 1) ? plane_r : 
                                       (band_idx == 2) ? plane_g : plane_b;
                int target_stride = (band_idx == 1) ? stride_r : 
                                   (band_idx == 2) ? stride_g : stride_b;
                
                // Use GDAL's RasterIO to downsample directly
                CPLErr err = band->RasterIO(
                    GF_Read,
                    src_x_offset, src_y_offset,    // Source offset
                    src_width, src_height,          // Source size
                    target_plane,                   // Target buffer
                    ov_actual_width, ov_actual_height, // Target size (downsampled)
                    GDT_Byte,
                    0,                              // Pixel spacing
                    target_stride,                  // Line spacing
                    nullptr);
                
                if (err != CE_None) {
                    std::cerr << "      [TILE " << tile_x << "," << tile_y 
                              << "] Failed to read band " << band_idx << std::endl;
                    heif_image_release(tile_img);
                    if (ov_tiled_handle) heif_image_handle_release(ov_tiled_handle);
                    return false;
                }
            }
            
            // If grayscale, duplicate R to G and B
            if (bands == 1) {
                for (int y = 0; y < ov_actual_height; y++) {
                    memcpy(plane_g + y * stride_g, plane_r + y * stride_r, ov_actual_width);
                    memcpy(plane_b + y * stride_b, plane_r + y * stride_r, ov_actual_width);
                }
            }
            
            // Add tile to overview image
            error = heif_context_add_image_tile(ctx, ov_tiled_handle,
                                               tile_x, tile_y,
                                               tile_img, encoder);
            
            heif_image_release(tile_img);
            
            if (!checkHeifError(error, "add overview tile")) {
                if (ov_tiled_handle) heif_image_handle_release(ov_tiled_handle);
                return false;
            }
        }
    }
    
    std::cout << "    [OVERVIEW " << level << "] All tiles encoded" << std::endl;
    
    // The overview is now in the context and linked via the tiled structure
    // Release the handle
    if (ov_tiled_handle) {
        heif_image_handle_release(ov_tiled_handle);
    }
    
    std::cout << "    [OVERVIEW " << level << "] Added to context" << std::endl;
    return true;
}

// Enhanced createGeoHEIF with proper tiled overviews
bool createGeoHEIFWithOverviews(GDALDataset* src_ds, 
                                const std::string& output_heif,
                                const CompressionConfig& comp_config, 
                                int tile_size,
                                int num_pyramid_levels,
                                int width, int height,
                                TilingScheme scheme) {
    
    std::cout << "[HEIF] Creating with " << num_pyramid_levels << " overview levels..." << std::endl;
    std::cout << "[HEIF] Tiling scheme: ";
    switch (scheme) {
        case TilingScheme::Grid: std::cout << "grid"; break;
        case TilingScheme::Unci: std::cout << "unci"; break;
        case TilingScheme::Tili: std::cout << "tili"; break;
    }
    std::cout << std::endl;
    
    HEIFWrappers::HeifContext ctx;
    if (!ctx.get()) {
        std::cerr << "[HEIF] Failed to create context" << std::endl;
        return false;
    }
    
    // Step 1: Get encoder
    HEIFWrappers::HeifEncoder encoder;
    heif_error error = heif_context_get_encoder_for_format(ctx.get(), 
                                                           comp_config.format, 
                                                           encoder.ptr());
    
    if (!checkHeifError(error, "get encoder")) return false;
    
    // Configure encoder
    if (comp_config.lossless) {
        heif_encoder_set_lossless(encoder.get(), 1);
    }
    heif_encoder_set_parameter_integer(encoder.get(), "quality", comp_config.quality);
    
    for (const auto& param : comp_config.parameters) {
        heif_encoder_set_parameter_string(encoder.get(), 
                                          param.first.c_str(), 
                                          param.second.c_str());
    }
    
    // Step 2: Create main tiled image
    std::cout << "[HEIF] Creating main tiled image " << width << "x" << height << std::endl;
    
    HEIFWrappers::HeifImageHandle main_handle;
    
    if (scheme == TilingScheme::Grid) {
        int tile_cols = (width + tile_size - 1) / tile_size;
        int tile_rows = (height + tile_size - 1) / tile_size;
        error = heif_context_add_grid_image(ctx.get(), width, height,
                                           tile_cols, tile_rows,
                                           nullptr, main_handle.ptr());
    } else if (scheme == TilingScheme::Tili) {
        heif_tiled_image_parameters tili_params{};
        tili_params.version = 1;
        tili_params.image_width = width;
        tili_params.image_height = height;
        tili_params.tile_width = tile_size;
        tili_params.tile_height = tile_size;
        tili_params.offset_field_length = 32;
        tili_params.size_field_length = 24;
        tili_params.tiles_are_sequential = 1;
        
        error = heif_context_add_tiled_image(ctx.get(), &tili_params,
                                            nullptr, encoder.get(),
                                            main_handle.ptr());
    } else { // Unci
        int tile_cols = (width + tile_size - 1) / tile_size;
        int tile_rows = (height + tile_size - 1) / tile_size;
        int padded_width = tile_cols * tile_size;
        int padded_height = tile_rows * tile_size;
        
        heif_image* prototype_raw = createPrototypeImageRaw(3);
        if (!prototype_raw) return false;
        HEIFWrappers::HeifImage prototype(prototype_raw);
        
        heif_unci_image_parameters unci_params{};
        unci_params.version = 1;
        unci_params.image_width = padded_width;
        unci_params.image_height = padded_height;
        unci_params.tile_width = tile_size;
        unci_params.tile_height = tile_size;
        unci_params.compression = comp_config.unci_compression;
        
        error = heif_context_add_empty_unci_image(ctx.get(), &unci_params,
                                                  nullptr, prototype.get(),
                                                  main_handle.ptr());
    }
    
    if (!checkHeifError(error, "create main tiled image")) return false;
    
    // Step 3: Add tiles to main image
    std::cout << "[HEIF] Adding tiles to main image..." << std::endl;
    bool success = addTilesToImage(ctx.get(), main_handle.get(), encoder.get(),
                                   src_ds, tile_size, tile_size, width, height,
                                   "MAIN");
    
    if (!success) return false;
    
    // Step 4: Create tiled overview levels
    // NOTE: In HEIF, multiple resolution levels in a single file are typically
    // stored as separate images in the same context, not as hierarchical thumbnails.
    // The decoder/reader would need to know which image is which resolution.
    // For now, we'll add them as independent images in the same file.
    
    if (num_pyramid_levels > 0) {
        std::cout << "[HEIF] Creating " << num_pyramid_levels 
                  << " tiled overview levels..." << std::endl;
        std::cout << "[HEIF] Note: Overviews stored as separate images in same file" << std::endl;
        
        for (int level = 1; level <= num_pyramid_levels; level++) {
            success = createOverviewLevel(
                ctx.get(), src_ds, main_handle.get(), encoder.get(), 
                comp_config, scheme, tile_size, level, width, height);
            
            if (!success) {
                std::cerr << "[HEIF] Warning: Failed to create overview level " 
                          << level << std::endl;
                // Continue with other levels
            }
        }
    }
    
    // Step 5: Set main image as primary and write
    heif_context_set_primary_image(ctx.get(), main_handle.get());
    
    error = heif_context_write_to_file(ctx.get(), output_heif.c_str());
    if (!checkHeifError(error, "write to file")) return false;
    
    std::cout << "[HEIF] Successfully created with tiled overviews" << std::endl;
    std::cout << "[HEIF] Main image set as primary, overviews are auxiliary images" << std::endl;
    return true;
}

// Main createGeoHEIF function (updated)
bool createGeoHEIF(const std::string& input_tiff,
                   const std::string& output_heif,
                   const std::string& compression = "DEFLATE",
                   int tile_size = 256,
                   int pyramid_levels = 0,
                   bool verbose = true) {
    
    if (verbose) {
        std::cout << "\n[START] Creating GeoHEIF: " << output_heif << std::endl;
        std::cout << "  Compression: " << compression << std::endl;
        std::cout << "  Tile size: " << tile_size << std::endl;
        std::cout << "  Pyramid levels: " << pyramid_levels << std::endl;
    }
    
    CompressionConfig comp_config = getCompressionConfig(compression);
    
    if (verbose) {
        std::cout << "  Using: " << comp_config.name << std::endl;
        std::cout << "  Tiling scheme: ";
        switch (comp_config.tiling_scheme) {
            case TilingScheme::Grid: std::cout << "grid"; break;
            case TilingScheme::Unci: std::cout << "unci"; break;
            case TilingScheme::Tili: std::cout << "tili"; break;
        }
        std::cout << std::endl;
    }
    
    GDALDataset* src_ds = static_cast<GDALDataset*>(GDALOpen(input_tiff.c_str(), GA_ReadOnly));
    if (!src_ds) {
        std::cerr << "[ERROR] Failed to open source: " << input_tiff << std::endl;
        return false;
    }
    
    int width = src_ds->GetRasterXSize();
    int height = src_ds->GetRasterYSize();
    int bands = src_ds->GetRasterCount();
    
    if (verbose) {
        std::cout << "[GDAL] Dimensions: " << width << "x" << height << "x" << bands << std::endl;
    }
    
    bool success;
    
    if (pyramid_levels > 0) {
        // Use the new tiled overview function
        success = createGeoHEIFWithOverviews(src_ds, output_heif, comp_config,
                                            tile_size, pyramid_levels,
                                            width, height,
                                            comp_config.tiling_scheme);
    } else {
        // Use existing single-resolution creation (no overviews needed)
        if (comp_config.tiling_scheme == TilingScheme::Unci) {
            success = createGeoHEIFUnci(src_ds, output_heif, comp_config, tile_size, width, height);
        } else if (comp_config.tiling_scheme == TilingScheme::Tili) {
            success = createGeoHEIFTili(src_ds, output_heif, comp_config, tile_size, width, height);
        } else {
            success = createGeoHEIFGrid(src_ds, output_heif, comp_config, tile_size, width, height);
        }
    }
    
    if (success) {
        // Create sidecar files for georeferencing
        double adfGeoTransform[6];
        if (src_ds->GetGeoTransform(adfGeoTransform) == CE_None) {
            std::string wld_file = output_heif.substr(0, output_heif.find_last_of('.')) + ".wld";
            std::ofstream wld(wld_file);
            if (wld.is_open()) {
                wld << std::fixed << std::setprecision(10);
                wld << adfGeoTransform[1] << "\n";
                wld << adfGeoTransform[4] << "\n";
                wld << adfGeoTransform[2] << "\n";
                wld << adfGeoTransform[5] << "\n";
                wld << adfGeoTransform[0] << "\n";
                wld << adfGeoTransform[3] << "\n";
                wld.close();
                if (verbose) std::cout << "[SIDECAR] Created .wld file" << std::endl;
            }
        }
        
        const char* projection = src_ds->GetProjectionRef();
        if (projection && strlen(projection) > 0) {
            std::string aux_file = output_heif + ".aux.xml";
            std::ofstream aux(aux_file);
            if (aux.is_open()) {
                aux << "<PAMDataset>\n";
                aux << "  <SRS>" << projection << "</SRS>\n";
                if (src_ds->GetGeoTransform(adfGeoTransform) == CE_None) {
                    aux << "  <GeoTransform>";
                    for (int i = 0; i < 6; i++) {
                        if (i > 0) aux << ", ";
                        aux << adfGeoTransform[i];
                    }
                    aux << "</GeoTransform>\n";
                }
                aux << "</PAMDataset>\n";
                aux.close();
                if (verbose) std::cout << "[SIDECAR] Created .aux.xml file" << std::endl;
            }
        }
        
        if (verbose) {
            std::cout << "[SUCCESS] GeoHEIF created: " << output_heif << std::endl;
            long file_size = BenchmarkUtils::getFileSize(output_heif);
            if (file_size > 0) {
                std::cout << "[SUCCESS] File size: " << (file_size / 1024.0 / 1024.0) << " MB" << std::endl;
            }
        }
    }
    
    GDALClose(src_ds);
    
    if (verbose) {
        std::cout << "[END] " << (success ? "Completed" : "Failed") << "\n" << std::endl;
    }
    
    return success;
}


} // namespace HEIFCreator

std::cout << "HEIF creator with grid/unci/tili support loaded (FINAL - unified tile adding)." << std::endl;
std::cout << "✅ All three tiling methods use the same heif_context_add_image_tile() function" << std::endl;
std::cout << "✅ Tiles are automatically extended to full size with zeros for edge tiles" << std::endl;

### Quick test of HEIF Creator

In [ ]:
// Quick test of HEIF creation
bool test_result = HEIFCreator::createGeoHEIF(
    "test_input.tif",
    "test_output.heic",
    "DEFLATE",
    256,
    0,
    true
);

std::cout << "Test result: " << (test_result ? "SUCCESS" : "FAILED") << std::endl;

### Fallback classes

In [ ]:
namespace HEIFCreator {

// Create simple (untiled, no-pyramid) GeoHEIF with proper compression support
bool createGeoHEIFSimple(const std::string& input_tiff,
                         const std::string& output_heif,
                         const std::string& compression = "DEFLATE",
                         bool verbose = true) {
    
    if (verbose) {
        std::cout << "\n[HEIF-SIMPLE] Creating untiled HEIF: " << output_heif << std::endl;
        std::cout << "  Compression: " << compression << std::endl;
    }
    
    // Get compression configuration
    CompressionConfig comp_config = getCompressionConfig(compression);
    
    if (verbose) {
        std::cout << "  Using: " << comp_config.name << std::endl;
    }
    
    // Open source
    GDALDataset* src_ds = static_cast<GDALDataset*>(GDALOpen(input_tiff.c_str(), GA_ReadOnly));
    if (!src_ds) {
        std::cerr << "[HEIF-SIMPLE] Failed to open source: " << input_tiff << std::endl;
        return false;
    }
    
    int width = src_ds->GetRasterXSize();
    int height = src_ds->GetRasterYSize();
    int bands = std::min(3, src_ds->GetRasterCount());
    
    if (verbose) {
        std::cout << "[HEIF-SIMPLE] Dimensions: " << width << "x" << height << "x" << bands << std::endl;
    }
    
    // Create HEIF context
    HEIFWrappers::HeifContext ctx;
    if (!ctx.get()) {
        GDALClose(src_ds);
        return false;
    }
    
    // Get appropriate encoder
    HEIFWrappers::HeifEncoder encoder;
    heif_error error = heif_context_get_encoder_for_format(ctx.get(), 
                                                           comp_config.format,
                                                           encoder.ptr());
    
    if (error.code != heif_error_Ok) {
        std::cerr << "[HEIF-SIMPLE] Failed to get encoder: " << error.message << std::endl;
        GDALClose(src_ds);
        return false;
    }
    
    // Configure encoder based on compression config
    if (comp_config.lossless) {
        heif_encoder_set_lossless(encoder.get(), 1);
    }
    heif_encoder_set_parameter_integer(encoder.get(), "quality", comp_config.quality);
    
    // Set additional parameters
    for (const auto& param : comp_config.parameters) {
        heif_encoder_set_parameter_string(encoder.get(), 
                                          param.first.c_str(), 
                                          param.second.c_str());
    }
    
    if (verbose) {
        std::cout << "[HEIF-SIMPLE] Encoder: " << heif_encoder_get_name(encoder.get()) << std::endl;
    }
    
    // Create image based on compression type
    HEIFWrappers::HeifImage img;
    
    if (comp_config.format == heif_compression_uncompressed) {
        // For uncompressed/unci, we need to use the uncompressed format
        // But for a simple non-tiled image, we can just encode with uncompressed encoder
        
        error = heif_image_create(width, height, 
                                 heif_colorspace_RGB,
                                 heif_chroma_444,
                                 img.ptr());
        
        if (error.code != heif_error_Ok) {
            std::cerr << "[HEIF-SIMPLE] Failed to create image" << std::endl;
            GDALClose(src_ds);
            return false;
        }
        
        // Add planes
        heif_image_add_plane(img.get(), heif_channel_R, width, height, 8);
        heif_image_add_plane(img.get(), heif_channel_G, width, height, 8);
        heif_image_add_plane(img.get(), heif_channel_B, width, height, 8);
        
        // Read data
        int stride_r, stride_g, stride_b;
        uint8_t* plane_r = heif_image_get_plane(img.get(), heif_channel_R, &stride_r);
        uint8_t* plane_g = heif_image_get_plane(img.get(), heif_channel_G, &stride_g);
        uint8_t* plane_b = heif_image_get_plane(img.get(), heif_channel_B, &stride_b);
        
        if (!plane_r || !plane_g || !plane_b) {
            GDALClose(src_ds);
            return false;
        }
        
        // Read each band from GDAL
        for (int band_idx = 1; band_idx <= bands; band_idx++) {
            GDALRasterBand* band = src_ds->GetRasterBand(band_idx);
            if (!band) continue;
            
            uint8_t* target_plane = (band_idx == 1) ? plane_r : 
                                   (band_idx == 2) ? plane_g : plane_b;
            int target_stride = (band_idx == 1) ? stride_r : 
                               (band_idx == 2) ? stride_g : stride_b;
            
            CPLErr err = band->RasterIO(GF_Read, 0, 0, width, height,
                                       target_plane, width, height,
                                       GDT_Byte, 0, target_stride, nullptr);
            
            if (err != CE_None) {
                std::cerr << "[HEIF-SIMPLE] Failed to read band " << band_idx << std::endl;
                GDALClose(src_ds);
                return false;
            }
        }
        
        // If grayscale, duplicate R channel to G and B
        if (bands == 1) {
            for (int y = 0; y < height; y++) {
                memcpy(plane_g + y * stride_g, plane_r + y * stride_r, width);
                memcpy(plane_b + y * stride_b, plane_r + y * stride_r, width);
            }
        }
        
    } else {
        // For other formats (HEVC, AV1, JPEG, etc.), use standard RGB interleaved
        error = heif_image_create(width, height,
                                 heif_colorspace_RGB,
                                 heif_chroma_interleaved_RGB,
                                 img.ptr());
        
        if (error.code != heif_error_Ok) {
            std::cerr << "[HEIF-SIMPLE] Failed to create image" << std::endl;
            GDALClose(src_ds);
            return false;
        }
        
        // Add interleaved plane
        heif_image_add_plane(img.get(), heif_channel_interleaved, width, height, 8);
        
        // Get plane pointer
        int stride;
        uint8_t* data = heif_image_get_plane(img.get(), heif_channel_interleaved, &stride);
        
        if (!data) {
            GDALClose(src_ds);
            return false;
        }
        
        // Read data band by band and interleave
        std::vector<uint8_t> band_data(width * height);
        
        for (int band_idx = 1; band_idx <= bands; band_idx++) {
            GDALRasterBand* band = src_ds->GetRasterBand(band_idx);
            if (!band) continue;
            
            CPLErr err = band->RasterIO(GF_Read, 0, 0, width, height,
                                       band_data.data(), width, height,
                                       GDT_Byte, 0, 0, nullptr);
            
            if (err != CE_None) {
                std::cerr << "[HEIF-SIMPLE] Failed to read band " << band_idx << std::endl;
                GDALClose(src_ds);
                return false;
            }
            
            // Interleave into RGB buffer
            for (int y = 0; y < height; y++) {
                for (int x = 0; x < width; x++) {
                    data[y * stride + x * 3 + (band_idx - 1)] = band_data[y * width + x];
                }
            }
        }
        
        // If grayscale, duplicate to all channels
        if (bands == 1) {
            for (int y = 0; y < height; y++) {
                for (int x = 0; x < width; x++) {
                    uint8_t val = data[y * stride + x * 3];
                    data[y * stride + x * 3 + 1] = val;
                    data[y * stride + x * 3 + 2] = val;
                }
            }
        }
    }
    
    // Encode the image
    HEIFWrappers::HeifImageHandle handle;
    
    if (comp_config.format == heif_compression_uncompressed && 
        comp_config.unci_compression != heif_unci_compression_off) {
        // For UNCI with compression, create a simple unci image
        heif_image* prototype = createPrototypeImageRaw(3);
        if (!prototype) {
            GDALClose(src_ds);
            return false;
        }
        
        heif_unci_image_parameters unci_params{};
        unci_params.version = 1;
        unci_params.image_width = width;
        unci_params.image_height = height;
        unci_params.tile_width = width;   // Single tile = entire image
        unci_params.tile_height = height;
        unci_params.compression = comp_config.unci_compression;
        
        error = heif_context_add_empty_unci_image(ctx.get(), &unci_params,
                                                  nullptr, prototype,
                                                  handle.ptr());
        heif_image_release(prototype);
        
        if (error.code != heif_error_Ok) {
            std::cerr << "[HEIF-SIMPLE] Failed to create unci image: " << error.message << std::endl;
            GDALClose(src_ds);
            return false;
        }
        
        // Add the single tile (entire image)
        error = heif_context_add_image_tile(ctx.get(), handle.get(),
                                           0, 0,  // tile position
                                           img.get(), encoder.get());
        
        if (error.code != heif_error_Ok) {
            std::cerr << "[HEIF-SIMPLE] Failed to add image tile: " << error.message << std::endl;
            GDALClose(src_ds);
            return false;
        }
        
    } else {
        // Standard encoding for other formats
        error = heif_context_encode_image(ctx.get(), img.get(), encoder.get(),
                                          nullptr, handle.ptr());
        
        if (error.code != heif_error_Ok) {
            std::cerr << "[HEIF-SIMPLE] Failed to encode image: " << error.message << std::endl;
            GDALClose(src_ds);
            return false;
        }
    }
    
    // Set as primary
    heif_context_set_primary_image(ctx.get(), handle.get());
    
    // Write to file
    error = heif_context_write_to_file(ctx.get(), output_heif.c_str());
    
    if (error.code != heif_error_Ok) {
        std::cerr << "[HEIF-SIMPLE] Failed to write file: " << error.message << std::endl;
        GDALClose(src_ds);
        return false;
    }
    
    // Create sidecar files for georeferencing
    double adfGeoTransform[6];
    if (src_ds->GetGeoTransform(adfGeoTransform) == CE_None) {
        std::string wld_file = output_heif.substr(0, output_heif.find_last_of('.')) + ".wld";
        std::ofstream wld(wld_file);
        if (wld.is_open()) {
            wld << std::fixed << std::setprecision(10);
            wld << adfGeoTransform[1] << "\n";
            wld << adfGeoTransform[4] << "\n";
            wld << adfGeoTransform[2] << "\n";
            wld << adfGeoTransform[5] << "\n";
            wld << adfGeoTransform[0] << "\n";
            wld << adfGeoTransform[3] << "\n";
            wld.close();
        }
    }
    
    const char* projection = src_ds->GetProjectionRef();
    if (projection && strlen(projection) > 0) {
        std::string aux_file = output_heif + ".aux.xml";
        std::ofstream aux(aux_file);
        if (aux.is_open()) {
            aux << "<PAMDataset>\n";
            aux << "  <SRS>" << projection << "</SRS>\n";
            if (src_ds->GetGeoTransform(adfGeoTransform) == CE_None) {
                aux << "  <GeoTransform>";
                for (int i = 0; i < 6; i++) {
                    if (i > 0) aux << ", ";
                    aux << adfGeoTransform[i];
                }
                aux << "</GeoTransform>\n";
            }
            aux << "</PAMDataset>\n";
            aux.close();
        }
    }
    
    GDALClose(src_ds);
    
    if (verbose) {
        long file_size = BenchmarkUtils::getFileSize(output_heif);
        std::cout << "[HEIF-SIMPLE] Created successfully" << std::endl;
        std::cout << "[HEIF-SIMPLE] File size: " << std::fixed << std::setprecision(2) 
                  << (file_size / 1024.0 / 1024.0) << " MB" << std::endl;
    }
    
    return true;
}

} // namespace HEIFCreator

std::cout << "HEIF simple creator updated with proper compression support." << std::endl;

## Benchmarking Functions - Part 1: COG

In [ ]:
namespace Benchmarker {

// Benchmark COG read with multiple iterations
BenchmarkUtils::PerformanceMetrics benchmarkCOGRead(const std::string& cog_path,
                                    const std::string& compression,
                                    int tile_size,
                                    int pyramid_levels,
                                    int iterations = 5,
                                    bool is_subset = false,
                                    int x_off = 0, int y_off = 0,
                                    int x_size = 0, int y_size = 0) {
    
    BenchmarkUtils::PerformanceMetrics metrics;
    metrics.format = "COG";
    metrics.operation = is_subset ? "subset_read" : "full_read";
    metrics.compression = compression;
    metrics.tile_size = tile_size;
    metrics.pyramid_levels = pyramid_levels;
    metrics.num_iterations = iterations;
    
    std::vector<double> read_times;
    std::vector<long> bytes_read_samples;
    std::vector<long> memory_samples;
    
    for (int iter = 0; iter < iterations; iter++) {
        long mem_before = BenchmarkUtils::getMemoryUsageKB();
        long io_before = BenchmarkUtils::getBytesRead();
        double cpu_before = BenchmarkUtils::getCPUTime();
        BenchmarkUtils::Timer timer;
        
        // Open dataset
        GDALDataset* ds = static_cast<GDALDataset*>(GDALOpen(cog_path.c_str(), GA_ReadOnly));
        if (!ds) {
            std::cerr << "Failed to open: " << cog_path << std::endl;
            break;
        }
        
        if (iter == 0) {
            metrics.width = ds->GetRasterXSize();
            metrics.height = ds->GetRasterYSize();
            metrics.bands = ds->GetRasterCount();
            metrics.file_size_bytes = BenchmarkUtils::getFileSize(cog_path);
            metrics.header_size_bytes = std::min(16384L, metrics.file_size_bytes);
        }
        
        // Determine read dimensions
        int read_x_off = is_subset ? x_off : 0;
        int read_y_off = is_subset ? y_off : 0;
        int read_x_size = is_subset ? x_size : metrics.width;
        int read_y_size = is_subset ? y_size : metrics.height;
        
        // Read data
        GDALRasterBand* band = ds->GetRasterBand(1);
        GDALDataType dtype = band->GetRasterDataType();
        int pixel_size = GDALGetDataTypeSizeBytes(dtype);
        
        size_t buffer_size = static_cast<size_t>(read_x_size) * read_y_size * pixel_size;
        void* buffer = malloc(buffer_size);
        
        if (buffer) {
            CPLErr err = band->RasterIO(GF_Read, read_x_off, read_y_off, 
                                       read_x_size, read_y_size,
                                       buffer, read_x_size, read_y_size,
                                       dtype, 0, 0, nullptr);
            
            read_times.push_back(timer.elapsed_ms());
            
            free(buffer);
            buffer = nullptr;
        }
        
        long io_after = BenchmarkUtils::getBytesRead();
        long mem_after = BenchmarkUtils::getMemoryUsageKB();
        double cpu_after = BenchmarkUtils::getCPUTime();
        
        bytes_read_samples.push_back(io_after - io_before);
        memory_samples.push_back(mem_after - mem_before);
        
        if (iter == 0) {
            double elapsed_sec = read_times[0] / 1000.0;
            if (elapsed_sec > 0) {
                metrics.cpu_usage_percent = ((cpu_after - cpu_before) / elapsed_sec) * 100.0;
            }
        }
        
        GDALClose(ds);
        ds = nullptr;
    }
    
    // Calculate statistics
    if (!read_times.empty()) {
        double sum = 0;
        for (double t : read_times) sum += t;
        metrics.read_time_ms = sum / read_times.size();
        metrics.time_std_dev = BenchmarkUtils::calcStdDev(read_times);
    }
    
    if (!bytes_read_samples.empty()) {
        long sum = 0;
        for (long b : bytes_read_samples) sum += b;
        metrics.bytes_read_from_disk = sum / bytes_read_samples.size();
    }
    
    if (!memory_samples.empty()) {
        long sum = 0;
        for (long m : memory_samples) sum += m;
        metrics.memory_usage_kb = sum / memory_samples.size();
    }
    
    metrics.peak_memory_kb = BenchmarkUtils::getPeakMemoryKB();
    
    return metrics;
}

} // namespace Benchmarker

std::cout << "COG benchmarking functions loaded." << std::endl;

## Benchmarking Functions - Part 2: HEIF

In [ ]:
namespace Benchmarker {

// Benchmark HEIF read with libheif API
BenchmarkUtils::PerformanceMetrics benchmarkHEIFRead(const std::string& heif_path,
                                     const std::string& compression,
                                     int tile_size,
                                     int pyramid_levels,
                                     int iterations = 5,
                                     bool is_subset = false,
                                     int x_off = 0, int y_off = 0,
                                     int x_size = 0, int y_size = 0) {
    
    BenchmarkUtils::PerformanceMetrics metrics;
    metrics.format = "GeoHEIF";
    metrics.operation = is_subset ? "subset_read" : "full_read";
    metrics.compression = compression;
    metrics.tile_size = tile_size;
    metrics.pyramid_levels = pyramid_levels;
    metrics.num_iterations = iterations;
    
    std::vector<double> read_times;
    std::vector<long> bytes_read_samples;
    std::vector<long> memory_samples;
    
    for (int iter = 0; iter < iterations; iter++) {
        long mem_before = BenchmarkUtils::getMemoryUsageKB();
        long io_before = BenchmarkUtils::getBytesRead();
        double cpu_before = BenchmarkUtils::getCPUTime();
        BenchmarkUtils::Timer timer;
        
        // Create HEIF context with RAII
        HEIFWrappers::HeifContext ctx;
        if (!ctx.get()) {
            std::cerr << "Failed to create HEIF context" << std::endl;
            break;
        }
        
        // Read file
        heif_error error = heif_context_read_from_file(ctx.get(), heif_path.c_str(), nullptr);
        if (error.code != heif_error_Ok) {
            std::cerr << "Failed to read HEIF: " << error.message << std::endl;
            break;
        }
        
        // Get primary image handle with RAII
        HEIFWrappers::HeifImageHandle img_handle;
        error = heif_context_get_primary_image_handle(ctx.get(), img_handle.ptr());
        if (error.code != heif_error_Ok) {
            std::cerr << "Failed to get image handle: " << error.message << std::endl;
            break;
        }
        
        if (iter == 0) {
            metrics.width = heif_image_handle_get_width(img_handle.get());
            metrics.height = heif_image_handle_get_height(img_handle.get());
            metrics.bands = 3;  // RGB
            metrics.file_size_bytes = BenchmarkUtils::getFileSize(heif_path);
            metrics.header_size_bytes = std::min(16384L, metrics.file_size_bytes);
        }
        
        // Decode image with RAII
        HEIFWrappers::HeifImage img;
        error = heif_decode_image(img_handle.get(), img.ptr(),
                                 heif_colorspace_RGB,
                                 heif_chroma_interleaved_RGB,
                                 nullptr);
        
        if (error.code != heif_error_Ok) {
            std::cerr << "Failed to decode image: " << error.message << std::endl;
            break;
        }
        
        // Get image data
        int stride;
        const uint8_t* data = heif_image_get_plane_readonly(img.get(), 
                                                            heif_channel_interleaved,
                                                            &stride);
        
        // If subset, extract the region
        if (is_subset && data) {
            size_t subset_size = static_cast<size_t>(x_size) * y_size * 3;
            void* subset_buffer = malloc(subset_size);
            if (subset_buffer) {
                // Copy subset data
                for (int y = 0; y < y_size; y++) {
                    const uint8_t* src_row = data + ((y_off + y) * stride) + (x_off * 3);
                    uint8_t* dst_row = static_cast<uint8_t*>(subset_buffer) + (y * x_size * 3);
                    memcpy(dst_row, src_row, x_size * 3);
                }
                free(subset_buffer);
                subset_buffer = nullptr;
            }
        }
        
        read_times.push_back(timer.elapsed_ms());
        
        // RAII handles cleanup automatically
        
        long io_after = BenchmarkUtils::getBytesRead();
        long mem_after = BenchmarkUtils::getMemoryUsageKB();
        double cpu_after = BenchmarkUtils::getCPUTime();
        
        bytes_read_samples.push_back(io_after - io_before);
        memory_samples.push_back(mem_after - mem_before);
        
        if (iter == 0) {
            double elapsed_sec = read_times[0] / 1000.0;
            if (elapsed_sec > 0) {
                metrics.cpu_usage_percent = ((cpu_after - cpu_before) / elapsed_sec) * 100.0;
            }
        }
    }
    
    // Calculate statistics
    if (!read_times.empty()) {
        double sum = 0;
        for (double t : read_times) sum += t;
        metrics.read_time_ms = sum / read_times.size();
        metrics.time_std_dev = BenchmarkUtils::calcStdDev(read_times);
    }
    
    if (!bytes_read_samples.empty()) {
        long sum = 0;
        for (long b : bytes_read_samples) sum += b;
        metrics.bytes_read_from_disk = sum / bytes_read_samples.size();
    }
    
    if (!memory_samples.empty()) {
        long sum = 0;
        for (long m : memory_samples) sum += m;
        metrics.memory_usage_kb = sum / memory_samples.size();
    }
    
    metrics.peak_memory_kb = BenchmarkUtils::getPeakMemoryKB();
    
    return metrics;
}

} // namespace Benchmarker

std::cout << "HEIF benchmarking functions loaded." << std::endl;

## Benchmarking Functions - Part 3: Metadata & First-Byte

In [ ]:
namespace Benchmarker {

// Benchmark metadata extraction specifically
BenchmarkUtils::PerformanceMetrics benchmarkMetadataRetrieval(const std::string& file_path,
                                              const std::string& format,
                                              const std::string& compression,
                                              int tile_size,
                                              int pyramid_levels,
                                              int iterations = 5) {
    BenchmarkUtils::PerformanceMetrics metrics;
    metrics.format = format;
    metrics.operation = "metadata_retrieval";
    metrics.compression = compression;
    metrics.tile_size = tile_size;
    metrics.pyramid_levels = pyramid_levels;
    metrics.num_iterations = iterations;
    
    std::vector<double> retrieval_times;
    
    for (int iter = 0; iter < iterations; iter++) {
        BenchmarkUtils::Timer timer;
        
        if (format == "COG") {
            GDALDataset* ds = static_cast<GDALDataset*>(GDALOpen(file_path.c_str(), GA_ReadOnly));
            if (ds) {
                // Extract metadata only
                int width = ds->GetRasterXSize();
                int height = ds->GetRasterYSize();
                int bands = ds->GetRasterCount();
                const char* projection = ds->GetProjectionRef();
                double geotransform[6];
                ds->GetGeoTransform(geotransform);
                
                // Get band metadata
                for (int i = 1; i <= bands && i <= 3; i++) {
                    GDALRasterBand* band = ds->GetRasterBand(i);
                    if (band) {
                        GDALDataType dtype = band->GetRasterDataType();
                        int has_nodata;
                        double nodata = band->GetNoDataValue(&has_nodata);
                    }
                }
                
                if (iter == 0) {
                    metrics.width = width;
                    metrics.height = height;
                    metrics.bands = bands;
                    metrics.file_size_bytes = BenchmarkUtils::getFileSize(file_path);
                }
                
                GDALClose(ds);
            }
        } else if (format == "GeoHEIF") {
            HEIFWrappers::HeifContext ctx;
            if (ctx.get()) {
                heif_error error = heif_context_read_from_file(ctx.get(), file_path.c_str(), nullptr);
                if (error.code == heif_error_Ok) {
                    HEIFWrappers::HeifImageHandle handle;
                    error = heif_context_get_primary_image_handle(ctx.get(), handle.ptr());
                    if (error.code == heif_error_Ok) {
                        int width = heif_image_handle_get_width(handle.get());
                        int height = heif_image_handle_get_height(handle.get());
                        bool has_alpha = heif_image_handle_has_alpha_channel(handle.get());
                        int depth = heif_image_handle_get_luma_bits_per_pixel(handle.get());
                        
                        if (iter == 0) {
                            metrics.width = width;
                            metrics.height = height;
                            metrics.bands = 3;
                            metrics.file_size_bytes = BenchmarkUtils::getFileSize(file_path);
                        }
                    }
                }
            }
        }
        
        retrieval_times.push_back(timer.elapsed_ms());
    }
    
    // Calculate statistics
    if (!retrieval_times.empty()) {
        double sum = 0;
        for (double t : retrieval_times) sum += t;
        metrics.metadata_retrieval_time_ms = sum / retrieval_times.size();
        metrics.time_std_dev = BenchmarkUtils::calcStdDev(retrieval_times);
    }
    
    return metrics;
}

// Measure time to first byte (cloud optimization metric)
BenchmarkUtils::PerformanceMetrics benchmarkFirstByte(const std::string& file_path,
                                     const std::string& format,
                                     const std::string& compression,
                                     int tile_size,
                                     int pyramid_levels) {
    BenchmarkUtils::PerformanceMetrics metrics;
    metrics.format = format;
    metrics.operation = "first_byte";
    metrics.compression = compression;
    metrics.tile_size = tile_size;
    metrics.pyramid_levels = pyramid_levels;
    metrics.file_size_bytes = BenchmarkUtils::getFileSize(file_path);
    
    BenchmarkUtils::Timer timer;
    
    if (format == "COG") {
        // Open file with VSI for range request simulation
        VSILFILE* fp = VSIFOpenL(file_path.c_str(), "rb");
        if (fp) {
            // Read just the header (first 16KB)
            char buffer[16384];
            size_t bytes_read = VSIFReadL(buffer, 1, 16384, fp);
            metrics.first_byte_time_ms = timer.elapsed_ms();
            metrics.header_size_bytes = bytes_read;
            VSIFCloseL(fp);
        }
    } else if (format == "GeoHEIF") {
        // For HEIF, measure time to read header and get basic info
        HEIFWrappers::HeifContext ctx;
        if (ctx.get()) {
            heif_error error = heif_context_read_from_file(ctx.get(), file_path.c_str(), nullptr);
            if (error.code == heif_error_Ok) {
                HEIFWrappers::HeifImageHandle handle;
                error = heif_context_get_primary_image_handle(ctx.get(), handle.ptr());
                metrics.first_byte_time_ms = timer.elapsed_ms();
                
                if (error.code == heif_error_Ok) {
                    metrics.width = heif_image_handle_get_width(handle.get());
                    metrics.height = heif_image_handle_get_height(handle.get());
                    metrics.bands = 3;
                }
            }
        }
    }
    
    return metrics;
}

} // namespace Benchmarker

std::cout << "Metadata and first-byte benchmarking functions loaded." << std::endl;

## Dataset Validation System

In [ ]:
namespace Validation {

struct ValidationResult {
    bool valid;
    std::string error_message;
    std::map<std::string, std::string> properties;
};

ValidationResult validateCOG(const std::string& cog_path) {
    ValidationResult result;
    result.valid = false;
    
    GDALDataset* ds = static_cast<GDALDataset*>(GDALOpen(cog_path.c_str(), GA_ReadOnly));
    if (!ds) {
        result.error_message = "Failed to open file";
        return result;
    }
    
    // Check if it's a GeoTIFF
    const char* driver_name = ds->GetDriver()->GetDescription();
    if (std::string(driver_name) != "GTiff" && std::string(driver_name) != "COG") {
        result.error_message = "Not a GeoTIFF/COG file";
        GDALClose(ds);
        return result;
    }
    
    result.properties["driver"] = driver_name;
    result.properties["width"] = std::to_string(ds->GetRasterXSize());
    result.properties["height"] = std::to_string(ds->GetRasterYSize());
    result.properties["bands"] = std::to_string(ds->GetRasterCount());
    
    // Check for tiling
    int block_x, block_y;
    GDALRasterBand* band = ds->GetRasterBand(1);
    if (band) {
        band->GetBlockSize(&block_x, &block_y);
        result.properties["tile_width"] = std::to_string(block_x);
        result.properties["tile_height"] = std::to_string(block_y);
        
        bool is_tiled = (block_x < ds->GetRasterXSize() || block_y < ds->GetRasterYSize());
        result.properties["tiled"] = is_tiled ? "yes" : "no";
    }
    
    // Check for overviews
    int overview_count = band ? band->GetOverviewCount() : 0;
    result.properties["overview_count"] = std::to_string(overview_count);
    
    result.valid = true;
    GDALClose(ds);
    return result;
}

ValidationResult validateHEIF(const std::string& heif_path) {
    ValidationResult result;
    result.valid = false;
    
    HEIFWrappers::HeifContext ctx;
    if (!ctx.get()) {
        result.error_message = "Failed to create HEIF context";
        return result;
    }
    
    heif_error error = heif_context_read_from_file(ctx.get(), heif_path.c_str(), nullptr);
    if (error.code != heif_error_Ok) {
        result.error_message = std::string("Failed to read HEIF: ") + error.message;
        return result;
    }
    
    HEIFWrappers::HeifImageHandle handle;
    error = heif_context_get_primary_image_handle(ctx.get(), handle.ptr());
    if (error.code != heif_error_Ok) {
        result.error_message = "Failed to get image handle";
        return result;
    }
    
    result.properties["width"] = std::to_string(heif_image_handle_get_width(handle.get()));
    result.properties["height"] = std::to_string(heif_image_handle_get_height(handle.get()));
    result.properties["has_alpha"] = heif_image_handle_has_alpha_channel(handle.get()) ? "yes" : "no";
    
    // Check for thumbnails (pyramid levels)
    int num_thumbnails = heif_image_handle_get_number_of_thumbnails(handle.get());
    result.properties["thumbnail_count"] = std::to_string(num_thumbnails);
    
    result.valid = true;
    return result;
}

} // namespace Validation

std::cout << "Dataset validation system loaded." << std::endl;

## Visualization with gnuplot

In [ ]:
namespace Visualization {

// Use filesystem namespace alias
namespace fs = std::filesystem;

// Helper to get file size
long getFileSize(const std::string& filename) {
    struct stat stat_buf;
    int rc = stat(filename.c_str(), &stat_buf);
    return rc == 0 ? stat_buf.st_size : -1;
}

// Helper to check if file exists
bool fileExists(const std::string& filename) {
    struct stat buffer;
    return (stat(filename.c_str(), &buffer) == 0);
}

// Detect available gnuplot terminal
std::string detectGnuplotTerminal() {
    #ifdef __APPLE__
        return "png size 1920,1080";
    #else
        return "pngcairo size 1920,1080 enhanced";
    #endif
}

// Export metrics to CSV
void exportMetricsToCSV(const std::vector<BenchmarkUtils::PerformanceMetrics>& metrics,
                        const std::string& csv_path) {
    std::ofstream csv(csv_path);
    if (!csv.is_open()) {
        std::cerr << "Failed to open CSV: " << csv_path << std::endl;
        return;
    }
    
    // Header
    csv << "format,operation,compression,tile_size,pyramid_levels,"
        << "file_size_bytes,metadata_size_bytes,header_size_bytes,"
        << "read_time_ms,write_time_ms,metadata_retrieval_time_ms,first_byte_time_ms,"
        << "bytes_read_from_disk,cpu_usage_percent,memory_usage_kb,peak_memory_kb,"
        << "width,height,bands,time_std_dev,num_iterations\n";
    
    // Data
    for (const auto& m : metrics) {
        csv << m.format << ","
            << m.operation << ","
            << m.compression << ","
            << m.tile_size << ","
            << m.pyramid_levels << ","
            << m.file_size_bytes << ","
            << m.metadata_size_bytes << ","
            << m.header_size_bytes << ","
            << m.read_time_ms << ","
            << m.write_time_ms << ","
            << m.metadata_retrieval_time_ms << ","
            << m.first_byte_time_ms << ","
            << m.bytes_read_from_disk << ","
            << m.cpu_usage_percent << ","
            << m.memory_usage_kb << ","
            << m.peak_memory_kb << ","
            << m.width << ","
            << m.height << ","
            << m.bands << ","
            << m.time_std_dev << ","
            << m.num_iterations << "\n";
    }
    
    csv.close();
    std::cout << "✓ Metrics exported to: " << csv_path << std::endl;
}

// Create gnuplot script
void createGnuplotScript(const std::string& csv_path, 
                         const std::string& output_dir) {
    
    std::cout << "\nCreating gnuplot visualization..." << std::endl;
    
    // Build paths using filesystem
    fs::path output_dir_path(output_dir);
    fs::path script_path = output_dir_path / "plot_results.gp";
    fs::path output_png = output_dir_path / "benchmark_results.png";
    
    std::string script_path_str = script_path.string();
    std::string output_png_str = output_png.string();
    
    std::ofstream script(script_path);
    
    if (!script.is_open()) {
        std::cerr << "Failed to create gnuplot script: " << script_path_str << std::endl;
        return;
    }
    
    // Detect terminal
    std::string terminal = detectGnuplotTerminal();
    
    // Write gnuplot script
    script << "# Gnuplot script for benchmarking results\n";
    script << "set terminal " << terminal << "\n";
    script << "set output '" << output_png_str << "'\n";
    script << "set datafile separator ','\n";
    script << "set key outside right top\n";
    script << "set grid\n";
    script << "set style fill solid 0.5\n";
    script << "set boxwidth 0.9\n";
    script << "set style data histogram\n";
    script << "set style histogram cluster gap 1\n";
    script << "\n";
    script << "set multiplot layout 2,3 title 'HEIF vs COG Performance'\n";
    script << "\n";
    
    // File Size
    script << "# File Size\n";
    script << "set title 'File Size'\n";
    script << "set ylabel 'Size (MB)'\n";
    script << "plot '" << csv_path << "' every ::1 using (stringcolumn(1) eq \"COG\" ? $6/1048576 : 1/0):xtic(3) title 'COG' lc rgb 'blue', \\\n";
    script << "     '' every ::1 using (stringcolumn(1) eq \"GeoHEIF\" ? $6/1048576 : 1/0) title 'GeoHEIF' lc rgb 'red'\n";
    script << "\n";
    
    // Read Time
    script << "# Read Time\n";
    script << "set title 'Full Read Time'\n";
    script << "set ylabel 'Time (ms)'\n";
    script << "plot '" << csv_path << "' every ::1 using (stringcolumn(1) eq \"COG\" && stringcolumn(2) eq \"full_read\" ? $9 : 1/0):xtic(3) title 'COG' lc rgb 'blue', \\\n";
    script << "     '' every ::1 using (stringcolumn(1) eq \"GeoHEIF\" && stringcolumn(2) eq \"full_read\" ? $9 : 1/0) title 'GeoHEIF' lc rgb 'red'\n";
    script << "\n";
    
    // Subset Read
    script << "# Subset Read\n";
    script << "set title 'Subset Read Time'\n";
    script << "set ylabel 'Time (ms)'\n";
    script << "plot '" << csv_path << "' every ::1 using (stringcolumn(1) eq \"COG\" && stringcolumn(2) eq \"subset_read\" ? $9 : 1/0):xtic(3) title 'COG' lc rgb 'blue', \\\n";
    script << "     '' every ::1 using (stringcolumn(1) eq \"GeoHEIF\" && stringcolumn(2) eq \"subset_read\" ? $9 : 1/0) title 'GeoHEIF' lc rgb 'red'\n";
    script << "\n";
    
    // Memory
    script << "# Memory\n";
    script << "set title 'Memory Usage'\n";
    script << "set ylabel 'Memory (MB)'\n";
    script << "plot '" << csv_path << "' every ::1 using (stringcolumn(1) eq \"COG\" ? $15/1024 : 1/0):xtic(3) title 'COG' lc rgb 'blue', \\\n";
    script << "     '' every ::1 using (stringcolumn(1) eq \"GeoHEIF\" ? $15/1024 : 1/0) title 'GeoHEIF' lc rgb 'red'\n";
    script << "\n";
    
    // Metadata Retrieval
    script << "# Metadata Retrieval\n";
    script << "set title 'Metadata Retrieval Time'\n";
    script << "set ylabel 'Time (ms)'\n";
    script << "plot '" << csv_path << "' every ::1 using (stringcolumn(1) eq \"COG\" && stringcolumn(2) eq \"metadata_retrieval\" ? $11 : 1/0):xtic(3) title 'COG' lc rgb 'blue', \\\n";
    script << "     '' every ::1 using (stringcolumn(1) eq \"GeoHEIF\" && stringcolumn(2) eq \"metadata_retrieval\" ? $11 : 1/0) title 'GeoHEIF' lc rgb 'red'\n";
    script << "\n";
    
    // First Byte Time
    script << "# First Byte Time\n";
    script << "set title 'First Byte Time (Cloud Optimization)'\n";
    script << "set ylabel 'Time (ms)'\n";
    script << "plot '" << csv_path << "' every ::1 using (stringcolumn(1) eq \"COG\" && stringcolumn(2) eq \"first_byte\" ? $12 : 1/0):xtic(3) title 'COG' lc rgb 'blue', \\\n";
    script << "     '' every ::1 using (stringcolumn(1) eq \"GeoHEIF\" && stringcolumn(2) eq \"first_byte\" ? $12 : 1/0) title 'GeoHEIF' lc rgb 'red'\n";
    script << "\n";
    
    script << "unset multiplot\n";
    script << "quit\n";
    
    script.close();
    
    std::cout << "✓ Gnuplot script created: " << script_path_str << std::endl;
    std::cout << "\n" << std::string(60, '=') << std::endl;
    std::cout << "To generate visualization, run:\n";
    std::cout << "  gnuplot \"" << script_path_str << "\"\n";
    std::cout << "Output: " << output_png_str << "\n";
    std::cout << std::string(60, '=') << std::endl;
}

} // namespace Visualization

std::cout << "Visualization functions loaded." << std::endl;

## Complete Dialog functions under DialogSystem

In [ ]:
namespace DialogSystem {

// Force complete output flush
void forceFlush() {
    std::cout << std::flush;
    std::cerr << std::flush;
    // Give the output buffer time to flush
    std::this_thread::sleep_for(std::chrono::milliseconds(200));
}

// Clear all input
void clearAllInput() {
    std::cin.clear();
    std::cin.sync();
    // Discard any pending input
    while (std::cin.rdbuf()->in_avail() > 0) {
        std::cin.ignore();
    }
}


// Add filesystem namespace alias at the top of the namespace
namespace fs = std::filesystem;

// Path utilities for cross-platform support
class PathUtils {
public:
    static fs::path getCurrentWorkingDirectory() {
        return fs::current_path();
    }
    
    static fs::path getDefaultSourceDataDir() {
        return getCurrentWorkingDirectory() / "srcdata";
    }
    
    static fs::path getDefaultSourceFile() {
        return getDefaultSourceDataDir() / "ACT2017.tiff";
    }
    
    static fs::path getDefaultOutputDirectory() {
        return getCurrentWorkingDirectory() / "benchmark_output";
    }
    
    static bool exists(const fs::path& path) {
        std::error_code ec;
        return fs::exists(path, ec);
    }
    
    static bool isFile(const fs::path& path) {
        std::error_code ec;
        return fs::is_regular_file(path, ec);
    }
    
    static bool createDirectories(const fs::path& path) {
        std::error_code ec;
        return fs::create_directories(path, ec);
    }
    
    static std::string toString(const fs::path& path) {
        return path.string();
    }
    
    static fs::path fromString(const std::string& str) {
        return fs::path(str);
    }
    
    static fs::path getAbsolutePath(const fs::path& path) {
        std::error_code ec;
        fs::path abs_path = fs::absolute(path, ec);
        if (ec) return path;
        return abs_path;
    }
};

// Helper function to convert vector to string
template<typename T>
std::string vectorToString(const std::vector<T>& vec) {
    std::ostringstream oss;
    for (size_t i = 0; i < vec.size(); i++) {
        if (i > 0) oss << ",";
        oss << vec[i];
    }
    if (vec.empty()) {
        oss << "none";
    }
    return oss.str();
}

// Specialization for string vectors
template<>
std::string vectorToString<std::string>(const std::vector<std::string>& vec) {
    std::ostringstream oss;
    for (size_t i = 0; i < vec.size(); i++) {
        if (i > 0) oss << ",";
        oss << vec[i];
    }
    if (vec.empty()) {
        oss << "none";
    }
    return oss.str();
}


// Preset configurations
class PresetConfigurations {
public:
    // Quick test preset
    static BenchmarkUtils::BenchmarkConfig getQuickTestPreset() {
        BenchmarkUtils::BenchmarkConfig config;
        config.tile_sizes = {256};
        config.pyramid_levels = {0};
        config.compressions = {"NONE", "DEFLATE"};
        config.benchmark_iterations = 3;
        config.include_no_tiling = true;
        config.include_no_pyramids = true;
        config.enable_cloud_optimization = true;
        config.enable_validation = true;
        config.show_progress = true;
        return config;
    }
    
    // Standard preset
    static BenchmarkUtils::BenchmarkConfig getStandardPreset() {
        BenchmarkUtils::BenchmarkConfig config;
        config.tile_sizes = {256, 512};
        config.pyramid_levels = {0, 3};
        config.compressions = {"NONE", "LZW", "DEFLATE", "ZSTD"};
        config.benchmark_iterations = 5;
        config.include_no_tiling = true;
        config.include_no_pyramids = true;
        config.enable_cloud_optimization = true;
        config.enable_validation = true;
        config.show_progress = true;
        return config;
    }
    
    // Comprehensive preset
    static BenchmarkUtils::BenchmarkConfig getComprehensivePreset() {
        BenchmarkUtils::BenchmarkConfig config;
        config.tile_sizes = {128, 256, 512, 1024};
        config.pyramid_levels = {0, 2, 3, 5};
        config.compressions = {"NONE", "LZW", "DEFLATE", "ZSTD"};
        config.benchmark_iterations = 10;
        config.include_no_tiling = true;
        config.include_no_pyramids = true;
        config.enable_cloud_optimization = true;
        config.enable_validation = true;
        config.show_progress = true;
        return config;
    }
    
    // Simple preset - only untiled, no pyramids
    static BenchmarkUtils::BenchmarkConfig getSimplePreset() {
        BenchmarkUtils::BenchmarkConfig config;
        config.tile_sizes = {};  // Empty - will create only untiled
        config.pyramid_levels = {0};
        config.compressions = {"NONE", "LZW", "DEFLATE", "ZSTD"};
        config.benchmark_iterations = 5;
        config.include_no_tiling = true;
        config.include_no_pyramids = true;
        config.enable_cloud_optimization = true;
        config.enable_validation = true;
        config.show_progress = true;
        return config;
    }
    
    // Display preset menu - COMPLETELY REWRITTEN
    static int showPresetMenu() {
        // Aggressive input clearing
        clearAllInput();
        
        // Force output and wait
        std::cout << "\n\n\n";  // Extra newlines for visibility
        forceFlush();
        
        std::cout << "========================================================\n";
        forceFlush();
        std::cout << "Select Configuration Preset\n";
        forceFlush();
        std::cout << "========================================================\n";
        forceFlush();
        std::cout << "\n";
        forceFlush();
        
        std::cout << "1. Quick Test\n";
        std::cout << "   - Tile sizes: 256 + untiled\n";
        std::cout << "   - Pyramids: 0\n";
        std::cout << "   - Compressions: NONE, DEFLATE\n";
        std::cout << "   - Iterations: 3\n";
        std::cout << "   - Est. time: ~5-10 minutes\n";
        forceFlush();
        std::cout << "\n";
        
        std::cout << "2. Standard (Recommended)\n";
        std::cout << "   - Tile sizes: 256, 512 + untiled\n";
        std::cout << "   - Pyramids: 0, 3\n";
        std::cout << "   - Compressions: NONE, LZW, DEFLATE, ZSTD\n";
        std::cout << "   - Iterations: 5\n";
        std::cout << "   - Est. time: ~30-45 minutes\n";
        forceFlush();
        std::cout << "\n";
        
        std::cout << "3. Comprehensive\n";
        std::cout << "   - Tile sizes: 128, 256, 512, 1024 + untiled\n";
        std::cout << "   - Pyramids: 0, 2, 3, 5\n";
        std::cout << "   - Compressions: NONE, LZW, DEFLATE, ZSTD\n";
        std::cout << "   - Iterations: 10\n";
        std::cout << "   - Est. time: ~2-3 hours\n";
        forceFlush();
        std::cout << "\n";
        
        std::cout << "4. Simple (Untiled Only)\n";
        std::cout << "   - No tiling, no pyramids\n";
        std::cout << "   - Compressions: NONE, LZW, DEFLATE, ZSTD\n";
        std::cout << "   - Iterations: 5\n";
        std::cout << "   - Est. time: ~10-15 minutes\n";
        forceFlush();
        std::cout << "\n";
        
        std::cout << "5. Custom (Interactive Dialog)\n";
        forceFlush();
        std::cout << "\n";
        
        std::cout << "========================================================\n";
        forceFlush();
        std::cout << "\n";
        
        std::cout << "Select preset [1-5] [default=2]: ";
        forceFlush();
        
        // Clear input one more time right before reading
        clearAllInput();
        
        std::string input;
        if (!std::getline(std::cin, input)) {
            std::cout << "\n→ Error reading input, using default: 2\n";
            return 2;
        }
        
        // Trim the input
        size_t first = input.find_first_not_of(" \t\n\r");
        if (first != std::string::npos) {
            size_t last = input.find_last_not_of(" \t\n\r");
            input = input.substr(first, last - first + 1);
        } else {
            input = "";
        }
        
        if (input.empty()) {
            std::cout << "→ Using default: 2 (Standard)\n";
            forceFlush();
            return 2;
        }
        
        try {
            int choice = std::stoi(input);
            if (choice >= 1 && choice <= 5) {
                std::cout << "→ Selected: " << choice << "\n";
                forceFlush();
                return choice;
            } else {
                std::cout << "→ Invalid choice, using default: 2\n";
                forceFlush();
                return 2;
            }
        } catch (...) {
            std::cout << "→ Invalid input, using default: 2\n";
            forceFlush();
            return 2;
        }
    }
};

// Full interactive configuration dialog
class BenchmarkConfigDialog {
private:
    // Current configuration
    fs::path source_file;
    fs::path output_directory;
    std::vector<int> tile_sizes;
    std::vector<int> pyramid_levels;
    std::vector<std::string> compressions;
    int benchmark_iterations;
    bool include_no_tiling;
    bool include_no_pyramids;
    bool enable_cloud_optimization;
    bool enable_validation;
    bool show_progress;
    
    // Immutable defaults
    const fs::path DEFAULT_SOURCE_FILE;
    const fs::path DEFAULT_OUTPUT_DIR;
    const std::vector<int> DEFAULT_TILE_SIZES;
    const std::vector<int> DEFAULT_PYRAMID_LEVELS;
    const std::vector<std::string> DEFAULT_COMPRESSIONS;
    const int DEFAULT_ITERATIONS;
    const bool DEFAULT_INCLUDE_NO_TILING;
    const bool DEFAULT_INCLUDE_NO_PYRAMIDS;
    const bool DEFAULT_CLOUD_OPTIMIZATION;
    const bool DEFAULT_VALIDATION;
    const bool DEFAULT_SHOW_PROGRESS;
    
    // Valid options
    const std::vector<std::string> VALID_COMPRESSIONS = {"NONE", "LZW", "DEFLATE", "ZSTD", "BROTLI", "JPEG"};
    
    // Helper to trim whitespace
    std::string trim(const std::string& str) const {
        size_t first = str.find_first_not_of(" \t\n\r");
        if (first == std::string::npos) return "";
        size_t last = str.find_last_not_of(" \t\n\r");
        return str.substr(first, last - first + 1);
    }
    
    // Helper to flush output
    void flushOutput() const {
        std::cout << std::flush;
        std::this_thread::sleep_for(std::chrono::milliseconds(50));
    }
    
    // Clear input buffer
    void clearInputBuffer() const {
        std::cin.clear();
        if (std::cin.rdbuf()->in_avail() > 0) {
            std::cin.ignore(std::numeric_limits<std::streamsize>::max(), '\n');
        }
    }
    
    // Read line with default
    std::string readLineWithDefault(const std::string& prompt, const std::string& default_value) {
        clearInputBuffer();
        
        std::cout << "\n" << std::string(60, '=') << "\n";
        std::cout << prompt << "\n";
        if (!default_value.empty()) {
            std::cout << "[Default: " << default_value << "]\n";
        }
        std::cout << std::string(60, '=') << "\n";
        std::cout << ">> ";
        flushOutput();
        
        std::string input;
        std::getline(std::cin, input);
        input = trim(input);
        
        if (input.empty() && !default_value.empty()) {
            std::cout << "→ Using default: " << default_value << "\n";
            flushOutput();
            return default_value;
        }
        
        if (!input.empty()) {
            std::cout << "→ You entered: " << input << "\n";
            flushOutput();
        }
        
        return input.empty() ? default_value : input;
    }
    
    // Read integer with default
    int readIntWithDefault(const std::string& prompt, int default_value) {
        clearInputBuffer();
        
        std::string default_str = std::to_string(default_value);
        std::cout << "\n" << std::string(60, '=') << "\n";
        std::cout << prompt << "\n";
        std::cout << "[Default: " << default_str << "]\n";
        std::cout << std::string(60, '=') << "\n";
        std::cout << ">> ";
        flushOutput();
        
        std::string input;
        std::getline(std::cin, input);
        input = trim(input);
        
        if (input.empty()) {
            std::cout << "→ Using default: " << default_value << "\n";
            flushOutput();
            return default_value;
        }
        
        try {
            int value = std::stoi(input);
            std::cout << "→ Set to: " << value << "\n";
            flushOutput();
            return value;
        } catch (...) {
            std::cout << "→ Invalid input, using default: " << default_value << "\n";
            flushOutput();
            return default_value;
        }
    }
    
    // Read yes/no with default
    bool readYesNoWithDefault(const std::string& prompt, bool default_value) {
        clearInputBuffer();
        
        std::string default_str = default_value ? "yes" : "no";
        std::cout << "\n" << std::string(60, '=') << "\n";
        std::cout << prompt << " (y/n)\n";
        std::cout << "[Default: " << default_str << "]\n";
        std::cout << std::string(60, '=') << "\n";
        std::cout << ">> ";
        flushOutput();
        
        std::string input;
        std::getline(std::cin, input);
        input = trim(input);
        
        if (input.empty()) {
            std::cout << "→ Using default: " << default_str << "\n";
            flushOutput();
            return default_value;
        }
        
        char first_char = std::tolower(input[0]);
        bool result = (first_char == 'y' || first_char == '1' || first_char == 't');
        std::cout << "→ Selected: " << (result ? "yes" : "no") << "\n";
        flushOutput();
        return result;
    }
    
    // Parse comma-separated integers
    std::vector<int> parseIntList(const std::string& input) {
        std::vector<int> result;
        std::istringstream iss(input);
        std::string token;
        
        while (std::getline(iss, token, ',')) {
            token = trim(token);
            if (!token.empty()) {
                try {
                    int val = std::stoi(token);
                    if (val >= 0) {  // Changed to allow 0
                        result.push_back(val);
                    } else {
                        std::cout << "⚠ Skipping invalid value: " << token << " (must be >= 0)\n";
                        flushOutput();
                    }
                } catch (...) {
                    std::cout << "⚠ Skipping invalid value: " << token << " (not a number)\n";
                    flushOutput();
                }
            }
        }
        
        return result;
    }
    
    // Validate compression
    bool isValidCompression(const std::string& comp) const {
        for (const auto& valid : VALID_COMPRESSIONS) {
            if (comp == valid) return true;
        }
        return false;
    }
    
    // Parse comma-separated strings with validation
    std::vector<std::string> parseStringList(const std::string& input) {
        std::vector<std::string> result;
        std::istringstream iss(input);
        std::string token;
        
        while (std::getline(iss, token, ',')) {
            token = trim(token);
            if (!token.empty()) {
                std::transform(token.begin(), token.end(), token.begin(), ::toupper);
                
                if (isValidCompression(token)) {
                    result.push_back(token);
                } else {
                    std::cout << "⚠ Skipping invalid compression: " << token << "\n";
                    std::cout << "   Valid options: ";
                    for (size_t i = 0; i < VALID_COMPRESSIONS.size(); i++) {
                        if (i > 0) std::cout << ", ";
                        std::cout << VALID_COMPRESSIONS[i];
                    }
                    std::cout << "\n";
                    flushOutput();
                }
            }
        }
        
        return result;
    }
    
public:
    // Constructor with immutable defaults
    BenchmarkConfigDialog() : 
        DEFAULT_SOURCE_FILE(PathUtils::getDefaultSourceFile()),
        DEFAULT_OUTPUT_DIR(PathUtils::getDefaultOutputDirectory()),
        DEFAULT_TILE_SIZES({256, 512}),
        DEFAULT_PYRAMID_LEVELS({0, 3}),
        DEFAULT_COMPRESSIONS({"NONE", "DEFLATE", "ZSTD"}),
        DEFAULT_ITERATIONS(5),
        DEFAULT_INCLUDE_NO_TILING(true),
        DEFAULT_INCLUDE_NO_PYRAMIDS(true),
        DEFAULT_CLOUD_OPTIMIZATION(true),
        DEFAULT_VALIDATION(true),
        DEFAULT_SHOW_PROGRESS(true)
    {
        resetToDefaults();
    }
    
    // Reset to defaults
    void resetToDefaults() {
        source_file = DEFAULT_SOURCE_FILE;
        output_directory = DEFAULT_OUTPUT_DIR;
        tile_sizes = DEFAULT_TILE_SIZES;
        pyramid_levels = DEFAULT_PYRAMID_LEVELS;
        compressions = DEFAULT_COMPRESSIONS;
        benchmark_iterations = DEFAULT_ITERATIONS;
        include_no_tiling = DEFAULT_INCLUDE_NO_TILING;
        include_no_pyramids = DEFAULT_INCLUDE_NO_PYRAMIDS;
        enable_cloud_optimization = DEFAULT_CLOUD_OPTIMIZATION;
        enable_validation = DEFAULT_VALIDATION;
        show_progress = DEFAULT_SHOW_PROGRESS;
    }
    
    // Display banner
    void displayBanner() {
        clearInputBuffer();
        flushOutput();
        
        std::cout << "\n";
        std::cout << "========================================================\n";
        std::cout << "   HEIF/GeoHEIF vs COG/GeoTIFF Benchmarking Tool\n";
        std::cout << "========================================================\n";
        std::cout << "\n";
        std::cout << "This tool creates benchmark dataset pairs and compares\n";
        std::cout << "performance between HEIF/GeoHEIF and COG/GeoTIFF.\n";
        std::cout << "\n";
        std::cout << "Working directory: \n";
        std::cout << "  " << PathUtils::toString(PathUtils::getCurrentWorkingDirectory()) << "\n";
        std::cout << "\n";
        std::cout << "INSTRUCTIONS:\n";
        std::cout << "  • Default values are shown in [Default: value]\n";
        std::cout << "  • Press Enter to accept the default\n";
        std::cout << "  • The >> prompt shows where to type your input\n";
        std::cout << "  • Defaults never change between runs\n";
        std::cout << "\n";
        std::cout << "========================================================\n";
        flushOutput();
    }
    
    // Run interactive dialog (continued in next part due to length...)
    bool runDialog();  // Declaration here, implementation follows
    
    // Display summary and confirm
    bool displaySummaryAndConfirm();  // Declaration here, implementation follows
    
    // Convert to BenchmarkConfig
    BenchmarkUtils::BenchmarkConfig toBenchmarkConfig() const {
        BenchmarkUtils::BenchmarkConfig config;
        config.tile_sizes = tile_sizes;
        config.pyramid_levels = pyramid_levels;
        config.compressions = compressions;
        config.include_no_tiling = include_no_tiling;
        config.include_no_pyramids = include_no_pyramids;
        config.benchmark_iterations = benchmark_iterations;
        config.enable_cloud_optimization = enable_cloud_optimization;
        config.enable_validation = enable_validation;
        config.show_progress = show_progress;
        return config;
    }
    
    // Getters
    std::string getSourceFile() const { return PathUtils::toString(source_file); }
    std::string getOutputDirectory() const { return PathUtils::toString(output_directory); }
    fs::path getSourceFilePath() const { return source_file; }
    fs::path getOutputDirectoryPath() const { return output_directory; }
};

// Implementation of runDialog() - FULL IMPLEMENTATION
bool BenchmarkConfigDialog::runDialog() {
    resetToDefaults();
    clearInputBuffer();
    
    displayBanner();
    
    // Section 1: Input/Output
    std::cout << "\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    std::cout << "  INPUT/OUTPUT CONFIGURATION\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    flushOutput();
    
    // Source file
    while (true) {
        std::string input = readLineWithDefault(
            "Source GeoTIFF file path:",
            PathUtils::toString(DEFAULT_SOURCE_FILE));
        source_file = PathUtils::fromString(input);
        
        if (PathUtils::exists(source_file) && PathUtils::isFile(source_file)) {
            fs::path abs_path = PathUtils::getAbsolutePath(source_file);
            std::cout << "✓ File found!\n";
            std::cout << "  " << PathUtils::toString(abs_path) << "\n";
            flushOutput();
            source_file = abs_path;
            break;
        } else {
            std::cout << "✗ File not found: " << PathUtils::toString(source_file) << "\n";
            flushOutput();
            
            clearInputBuffer();
            std::cout << "\nTry again? (y/n)\n";
            std::cout << "[Default: yes]\n";
            std::cout << ">> ";
            flushOutput();
            
            std::string retry;
            std::getline(std::cin, retry);
            retry = trim(retry);
            
            if (!retry.empty() && std::tolower(retry[0]) != 'y') {
                std::cout << "⚠ Warning: Continuing with non-existent file.\n";
                flushOutput();
                break;
            }
        }
    }
    
    // Output directory
    std::string output_input = readLineWithDefault(
        "Output directory:", 
        PathUtils::toString(DEFAULT_OUTPUT_DIR));
    output_directory = PathUtils::fromString(output_input);
    
    if (!PathUtils::exists(output_directory)) {
        std::cout << "Creating output directory...\n";
        flushOutput();
        if (PathUtils::createDirectories(output_directory)) {
            std::cout << "✓ Directory created\n";
        } else {
            std::cout << "✗ Failed to create directory\n";
        }
    } else {
        std::cout << "→ Directory already exists\n";
    }
    flushOutput();
    
    // Section 2: Tiling
    std::cout << "\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    std::cout << "  TILING CONFIGURATION\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    flushOutput();
    
    // Ask if user wants tiled versions
    bool want_tiled = readYesNoWithDefault(
        "Create TILED versions?\n(If no, only untiled versions will be created)",
        true);
    
    if (want_tiled) {
        std::string tile_default = vectorToString<int>(DEFAULT_TILE_SIZES);
        std::string tile_input = readLineWithDefault(
            "Tile sizes (comma-separated positive integers):\nExample: 128,256,512",
            tile_default);
        
        std::vector<int> parsed_tiles = parseIntList(tile_input);
        if (!parsed_tiles.empty()) {
            tile_sizes = parsed_tiles;
            std::cout << "✓ Tile sizes set to: " << vectorToString<int>(tile_sizes) << "\n";
        } else {
            std::cout << "⚠ No valid tile sizes, using default: " << tile_default << "\n";
            tile_sizes = DEFAULT_TILE_SIZES;
        }
        flushOutput();
        
        // Ask about untiled versions
        include_no_tiling = readYesNoWithDefault(
            "Also include UNTILED versions?\n(Untiled = no internal tiling, scanline/strip layout)",
            DEFAULT_INCLUDE_NO_TILING);
    } else {
        // User doesn't want tiled versions, only untiled
        tile_sizes.clear();
        include_no_tiling = true;
        std::cout << "✓ Will create only UNTILED versions\n";
        flushOutput();
    }
    
    // Section 3: Pyramids
    std::cout << "\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    std::cout << "  PYRAMID/OVERVIEW CONFIGURATION\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    flushOutput();
    
    // Only ask about pyramids if user wants tiled versions
    if (want_tiled && !tile_sizes.empty()) {
        bool want_pyramids = readYesNoWithDefault(
            "Create versions WITH PYRAMIDS/OVERVIEWS?\n(If no, only versions without pyramids will be created)",
            true);
        
        if (want_pyramids) {
            std::string pyramid_default = vectorToString<int>(DEFAULT_PYRAMID_LEVELS);
            std::string pyramid_input = readLineWithDefault(
                "Pyramid levels (comma-separated non-negative integers):\nExample: 0,3,5  (0 means no pyramids)",
                pyramid_default);
            
            std::vector<int> parsed_pyramids = parseIntList(pyramid_input);
            
            // Allow 0 as valid pyramid level
            if (pyramid_input.find("0") != std::string::npos) {
                if (std::find(parsed_pyramids.begin(), parsed_pyramids.end(), 0) == parsed_pyramids.end()) {
                    parsed_pyramids.insert(parsed_pyramids.begin(), 0);
                }
            }
            
            if (!parsed_pyramids.empty()) {
                pyramid_levels = parsed_pyramids;
                std::cout << "✓ Pyramid levels set to: " << vectorToString<int>(pyramid_levels) << "\n";
            } else {
                std::cout << "⚠ No valid pyramid levels, using default: " << pyramid_default << "\n";
                pyramid_levels = DEFAULT_PYRAMID_LEVELS;
            }
            flushOutput();
            
            // Check if 0 is in the list
            bool has_zero = std::find(pyramid_levels.begin(), pyramid_levels.end(), 0) != pyramid_levels.end();
            
            if (!has_zero) {
                include_no_pyramids = readYesNoWithDefault(
                    "Also include versions WITHOUT pyramids (pyramid level 0)?\n(Recommended for baseline comparison)",
                    DEFAULT_INCLUDE_NO_PYRAMIDS);
            } else {
                include_no_pyramids = true;
                std::cout << "✓ Pyramid level 0 already included\n";
                flushOutput();
            }
        } else {
            // User doesn't want pyramids, only level 0
            pyramid_levels = {0};
            include_no_pyramids = true;
            std::cout << "✓ Will create only versions WITHOUT pyramids (level 0)\n";
            flushOutput();
        }
    } else {
        // Untiled versions don't support pyramids
        pyramid_levels = {0};
        include_no_pyramids = true;
        std::cout << "ℹ Untiled versions do not support pyramids\n";
        std::cout << "✓ Pyramid level set to: 0\n";
        flushOutput();
    }
    
    // Section 4: Compression
    std::cout << "\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    std::cout << "  COMPRESSION CONFIGURATION\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    std::cout << "\nVALID compression algorithms:\n";
    std::cout << "  • NONE      - Uncompressed\n";
    std::cout << "  • LZW       - Lossless (COG: LZW, HEIF: mapped to DEFLATE)\n";
    std::cout << "  • DEFLATE   - Lossless (both COG and HEIF)\n";
    std::cout << "  • ZSTD      - Lossless (COG: ZSTD, HEIF: mapped to BROTLI)\n";
    std::cout << "  • JPEG      - Lossy (optional, for comparison)\n";
    std::cout << "\nONLY these values will be accepted (case-insensitive)\n";
    flushOutput();
    
    std::string compression_default = vectorToString<std::string>(DEFAULT_COMPRESSIONS);
    std::string compression_input = readLineWithDefault(
        "Compression algorithms (comma-separated):\nExample: NONE,DEFLATE,ZSTD",
        compression_default);
    
    std::vector<std::string> parsed_compressions = parseStringList(compression_input);
    if (!parsed_compressions.empty()) {
        compressions = parsed_compressions;
        std::cout << "✓ Compressions set to: " << vectorToString<std::string>(compressions) << "\n";
    } else {
        std::cout << "⚠ No valid compressions, using default: " << compression_default << "\n";
        compressions = DEFAULT_COMPRESSIONS;
    }
    flushOutput();
    
    // Section 5: Benchmark Settings
    std::cout << "\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    std::cout << "  BENCHMARK CONFIGURATION\n";
    std::cout << "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n";
    flushOutput();
    
    benchmark_iterations = readIntWithDefault(
        "Number of iterations per test:\n(For statistical accuracy, minimum 1)",
        DEFAULT_ITERATIONS);
    
    if (benchmark_iterations < 1) {
        benchmark_iterations = 1;
        std::cout << "⚠ Adjusted to minimum: 1\n";
        flushOutput();
    }
    
    enable_cloud_optimization = readYesNoWithDefault(
        "Enable cloud optimization features?\n(Measures first-byte-time and header efficiency)",
        DEFAULT_CLOUD_OPTIMIZATION);
    
    enable_validation = readYesNoWithDefault(
        "Enable dataset validation?\n(Verifies created files after generation)",
        DEFAULT_VALIDATION);
    
    show_progress = readYesNoWithDefault(
        "Show progress bars?",
        DEFAULT_SHOW_PROGRESS);
    
    // Section 6: Summary and Confirmation
    return displaySummaryAndConfirm();
}

// Implementation of displaySummaryAndConfirm()
bool BenchmarkConfigDialog::displaySummaryAndConfirm() {
    clearInputBuffer();
    flushOutput();
    
    std::cout << "\n";
    std::cout << "========================================================\n";
    std::cout << "              CONFIGURATION SUMMARY\n";
    std::cout << "========================================================\n\n";
    
    std::cout << "📁 Input/Output:\n";
    std::cout << "   Source:    " << PathUtils::toString(source_file) << "\n";
    std::cout << "   Output:    " << PathUtils::toString(output_directory) << "\n\n";
    
    std::cout << "🔲 Tiling:\n";
    if (tile_sizes.empty()) {
        std::cout << "   Tiled versions: NONE\n";
    } else {
        std::cout << "   Tile sizes: " << vectorToString<int>(tile_sizes) << "\n";
    }
    std::cout << "   Include untiled: " << (include_no_tiling ? "Yes" : "No") << "\n\n";
    
    std::cout << "📊 Pyramids:\n";
    std::cout << "   Levels:    " << vectorToString<int>(pyramid_levels) << "\n";
    if (!tile_sizes.empty()) {
        std::cout << "   Include no-pyramid: " << (include_no_pyramids ? "Yes" : "No") << "\n";
    } else {
        std::cout << "   (Pyramids N/A for untiled-only)\n";
    }
    std::cout << "\n";
    
    std::cout << "🗜️  Compression:\n";
    std::cout << "   Types:     " << vectorToString<std::string>(compressions) << "\n\n";
    
    std::cout << "⚡ Benchmarking:\n";
    std::cout << "   Iterations:" << benchmark_iterations << " per test\n";
    std::cout << "   Cloud opt: " << (enable_cloud_optimization ? "Enabled" : "Disabled") << "\n";
    std::cout << "   Validation:" << (enable_validation ? "Enabled" : "Disabled") << "\n";
    std::cout << "   Progress:  " << (show_progress ? "Enabled" : "Disabled") << "\n\n";
    
    // Calculate number of dataset pairs
    int tiled_pairs = 0;
    if (!tile_sizes.empty()) {
        tiled_pairs = tile_sizes.size() * pyramid_levels.size() * compressions.size();
    }
    
    int untiled_pairs = 0;
    if (include_no_tiling) {
        untiled_pairs = compressions.size();
    }
    
    int total_pairs = tiled_pairs + untiled_pairs;
    
    std::cout << "📈 Estimated Workload:\n";
    std::cout << "   Tiled pairs:   " << tiled_pairs << "\n";
    std::cout << "   Untiled pairs: " << untiled_pairs << "\n";
    std::cout << "   Total pairs:   " << total_pairs << "\n";
    std::cout << "   Tests:         " << (total_pairs * 4) << "\n";
    std::cout << "   Total runs:    " << (total_pairs * 4 * benchmark_iterations) << "\n\n";
    
    std::cout << "========================================================\n";
    std::cout << "\nProceed with this configuration? (y/n)\n";
    std::cout << "[Default: yes]\n";
    std::cout << "========================================================\n";
    std::cout << ">> ";
    flushOutput();
    
    std::string confirm;
    std::getline(std::cin, confirm);
    confirm = trim(confirm);
    
    if (confirm.empty() || std::tolower(confirm[0]) == 'y') {
        std::cout << "\n✓ Configuration confirmed!\n";
        flushOutput();
        return true;
    } else {
        std::cout << "\n✗ Configuration cancelled.\n";
        flushOutput();
        return false;
    }
}

} // namespace DialogSystem

std::cout << "Complete DialogSystem namespace loaded (namespace fs declared)." << std::endl;

## Global Variables for Benchmark State

In [ ]:
// Global Variables for Benchmark State - CORRECTED
std::string g_source_file;
std::string g_output_directory;
BenchmarkUtils::BenchmarkConfig g_benchmark_config;  // Fixed namespace
bool g_create_from_heif = false;
bool g_config_ready = false;

// Global dataset pairs (will be set after creation)
std::vector<BenchmarkUtils::DatasetPair> g_dataset_pairs;  // Fixed namespace
bool g_datasets_ready = false;

std::cout << "Global benchmark state variables initialized." << std::endl;

## Main Configuration Entry Point

In [ ]:
// Print compression mapping information
void printCompressionMapping() {
    std::cout << "\n========================================" << std::endl;
    std::cout << "COMPRESSION ALGORITHM MAPPING" << std::endl;
    std::cout << "========================================\n" << std::endl;
    
    std::cout << "COG Format         →  HEIF Format\n";
    std::cout << "──────────────────────────────────────\n";
    std::cout << "NONE               →  UNCOMPRESSED (unci)\n";
    std::cout << "LZW                →  ZLIB(DEFLATE) (unci) *\n";
    std::cout << "DEFLATE            →  DEFLATE (unci)\n";
    std::cout << "ZSTD               →  BROTLI (unci) **\n";
    std::cout << "JPEG               →  JPEG (grid)\n";
    std::cout << "\n";
    std::cout << "* LZW not supported in HEIF, mapped to ZLIB(DEFLATE)\n";
    std::cout << "** ZSTD and BROTLI are both modern high-efficiency\n";
    std::cout << "   compression algorithms, making this a fair comparison\n";
    std::cout << "\n";
    std::cout << "Additional HEIF-only options:\n";
    std::cout << "  HEVC (H.265)     - Video codec, lossless mode\n";
    std::cout << "  AV1              - Video codec, lossless mode\n";
    std::cout << "\n========================================\n" << std::endl;
}

printCompressionMapping();

In [ ]:
// Main configuration function - WITH AGGRESSIVE FLUSHING
bool configureAndStart() {
    // Print a clear separator
    std::cout << "\n\n";
    std::cout << "╔════════════════════════════════════════════════════════╗\n";
    std::cout << "║  STARTING BENCHMARK CONFIGURATION                      ║\n";
    std::cout << "╚════════════════════════════════════════════════════════╝\n";
    std::cout << "\n";
    DialogSystem::forceFlush();
    
    // Clear any pending input
    DialogSystem::clearAllInput();
    
    // Small delay to ensure everything is settled
    std::this_thread::sleep_for(std::chrono::milliseconds(300));
    
    // Show preset menu
    int preset_choice = DialogSystem::PresetConfigurations::showPresetMenu();
    
    std::cout << "\n";
    std::cout << "Preset choice: " << preset_choice << "\n";
    DialogSystem::forceFlush();
    
    std::string g_source_file_temp;
    std::string g_output_directory_temp;
    BenchmarkUtils::BenchmarkConfig g_benchmark_config_temp;
    
    if (preset_choice == 5) {  // Custom
        // Custom configuration via FULL interactive dialog
        std::cout << "\n========================================\n";
        std::cout << "Starting Custom Configuration Dialog\n";
        std::cout << "========================================\n";
        DialogSystem::forceFlush();
        
        DialogSystem::BenchmarkConfigDialog dialog;
        
        if (!dialog.runDialog()) {
            std::cout << "\n✗ Configuration cancelled by user.\n";
            return false;
        }
        
        // Get values from custom dialog
        g_source_file_temp = dialog.getSourceFile();
        g_output_directory_temp = dialog.getOutputDirectory();
        g_benchmark_config_temp = dialog.toBenchmarkConfig();
        
        std::cout << "\n✓ Custom configuration complete!\n";
        
    } else {
        // Use preset configuration
        std::cout << "\n========================================\n";
        std::cout << "Using Preset Configuration\n";
        std::cout << "========================================\n\n";
        DialogSystem::forceFlush();
        
        // Get input/output configuration
        std::cout << "--- Input/Output Configuration ---\n\n";
        DialogSystem::forceFlush();
        
        // Clear input buffer
        DialogSystem::clearAllInput();
        
        std::filesystem::path default_source = DialogSystem::PathUtils::getDefaultSourceFile();
        std::cout << "Source GeoTIFF file path\n";
        std::cout << "[Default: " << DialogSystem::PathUtils::toString(default_source) << "]\n";
        std::cout << ">> ";
        DialogSystem::forceFlush();
        
        std::string input;
        std::getline(std::cin, input);
        
        // Trim whitespace
        size_t first = input.find_first_not_of(" \t\n\r");
        if (first != std::string::npos) {
            size_t last = input.find_last_not_of(" \t\n\r");
            input = input.substr(first, last - first + 1);
        } else {
            input = "";
        }
        
        if (input.empty()) {
            g_source_file_temp = DialogSystem::PathUtils::toString(default_source);
            std::cout << "→ Using default\n";
        } else {
            g_source_file_temp = input;
            std::cout << "→ Using: " << g_source_file_temp << "\n";
        }
        DialogSystem::forceFlush();
        
        std::cout << "\n";
        
        std::filesystem::path default_output = DialogSystem::PathUtils::getDefaultOutputDirectory();
        std::cout << "Output directory\n";
        std::cout << "[Default: " << DialogSystem::PathUtils::toString(default_output) << "]\n";
        std::cout << ">> ";
        DialogSystem::forceFlush();
        
        std::getline(std::cin, input);
        
        // Trim whitespace
        first = input.find_first_not_of(" \t\n\r");
        if (first != std::string::npos) {
            size_t last = input.find_last_not_of(" \t\n\r");
            input = input.substr(first, last - first + 1);
        } else {
            input = "";
        }
        
        if (input.empty()) {
            g_output_directory_temp = DialogSystem::PathUtils::toString(default_output);
            std::cout << "→ Using default\n";
        } else {
            g_output_directory_temp = input;
            std::cout << "→ Using: " << g_output_directory_temp << "\n";
        }
        DialogSystem::forceFlush();
        
        std::cout << "\n";
        
        // Set config based on preset
        switch (preset_choice) {
            case 1:
                g_benchmark_config_temp = DialogSystem::PresetConfigurations::getQuickTestPreset();
                std::cout << "✓ Using Quick Test preset\n";
                break;
            case 2:
                g_benchmark_config_temp = DialogSystem::PresetConfigurations::getStandardPreset();
                std::cout << "✓ Using Standard preset\n";
                break;
            case 3:
                g_benchmark_config_temp = DialogSystem::PresetConfigurations::getComprehensivePreset();
                std::cout << "✓ Using Comprehensive preset\n";
                break;
            case 4:
                g_benchmark_config_temp = DialogSystem::PresetConfigurations::getSimplePreset();
                std::cout << "✓ Using Simple (Untiled Only) preset\n";
                break;
            default:
                g_benchmark_config_temp = DialogSystem::PresetConfigurations::getStandardPreset();
                std::cout << "✓ Using Standard preset (default)\n";
                break;
        }
        DialogSystem::forceFlush();
    }
    
    // ... rest of function remains the same ...
    
    // Validate source file exists (warning only)
    std::filesystem::path source_path = DialogSystem::PathUtils::fromString(g_source_file_temp);
    if (!DialogSystem::PathUtils::exists(source_path)) {
        std::cout << "\n⚠ WARNING: Source file does not exist:\n";
        std::cout << "   " << g_source_file_temp << "\n";
        std::cout << "\nContinue anyway? (y/n) [n]: ";
        DialogSystem::forceFlush();
        
        std::string confirm;
        std::getline(std::cin, confirm);
        
        // Trim
        size_t first = confirm.find_first_not_of(" \t\n\r");
        if (first != std::string::npos) {
            size_t last = confirm.find_last_not_of(" \t\n\r");
            confirm = confirm.substr(first, last - first + 1);
        } else {
            confirm = "";
        }
        
        if (confirm.empty() || std::tolower(confirm[0]) != 'y') {
            std::cout << "\n✗ Configuration cancelled. Please provide valid source file.\n";
            return false;
        }
    }
    
    // Create output directory
    std::filesystem::path output_path = DialogSystem::PathUtils::fromString(g_output_directory_temp);
    if (!DialogSystem::PathUtils::exists(output_path)) {
        std::cout << "\nCreating output directory...\n";
        if (DialogSystem::PathUtils::createDirectories(output_path)) {
            std::cout << "✓ Directory created: " << g_output_directory_temp << "\n";
        } else {
            std::cout << "✗ Failed to create directory: " << g_output_directory_temp << "\n";
            return false;
        }
    }
    
    // Set global variables
    g_source_file = g_source_file_temp;
    g_output_directory = g_output_directory_temp;
    g_benchmark_config = g_benchmark_config_temp;
    g_config_ready = true;
    g_datasets_ready = false;
    
    // Save configuration...
    std::filesystem::path config_file = output_path / "benchmark_config.txt";
    std::ofstream cfg(config_file);
    if (cfg.is_open()) {
        cfg << "# HEIF/GeoHEIF vs COG/GeoTIFF Benchmark Configuration\n";
        cfg << "# Generated: " << std::time(nullptr) << "\n\n";
        cfg << "source_file=" << g_source_file << "\n";
        cfg << "output_directory=" << g_output_directory << "\n";
        cfg << "preset=" << preset_choice << "\n\n";
        cfg << "# Tile Sizes\n";
        cfg << "tile_sizes=";
        for (size_t i = 0; i < g_benchmark_config.tile_sizes.size(); i++) {
            if (i > 0) cfg << ",";
            cfg << g_benchmark_config.tile_sizes[i];
        }
        if (g_benchmark_config.tile_sizes.empty()) {
            cfg << "none";
        }
        cfg << "\n\n";
        cfg << "# Pyramid Levels\n";
        cfg << "pyramid_levels=";
        for (size_t i = 0; i < g_benchmark_config.pyramid_levels.size(); i++) {
            if (i > 0) cfg << ",";
            cfg << g_benchmark_config.pyramid_levels[i];
        }
        cfg << "\n\n";
        cfg << "# Compressions\n";
        cfg << "compressions=";
        for (size_t i = 0; i < g_benchmark_config.compressions.size(); i++) {
            if (i > 0) cfg << ",";
            cfg << g_benchmark_config.compressions[i];
        }
        cfg << "\n\n";
        cfg << "# Settings\n";
        cfg << "benchmark_iterations=" << g_benchmark_config.benchmark_iterations << "\n";
        cfg << "include_no_tiling=" << (g_benchmark_config.include_no_tiling ? "1" : "0") << "\n";
        cfg << "include_no_pyramids=" << (g_benchmark_config.include_no_pyramids ? "1" : "0") << "\n";
        cfg << "enable_cloud_optimization=" << (g_benchmark_config.enable_cloud_optimization ? "1" : "0") << "\n";
        cfg << "enable_validation=" << (g_benchmark_config.enable_validation ? "1" : "0") << "\n";
        cfg << "show_progress=" << (g_benchmark_config.show_progress ? "1" : "0") << "\n";
        cfg.close();
        std::cout << "\n✓ Configuration saved to: " << config_file.string() << "\n";
    }
    
    std::cout << "\n" << std::string(70, '=') << "\n";
    std::cout << "CONFIGURATION COMPLETE\n";
    std::cout << std::string(70, '=') << "\n";
    std::cout << "Source:          " << g_source_file << "\n";
    std::cout << "Output:          " << g_output_directory << "\n";
    std::cout << "Iterations:      " << g_benchmark_config.benchmark_iterations << "\n";
    std::cout << "Include untiled: " << (g_benchmark_config.include_no_tiling ? "Yes" : "No") << "\n";
    std::cout << std::string(70, '=') << "\n\n";
    
    return true;
}

std::cout << "Configuration system ready with AGGRESSIVE output flushing." << std::endl;

# Run configuration and prepare data

## Run Configuration Dialog

Execute the cell below to start the interactive configuration:

In [ ]:
// Run the configuration dialog
if (configureAndStart()) {
    std::cout << "\n✓ Ready to proceed with benchmarking!\n";
} else {
    std::cout << "\n✗ Configuration incomplete. Please run configureAndStart() again.\n";
}

## Create Dataset Pairs with Progress

In [ ]:
// Create Dataset Pairs with Untiled Support - UPDATED
if (!g_config_ready) {
    std::cerr << "ERROR: Configuration not complete!\n";
    std::cerr << "Please run: configureAndStart()\n";
} else {
    std::cout << "\n" << std::string(60, '=') << "\n";
    std::cout << "CREATING BENCHMARK DATASETS\n";
    std::cout << std::string(60, '=') << "\n";
    std::cout << "Source: " << g_source_file << "\n";
    std::cout << "Output: " << g_output_directory << "\n";
    std::cout << std::string(60, '=') << "\n\n";
    
    g_dataset_pairs.clear();
    
    // Calculate total datasets including untiled versions
    int tiled_datasets = g_benchmark_config.tile_sizes.size() * 
                        g_benchmark_config.pyramid_levels.size() * 
                        g_benchmark_config.compressions.size();
    
    int untiled_datasets = 0;
    if (g_benchmark_config.include_no_tiling) {
        untiled_datasets = g_benchmark_config.compressions.size();
    }
    
    int total_datasets = tiled_datasets + untiled_datasets;
    
    std::cout << "Creating " << total_datasets << " dataset pairs...\n";
    if (g_benchmark_config.include_no_tiling) {
        std::cout << "  Tiled: " << tiled_datasets << "\n";
        std::cout << "  Untiled: " << untiled_datasets << "\n";
    }
    std::cout << "\n";
    
    BenchmarkUtils::ProgressBar progress(total_datasets * 2, "Progress", 50, 
                                        g_benchmark_config.show_progress);
    int count = 0;
    int failed_heif = 0;
    
    // Create UNTILED versions first (if enabled)
    if (g_benchmark_config.include_no_tiling) {
        std::cout << "Creating UNTILED dataset pairs...\n";
        
        for (const auto& compression : g_benchmark_config.compressions) {
            std::string suffix = "_" + compression + "_notile_p0";
            
            BenchmarkUtils::DatasetPair pair;
            std::filesystem::path output_path(g_output_directory);
            pair.cog_path = (output_path / ("cog" + suffix + ".tif")).string();
            pair.geoheif_path = (output_path / ("heif" + suffix + ".heic")).string();
            pair.compression = compression;
            pair.tile_size = 0;      // 0 = no tiling
            pair.pyramid_levels = 0; // No pyramids for untiled
            pair.is_tiled = false;
            
            // Create untiled COG (use GTiff driver, not COG driver)
            bool cog_created = COGCreator::createCOG(g_source_file, pair.cog_path,
                                                    compression, 0, 0, false);
            progress.update();
            
            if (cog_created) {
                // Create untiled HEIF with compression parameter - CORRECTED
                bool heif_created = HEIFCreator::createGeoHEIFSimple(
                    g_source_file,      // input_tiff
                    pair.geoheif_path,  // output_heif
                    compression,        // compression (string)
                    false);             // verbose
                
                progress.update();
                
                if (heif_created) {
                    g_dataset_pairs.push_back(pair);
                    count++;
                } else {
                    failed_heif++;
                    std::cout << "\n⚠ Warning: HEIF creation failed for " << suffix << "\n";
                }
            } else {
                progress.update();
                std::cout << "\n⚠ Warning: Failed to create COG for " << suffix << "\n";
            }
        }
    }

    // Create TILED versions
    if (!g_benchmark_config.tile_sizes.empty()) {
        std::cout << "\nCreating TILED dataset pairs...\n";
        
        for (const auto& compression : g_benchmark_config.compressions) {
            for (int tile_size : g_benchmark_config.tile_sizes) {
                for (int pyr_levels : g_benchmark_config.pyramid_levels) {
                    std::string suffix = "_" + compression + 
                                    "_t" + std::to_string(tile_size) +
                                    "_p" + std::to_string(pyr_levels);
                    
                    BenchmarkUtils::DatasetPair pair;
                    std::filesystem::path output_path(g_output_directory);
                    pair.cog_path = (output_path / ("cog" + suffix + ".tif")).string();
                    pair.geoheif_path = (output_path / ("heif" + suffix + ".heic")).string();
                    pair.compression = compression;
                    pair.tile_size = tile_size;
                    pair.pyramid_levels = pyr_levels;
                    pair.is_tiled = true;
                    
                    // Create COG
                    bool cog_created = COGCreator::createCOG(g_source_file, pair.cog_path,
                                                            compression, tile_size, 
                                                            pyr_levels, false);
                    progress.update();
                    
                    if (cog_created) {
                        // Try advanced HEIF creation first
                        bool heif_created = HEIFCreator::createGeoHEIF(
                            g_source_file, pair.geoheif_path, compression,
                            tile_size, pyr_levels, false);
                        
                        // If failed, try simple fallback (with compression parameter)
                        if (!heif_created) {
                            std::cout << "\n[RETRY] Attempting simple HEIF creation for " 
                                    << suffix << "\n";
                            heif_created = HEIFCreator::createGeoHEIFSimple(
                                g_source_file,      // input_tiff
                                pair.geoheif_path,  // output_heif  
                                compression,        // compression (string)
                                false);             // verbose
                        }
                        
                        progress.update();
                        
                        if (heif_created) {
                            g_dataset_pairs.push_back(pair);
                            count++;
                        } else {
                            failed_heif++;
                            std::cout << "\n⚠ Warning: Both HEIF methods failed for " 
                                    << suffix << "\n";
                        }
                    } else {
                        progress.update();
                        std::cout << "\n⚠ Warning: Failed to create COG for " << suffix << "\n";
                    }
                }
            }
        }
    }   
    progress.finish();
    
    g_datasets_ready = !g_dataset_pairs.empty();
    
    std::cout << "\n" << std::string(60, '=') << "\n";
    std::cout << "Dataset Creation Complete\n";
    std::cout << "Created: " << g_dataset_pairs.size() << " dataset pairs\n";
    if (failed_heif > 0) {
        std::cout << "Failed HEIF: " << failed_heif << " datasets\n";
    }
    
    // Summary breakdown
    int untiled_count = 0;
    int tiled_count = 0;
    for (const auto& pair : g_dataset_pairs) {
        if (pair.is_tiled) {
            tiled_count++;
        } else {
            untiled_count++;
        }
    }
    
    if (untiled_count > 0) {
        std::cout << "  Untiled: " << untiled_count << " pairs\n";
    }
    if (tiled_count > 0) {
        std::cout << "  Tiled: " << tiled_count << " pairs\n";
    }
    
    std::cout << std::string(60, '=') << "\n";
    
    if (g_datasets_ready) {
        std::cout << "\n✓ Datasets ready for benchmarking!\n";
    } else {
        std::cout << "\n✗ No datasets were created. Please check errors above.\n";
    }
}

## Validate Created Datasets

In [ ]:
// Validate all created datasets
if (g_datasets_ready && g_benchmark_config.enable_validation) {
    std::cout << "\n" << std::string(60, '=') << "\n";
    std::cout << "DATASET VALIDATION\n";
    std::cout << std::string(60, '=') << "\n\n";
    
    int valid_count = 0;
    int invalid_count = 0;
    
    for (size_t i = 0; i < g_dataset_pairs.size(); i++) {
        const auto& pair = g_dataset_pairs[i];
        
        std::cout << "[" << (i+1) << "/" << g_dataset_pairs.size() << "] ";
        std::cout << "Tile:" << pair.tile_size << " Pyr:" << pair.pyramid_levels 
                  << " Comp:" << pair.compression << "\n";
        
        // Validate COG
        auto cog_result = Validation::validateCOG(pair.cog_path);
        std::cout << "  COG: ";
        if (cog_result.valid) {
            std::cout << "✓";
            valid_count++;
        } else {
            std::cout << "✗ " << cog_result.error_message;
            invalid_count++;
        }
        std::cout << "\n";
        
        // Validate HEIF
        auto heif_result = Validation::validateHEIF(pair.geoheif_path);
        std::cout << "  HEIF: ";
        if (heif_result.valid) {
            std::cout << "✓";
            valid_count++;
        } else {
            std::cout << "✗ " << heif_result.error_message;
            invalid_count++;
        }
        std::cout << "\n\n";
    }
    
    std::cout << std::string(60, '=') << "\n";
    std::cout << "Validation Summary:\n";
    std::cout << "  Valid:   " << valid_count << "\n";
    std::cout << "  Invalid: " << invalid_count << "\n";
    std::cout << std::string(60, '=') << "\n";
} else if (!g_datasets_ready) {
    std::cout << "No datasets to validate. Please create datasets first.\n";
}

# File analysis - main

## File Analysis Functions

### Metadata reading and analysis

#### Data structures and common functions for FileAnalysis

In [ ]:
namespace FileAnalysis {

// Structure to hold HEIF ftyp box information
struct HeifFtypInfo {
    bool valid;
    uint32_t box_size;          // Size of ftyp box
    std::string major_brand;    // 4-character major brand
    uint32_t minor_version;     // Minor version
    std::vector<std::string> compatible_brands;  // Compatible brands
    std::string error_message;
};

// Structure to hold GeoTIFF header information
struct GeoTiffHeaderInfo {
    bool valid;
    bool is_big_tiff;
    bool is_big_endian;
    uint16_t magic_number;      // 42 for standard TIFF, 43 for BigTIFF
    uint64_t ifd_offset;        // Offset to first IFD
    uint32_t header_size;       // Size of TIFF header (8 or 16 bytes)
    std::string error_message;
};

// Track file access operations (open, read, close cycle)
struct FileAccessRecord {
    int access_number;
    uint64_t offset;
    uint64_t length;
    std::string purpose;
    double time_ms;
    bool success;
};

/*
// Tile/Strip data location
struct TiffDataLocation {
    uint32_t index;            // Tile or strip index
    uint64_t offset;           // Offset in file
    uint64_t byte_count;       // Size of tile/strip
    bool is_tile;              // True for tile, false for strip
};
*/

// Enhanced tile/strip location with compression info
struct TiffDataLocation {
    uint32_t index;            // Tile or strip index
    uint64_t offset;           // Offset in file
    uint64_t byte_count;       // Compressed size
    bool is_tile;              // True for tile, false for strip
    std::string compression;   // Compression method for this specific tile
    uint32_t tile_x;          // Tile column index (for tiled images)
    uint32_t tile_y;          // Tile row index (for tiled images)
};


// TIFF IFD Entry
struct TiffIFDEntry {
    uint16_t tag;
    uint16_t type;
    uint64_t count;
    uint64_t value_offset;
    std::string tag_name;
};

/*// TIFF IFD (Image File Directory)
struct TiffIFD {
    uint64_t offset;
    uint64_t entry_count;
    std::vector<TiffIFDEntry> entries;
    uint64_t next_ifd_offset;
    bool is_overview;
    int level;  // 0 = full res, 1+ = overview levels
    
    // Decoded important tags
    uint32_t image_width;
    uint32_t image_height;
    uint32_t tile_width;
    uint32_t tile_height;
    uint32_t rows_per_strip;
    uint16_t samples_per_pixel;
    std::string compression;
    
    // Data locations (all tiles/strips for this IFD)
    std::vector<TiffDataLocation> data_locations;
    bool is_tiled;
    uint32_t tiles_across;
    uint32_t tiles_down;
    uint32_t strips_count;
};*/
// Enhanced IFD with complete tile compression info
struct TiffIFD {
    uint64_t offset;
    uint64_t entry_count;
    std::vector<TiffIFDEntry> entries;
    uint64_t next_ifd_offset;
    bool is_overview;
    int level;  // 0 = full res, 1+ = overview levels
    
    // Decoded important tags
    uint32_t image_width;
    uint32_t image_height;
    uint32_t tile_width;
    uint32_t tile_height;
    uint32_t rows_per_strip;
    uint16_t samples_per_pixel;
    std::string compression;      // Overall compression
    uint16_t compression_code;    // Numeric compression code
    uint16_t predictor;           // Predictor value (if any)
    
    // Data locations (all tiles/strips for this IFD with compression)
    std::vector<TiffDataLocation> data_locations;
    bool is_tiled;
    uint32_t tiles_across;
    uint32_t tiles_down;
    uint32_t strips_count;
    
    // Total data size
    uint64_t total_compressed_bytes;
    uint64_t total_uncompressed_bytes;
};



// Complete GeoTIFF metadata
struct GeoTiffMetadata {
    bool valid;
    std::string error_message;
    
    // Header info
    GeoTiffHeaderInfo header_info;
    
    // All IFDs
    std::vector<TiffIFD> ifds;
    TiffIFD main_ifd;
    std::vector<TiffIFD> overview_ifds;
    
    // File access tracking
    std::vector<FileAccessRecord> access_log;
    uint64_t total_metadata_bytes;
    int total_accesses;
    double total_access_time_ms;
};

// HEIF Box structure
struct HeifBox {
    uint64_t offset;
    uint64_t size;
    std::string type;
    uint64_t header_size;
    uint64_t data_offset;
    uint64_t data_size;
    std::vector<HeifBox> children;
};

/*
// HEIF Item data location
struct HeifItemLocation {
    uint32_t item_id;
    uint64_t base_offset;
    uint64_t extent_offset;
    uint64_t extent_length;
    uint64_t absolute_offset;
};*/

// HEIF Item Location
struct HeifItemLocation {
    uint32_t item_id;
    uint64_t base_offset;
    std::vector<std::pair<uint64_t, uint64_t>> extents;  // (offset, length) pairs
};


/*
// HEIF Item information
struct HeifItemInfo {
    uint32_t item_id;
    std::string item_type;
    std::string item_name;
    bool is_primary;
    bool is_thumbnail;
    uint32_t width;
    uint32_t height;
};
*/
// Enhanced HEIF item location with tile-specific information
struct HeifTileLocation {
    uint32_t tile_index;        // Tile index within the level
    uint32_t tile_x;           // Tile column
    uint32_t tile_y;           // Tile row
    uint64_t offset;           // Absolute byte offset in file
    uint64_t byte_count;       // Compressed size in bytes
    std::string compression;   // Compression method for this tile
};

/*
// Enhanced HEIF item with complete tile information
struct HeifItemInfo {
    uint32_t item_id;
    std::string item_type;
    std::string item_name;
    bool is_primary;
    bool is_thumbnail;
    bool is_tiled;            // True if this is a tiled image (grid/tili/unci)
    uint32_t width;
    uint32_t height;
    uint32_t tile_width;      // 0 if not tiled
    uint32_t tile_height;     // 0 if not tiled
    uint32_t tiles_across;
    uint32_t tiles_down;
    std::string compression;  // Main compression method
    std::string tiling_method; // "grid", "tili", "unci", or "none"
    
    // All tile locations for this item
    std::vector<HeifTileLocation> tile_locations;
    uint64_t total_compressed_bytes;
    uint64_t total_uncompressed_bytes;
    
    // For unci images
    bool is_unci;
    std::string unci_compression; // "none", "deflate", "zlib", "brotli"
};
*/
// HEIF Item Information
struct HeifItemInfo {
    uint32_t item_id;
    std::string item_type;
    uint32_t width;
    uint32_t height;
    bool is_grid;
    uint32_t grid_rows;
    uint32_t grid_columns;
    std::vector<uint32_t> grid_item_ids;
    std::string compression;
};

/*
// Complete HEIF metadata
struct HeifMetadata {
    bool valid;
    std::string error_message;
    
    HeifFtypInfo ftyp_info;
    std::vector<HeifBox> all_boxes;
    HeifBox meta_box;
    HeifBox idat_box;
    HeifBox mdat_box;
    
    std::vector<HeifItemInfo> items;
    std::vector<HeifItemLocation> item_locations;
    uint32_t primary_item_id;
    
    std::vector<FileAccessRecord> access_log;
    uint64_t total_metadata_bytes;
    int total_accesses;
    double total_access_time_ms;
};*/

// HEIF Metadata
struct HeifMetadata {
    bool valid;
    std::string error_message;
    uint64_t total_metadata_bytes;
    double total_parse_time_ms;
    
    std::vector<HeifBox> top_level_boxes;
    HeifBox ftyp_box;
    HeifBox meta_box;
    HeifBox idat_box;
    HeifBox mdat_box;
    
    std::vector<HeifItemInfo> items;
    std::vector<HeifItemLocation> item_locations;
    uint32_t primary_item_id;
};



    
}

#### Header

##### Header reading functions

In [ ]:
namespace FileAnalysis {

// Helper to read big-endian 32-bit integer
uint32_t readBigEndian32(const uint8_t* data) {
    return (static_cast<uint32_t>(data[0]) << 24) |
           (static_cast<uint32_t>(data[1]) << 16) |
           (static_cast<uint32_t>(data[2]) << 8) |
           (static_cast<uint32_t>(data[3]));
}

// Helper to read little-endian 32-bit integer
uint32_t readLittleEndian32(const uint8_t* data) {
    return (static_cast<uint32_t>(data[0])) |
           (static_cast<uint32_t>(data[1]) << 8) |
           (static_cast<uint32_t>(data[2]) << 16) |
           (static_cast<uint32_t>(data[3]) << 24);
}

// Helper to read little-endian 16-bit integer
uint16_t readLittleEndian16(const uint8_t* data) {
    return (static_cast<uint16_t>(data[0])) |
           (static_cast<uint16_t>(data[1]) << 8);
}

// Read and parse HEIF ftyp box
HeifFtypInfo readHeifFtyp(const std::string& filepath) {
    HeifFtypInfo info;
    info.valid = false;
    
    std::ifstream file(filepath, std::ios::binary);
    if (!file.is_open()) {
        info.error_message = "Failed to open file";
        return info;
    }
    
    // Read first 8 bytes (box size + box type)
    uint8_t header[8];
    file.read(reinterpret_cast<char*>(header), 8);
    if (!file) {
        info.error_message = "Failed to read header";
        return info;
    }
    
    // Parse box size (first 4 bytes, big-endian)
    uint32_t box_size = readBigEndian32(header);
    info.box_size = box_size;
    
    // Parse box type (next 4 bytes)
    std::string box_type(reinterpret_cast<char*>(header + 4), 4);
    
    // Check if this is an ftyp box
    if (box_type != "ftyp") {
        info.error_message = "First box is not ftyp, found: " + box_type;
        return info;
    }
    
    // Handle extended size (box_size == 1)
    if (box_size == 1) {
        uint8_t extended_size[8];
        file.read(reinterpret_cast<char*>(extended_size), 8);
        if (!file) {
            info.error_message = "Failed to read extended size";
            return info;
        }
        // Extended size is 64-bit big-endian
        uint64_t large_size = 0;
        for (int i = 0; i < 8; i++) {
            large_size = (large_size << 8) | extended_size[i];
        }
        info.box_size = static_cast<uint32_t>(large_size);  // Truncate for display
    }
    
    // Read ftyp box content (at least 8 more bytes for major brand + minor version)
    uint8_t ftyp_data[8];
    file.read(reinterpret_cast<char*>(ftyp_data), 8);
    if (!file) {
        info.error_message = "Failed to read ftyp data";
        return info;
    }
    
    // Parse major brand (4 bytes)
    info.major_brand = std::string(reinterpret_cast<char*>(ftyp_data), 4);
    
    // Parse minor version (4 bytes, big-endian)
    info.minor_version = readBigEndian32(ftyp_data + 4);
    
    // Parse compatible brands (remaining bytes in the box)
    // Each compatible brand is 4 bytes
    uint32_t remaining_bytes = box_size - 16;  // 8 (header) + 8 (major+minor)
    if (box_size == 1) {
        remaining_bytes = info.box_size - 24;  // Account for extended size
    }
    
    uint32_t num_brands = remaining_bytes / 4;
    for (uint32_t i = 0; i < num_brands && i < 10; i++) {  // Limit to first 10 brands
        uint8_t brand[4];
        file.read(reinterpret_cast<char*>(brand), 4);
        if (!file) break;
        info.compatible_brands.push_back(std::string(reinterpret_cast<char*>(brand), 4));
    }
    
    info.valid = true;
    file.close();
    return info;
}

// Print HEIF ftyp information
void printHeifFtyp(const std::string& filepath) {
    std::cout << "\n" << std::string(70, '=') << "\n";
    std::cout << "HEIF FILE ANALYSIS: " << filepath << "\n";
    std::cout << std::string(70, '=') << "\n";
    
    HeifFtypInfo info = readHeifFtyp(filepath);
    
    if (!info.valid) {
        std::cout << "✗ Error: " << info.error_message << "\n";
        std::cout << std::string(70, '=') << "\n";
        return;
    }
    
    std::cout << "✓ Valid HEIF file\n\n";
    std::cout << "ftyp Box Information:\n";
    std::cout << "  Box Size:       " << info.box_size << " bytes\n";
    std::cout << "  Major Brand:    " << info.major_brand << "\n";
    std::cout << "  Minor Version:  " << info.minor_version << "\n";
    
    if (!info.compatible_brands.empty()) {
        std::cout << "  Compatible Brands:\n";
        for (const auto& brand : info.compatible_brands) {
            std::cout << "    - " << brand << "\n";
        }
    }
    
    // Interpret common brands
    std::cout << "\nBrand Interpretation:\n";
    if (info.major_brand == "heic" || info.major_brand == "heix") {
        std::cout << "  Format: HEIF image (HEVC/H.265 encoded)\n";
    } else if (info.major_brand == "avif") {
        std::cout << "  Format: AVIF image (AV1 encoded)\n";
    } else if (info.major_brand == "mif1" || info.major_brand == "msf1") {
        std::cout << "  Format: HEIF base format\n";
    } else if (info.major_brand == "jpeg") {
        std::cout << "  Format: JPEG in HEIF container\n";
    } else {
        std::cout << "  Format: HEIF variant (" << info.major_brand << ")\n";
    }
    
    // Check for specific features
    bool has_miaf = false;
    bool has_1pic = false;
    for (const auto& brand : info.compatible_brands) {
        if (brand == "miaf") has_miaf = true;
        if (brand == "1pic") has_1pic = true;
    }
    
    if (has_miaf) {
        std::cout << "  ✓ MIAF compliant (Multi-Image Application Format)\n";
    }
    if (has_1pic) {
        std::cout << "  ✓ Single image HEIF\n";
    }
    
    std::cout << std::string(70, '=') << "\n";
}

// Read and parse GeoTIFF/COG header
GeoTiffHeaderInfo readGeoTiffHeader(const std::string& filepath) {
    GeoTiffHeaderInfo info;
    info.valid = false;
    info.is_big_tiff = false;
    info.is_big_endian = false;
    info.header_size = 8;
    
    std::ifstream file(filepath, std::ios::binary);
    if (!file.is_open()) {
        info.error_message = "Failed to open file";
        return info;
    }
    
    // Read first 16 bytes (enough for BigTIFF header)
    uint8_t header[16];
    file.read(reinterpret_cast<char*>(header), 16);
    if (!file) {
        info.error_message = "Failed to read header";
        return info;
    }
    
    // Check byte order (first 2 bytes)
    if (header[0] == 0x49 && header[1] == 0x49) {
        // "II" = Little-endian (Intel)
        info.is_big_endian = false;
    } else if (header[0] == 0x4D && header[1] == 0x4D) {
        // "MM" = Big-endian (Motorola)
        info.is_big_endian = true;
    } else {
        info.error_message = "Invalid byte order marker";
        return info;
    }
    
    // Read magic number (bytes 2-3)
    if (info.is_big_endian) {
        info.magic_number = (static_cast<uint16_t>(header[2]) << 8) | header[3];
    } else {
        info.magic_number = readLittleEndian16(header + 2);
    }
    
    // Check if standard TIFF (42) or BigTIFF (43)
    if (info.magic_number == 42) {
        // Standard TIFF
        info.is_big_tiff = false;
        info.header_size = 8;
        
        // Read IFD offset (bytes 4-7, 32-bit)
        if (info.is_big_endian) {
            info.ifd_offset = readBigEndian32(header + 4);
        } else {
            info.ifd_offset = readLittleEndian32(header + 4);
        }
        
    } else if (info.magic_number == 43) {
        // BigTIFF
        info.is_big_tiff = true;
        info.header_size = 16;
        
        // Byte 4-5: Bytesize of offsets (should be 8)
        uint16_t offset_size;
        if (info.is_big_endian) {
            offset_size = (static_cast<uint16_t>(header[4]) << 8) | header[5];
        } else {
            offset_size = readLittleEndian16(header + 4);
        }
        
        // Byte 6-7: Always 0
        // Bytes 8-15: IFD offset (64-bit)
        if (info.is_big_endian) {
            info.ifd_offset = 0;
            for (int i = 8; i < 16; i++) {
                info.ifd_offset = (info.ifd_offset << 8) | header[i];
            }
        } else {
            info.ifd_offset = 0;
            for (int i = 15; i >= 8; i--) {
                info.ifd_offset = (info.ifd_offset << 8) | header[i];
            }
        }
        
    } else {
        info.error_message = "Invalid magic number: " + std::to_string(info.magic_number);
        return info;
    }
    
    info.valid = true;
    file.close();
    return info;
}

// Print GeoTIFF/COG header information
void printGeoTiffHeader(const std::string& filepath) {
    std::cout << "\n" << std::string(70, '=') << "\n";
    std::cout << "GEOTIFF/COG FILE ANALYSIS: " << filepath << "\n";
    std::cout << std::string(70, '=') << "\n";
    
    GeoTiffHeaderInfo info = readGeoTiffHeader(filepath);
    
    if (!info.valid) {
        std::cout << "✗ Error: " << info.error_message << "\n";
        std::cout << std::string(70, '=') << "\n";
        return;
    }
    
    std::cout << "✓ Valid TIFF file\n\n";
    std::cout << "TIFF Header Information:\n";
    std::cout << "  Header Size:    " << info.header_size << " bytes\n";
    std::cout << "  Byte Order:     " << (info.is_big_endian ? "Big-endian (MM)" : "Little-endian (II)") << "\n";
    std::cout << "  Magic Number:   " << info.magic_number << "\n";
    std::cout << "  Format:         " << (info.is_big_tiff ? "BigTIFF" : "Standard TIFF") << "\n";
    std::cout << "  IFD Offset:     " << info.ifd_offset << " bytes\n";
    
    // The IFD offset tells us where the metadata (Image File Directory) starts
    std::cout << "\nMetadata Structure:\n";
    std::cout << "  First IFD at:   " << info.ifd_offset << " bytes from start\n";
    std::cout << "  Header region:  0-" << info.header_size << " bytes\n";
    
    // For COG, the IFD should be near the beginning for cloud optimization
    if (info.ifd_offset < 1024) {
        std::cout << "  ✓ IFD is close to start (likely COG-optimized)\n";
    } else if (info.ifd_offset < 16384) {
        std::cout << "  ⚠ IFD is moderately far from start\n";
    } else {
        std::cout << "  ✗ IFD is far from start (NOT cloud-optimized)\n";
    }
    
    std::cout << std::string(70, '=') << "\n";
}

// Analyze both file types in a dataset pair
void analyzeDatasetPair(const BenchmarkUtils::DatasetPair& pair) {
    std::cout << "\n" << std::string(70, '=') << "\n";
    std::cout << "DATASET PAIR ANALYSIS\n";
    std::cout << std::string(70, '=') << "\n";
    std::cout << "Configuration:\n";
    std::cout << "  Compression:    " << pair.compression << "\n";
    std::cout << "  Tile Size:      " << (pair.tile_size > 0 ? std::to_string(pair.tile_size) : "untiled") << "\n";
    std::cout << "  Pyramid Levels: " << pair.pyramid_levels << "\n";
    std::cout << "  Tiled:          " << (pair.is_tiled ? "Yes" : "No") << "\n";
    std::cout << std::string(70, '=') << "\n";
    
    // Analyze COG
    printGeoTiffHeader(pair.cog_path);
    
    // Analyze HEIF
    printHeifFtyp(pair.geoheif_path);
    
    // Compare metadata overhead
    std::cout << "\n" << std::string(70, '=') << "\n";
    std::cout << "METADATA OVERHEAD COMPARISON\n";
    std::cout << std::string(70, '=') << "\n";
    
    long cog_size = BenchmarkUtils::getFileSize(pair.cog_path);
    long heif_size = BenchmarkUtils::getFileSize(pair.geoheif_path);
    
    GeoTiffHeaderInfo cog_info = readGeoTiffHeader(pair.cog_path);
    HeifFtypInfo heif_info = readHeifFtyp(pair.geoheif_path);
    
    std::cout << "File Sizes:\n";
    std::cout << "  COG:   " << std::fixed << std::setprecision(2) << (cog_size / 1024.0 / 1024.0) << " MB\n";
    std::cout << "  HEIF:  " << std::fixed << std::setprecision(2) << (heif_size / 1024.0 / 1024.0) << " MB\n";
    std::cout << "  Ratio: " << std::fixed << std::setprecision(3) << (static_cast<double>(heif_size) / cog_size) << "\n";
    
    std::cout << "\nInitial Metadata:\n";
    std::cout << "  COG Header:     " << cog_info.header_size << " bytes\n";
    std::cout << "  HEIF ftyp box:  " << heif_info.box_size << " bytes\n";
    
    std::cout << "\nCloud Optimization:\n";
    std::cout << "  COG IFD offset: " << cog_info.ifd_offset << " bytes\n";
    std::cout << "  HEIF ftyp size: " << heif_info.box_size << " bytes (first box)\n";
    
    if (cog_info.ifd_offset < heif_info.box_size) {
        std::cout << "  → COG has faster metadata access\n";
    } else {
        std::cout << "  → HEIF has more compact initial metadata\n";
    }
    
    std::cout << std::string(70, '=') << "\n";
}

} // namespace FileAnalysis

std::cout << "File analysis functions loaded (HEIF ftyp and GeoTIFF header parsing)." << std::endl;

##### Examples to analyze header

In [ ]:
// Example usage: Analyze a single HEIF file
FileAnalysis::printHeifFtyp("benchmark_output/heif_DEFLATE_t256_p3.heic");

// Example usage: Analyze a single COG file
FileAnalysis::printGeoTiffHeader("benchmark_output/cog_DEFLATE_t256_p3.tif");

// Example usage: Analyze a complete dataset pair
if (g_datasets_ready && !g_dataset_pairs.empty()) {
    FileAnalysis::analyzeDatasetPair(g_dataset_pairs[0]);
}

// Example usage: Analyze all dataset pairs
if (g_datasets_ready) {
    for (size_t i = 0; i < g_dataset_pairs.size(); i++) {
        std::cout << "\n\n[PAIR " << (i+1) << "/" << g_dataset_pairs.size() << "]\n";
        FileAnalysis::analyzeDatasetPair(g_dataset_pairs[i]);
    }
}

#### Metadata

##### Metadata functions

In [ ]:
namespace FileAnalysis {



// Helper: Read bytes from file (complete open/read/close cycle)
FileAccessRecord readFileSegment(const std::string& filepath, uint64_t offset, 
                                 uint8_t* buffer, size_t length, 
                                 int access_number, const std::string& purpose) {
    FileAccessRecord record;
    record.access_number = access_number;
    record.offset = offset;
    record.length = length;
    record.purpose = purpose;
    record.success = false;
    
    BenchmarkUtils::Timer timer;
    
    std::ifstream file(filepath, std::ios::binary);
    if (!file.is_open()) {
        record.time_ms = timer.elapsed_ms();
        return record;
    }
    
    file.seekg(offset, std::ios::beg);
    file.read(reinterpret_cast<char*>(buffer), length);
    
    if (file.good() && file.gcount() == static_cast<std::streamsize>(length)) {
        record.success = true;
    }
    
    file.close();
    record.time_ms = timer.elapsed_ms();
    return record;
}

// Helper: Get TIFF tag name
std::string getTiffTagName(uint16_t tag) {
    static const std::map<uint16_t, std::string> tag_names = {
        {254, "NewSubfileType"}, {256, "ImageWidth"}, {257, "ImageHeight"},
        {258, "BitsPerSample"}, {259, "Compression"}, {262, "PhotometricInterpretation"},
        {273, "StripOffsets"}, {277, "SamplesPerPixel"}, {278, "RowsPerStrip"},
        {279, "StripByteCounts"}, {282, "XResolution"}, {283, "YResolution"},
        {284, "PlanarConfiguration"}, {296, "ResolutionUnit"},
        {322, "TileWidth"}, {323, "TileHeight"}, {324, "TileOffsets"}, {325, "TileByteCounts"},
        {339, "SampleFormat"}, {33550, "ModelPixelScaleTag"}, {33922, "ModelTiepointTag"},
        {34735, "GeoKeyDirectoryTag"}, {34736, "GeoDoubleParamsTag"}, {34737, "GeoAsciiParamsTag"},
        {42112, "GDAL_METADATA"}, {42113, "GDAL_NODATA"}
    };
    
    auto it = tag_names.find(tag);
    return it != tag_names.end() ? it->second : "Tag_" + std::to_string(tag);
}

// Read all GeoTIFF metadata with multiple accesses
GeoTiffMetadata readGeoTiffMetadata(const std::string& filepath, bool verbose = true) {
    GeoTiffMetadata meta;
    meta.valid = false;
    meta.total_metadata_bytes = 0;
    meta.total_accesses = 0;
    meta.total_access_time_ms = 0;
    
    if (verbose) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "READING GEOTIFF METADATA: " << filepath << "\n";
        std::cout << std::string(80, '=') << "\n";
    }
    
    // === ACCESS 1: Read TIFF header (16 bytes) ===
    uint8_t header[16];
    FileAccessRecord access1 = readFileSegment(filepath, 0, header, 16, 1, "Read TIFF header");
    meta.access_log.push_back(access1);
    meta.total_accesses++;
    meta.total_access_time_ms += access1.time_ms;
    meta.total_metadata_bytes += access1.length;
    
    if (verbose) {
        std::cout << "\nAccess 1: " << access1.purpose << "\n";
        std::cout << "  Offset: " << access1.offset << " bytes\n";
        std::cout << "  Length: " << access1.length << " bytes\n";
        std::cout << "  Time:   " << std::fixed << std::setprecision(3) << access1.time_ms << " ms\n";
    }
    
    if (!access1.success) {
        meta.error_message = "Failed to read header";
        return meta;
    }
    
    // Parse header
    bool is_big_endian = (header[0] == 0x4D && header[1] == 0x4D);
    uint16_t magic = is_big_endian ? ((header[2] << 8) | header[3]) : readLittleEndian16(header + 2);
    bool is_big_tiff = (magic == 43);
    
    meta.header_info.is_big_endian = is_big_endian;
    meta.header_info.is_big_tiff = is_big_tiff;
    meta.header_info.magic_number = magic;
    meta.header_info.valid = (magic == 42 || magic == 43);
    meta.header_info.header_size = is_big_tiff ? 16 : 8;
    
    if (verbose) {
        std::cout << "  Result: " << (is_big_endian ? "Big-endian" : "Little-endian") 
                  << ", " << (is_big_tiff ? "BigTIFF" : "Standard TIFF") << "\n";
    }
    
    if (!meta.header_info.valid) {
        meta.error_message = "Invalid TIFF magic number";
        return meta;
    }
    
    // Get first IFD offset
    uint64_t first_ifd_offset;
    if (is_big_tiff) {
        if (is_big_endian) {
            first_ifd_offset = 0;
            for (int i = 8; i < 16; i++) {
                first_ifd_offset = (first_ifd_offset << 8) | header[i];
            }
        } else {
            first_ifd_offset = 0;
            for (int i = 15; i >= 8; i--) {
                first_ifd_offset = (first_ifd_offset << 8) | header[i];
            }
        }
    } else {
        first_ifd_offset = is_big_endian ? readBigEndian32(header + 4) : readLittleEndian32(header + 4);
    }
    meta.header_info.ifd_offset = first_ifd_offset;
    
    if (verbose) {
        std::cout << "  First IFD offset: " << first_ifd_offset << " bytes\n";
    }
    
    // === ACCESS 2+: Read each IFD ===
    uint64_t current_ifd_offset = first_ifd_offset;
    int ifd_level = 0;
    
    while (current_ifd_offset != 0 && ifd_level < 20) {
        meta.total_accesses++;
        int access_num = meta.total_accesses;
        
        // Estimate IFD size (we'll read a generous chunk)
        size_t ifd_read_size = is_big_tiff ? 8192 : 4096;  // Should cover most IFDs
        std::vector<uint8_t> ifd_buffer(ifd_read_size);
        
        FileAccessRecord ifd_access = readFileSegment(filepath, current_ifd_offset,
                                                      ifd_buffer.data(), ifd_read_size,
                                                      access_num, 
                                                      "Read IFD " + std::to_string(ifd_level));
        meta.access_log.push_back(ifd_access);
        meta.total_access_time_ms += ifd_access.time_ms;
        
        if (!ifd_access.success) break;
        
        // Parse IFD
        TiffIFD ifd;
        ifd.offset = current_ifd_offset;
        ifd.level = ifd_level;
        ifd.is_overview = false;
        ifd.image_width = 0;
        ifd.image_height = 0;
        ifd.tile_width = 0;
        ifd.tile_height = 0;
        ifd.rows_per_strip = 0;
        ifd.samples_per_pixel = 1;
        ifd.is_tiled = false;
        
        uint8_t* ifd_data = ifd_buffer.data();
        size_t offset_in_buffer = 0;
        
        // Read entry count
        if (is_big_tiff) {
            if (is_big_endian) {
                ifd.entry_count = 0;
                for (int i = 0; i < 8; i++) {
                    ifd.entry_count = (ifd.entry_count << 8) | ifd_data[i];
                }
            } else {
                ifd.entry_count = 0;
                for (int i = 7; i >= 0; i--) {
                    ifd.entry_count = (ifd.entry_count << 8) | ifd_data[i];
                }
            }
            offset_in_buffer = 8;
        } else {
            ifd.entry_count = is_big_endian ? 
                ((ifd_data[0] << 8) | ifd_data[1]) : readLittleEndian16(ifd_data);
            offset_in_buffer = 2;
        }
        
        size_t entry_size = is_big_tiff ? 20 : 12;
        size_t actual_ifd_size = (is_big_tiff ? 8 : 2) + (ifd.entry_count * entry_size) + (is_big_tiff ? 8 : 4);
        meta.total_metadata_bytes += actual_ifd_size;
        
        if (verbose) {
            std::cout << "\nAccess " << access_num << ": " << ifd_access.purpose << "\n";
            std::cout << "  Offset: " << ifd_access.offset << " bytes\n";
            std::cout << "  Length: " << actual_ifd_size << " bytes (used of " << ifd_read_size << " read)\n";
            std::cout << "  Time:   " << std::fixed << std::setprecision(3) << ifd_access.time_ms << " ms\n";
            std::cout << "  Entries: " << ifd.entry_count << "\n";
        }
        
        // Parse all entries
        for (uint64_t i = 0; i < ifd.entry_count && i < 1000; i++) {
            uint8_t* entry_data = ifd_data + offset_in_buffer;
            TiffIFDEntry entry;
            
            if (is_big_tiff) {
                entry.tag = is_big_endian ? ((entry_data[0] << 8) | entry_data[1]) : readLittleEndian16(entry_data);
                entry.type = is_big_endian ? ((entry_data[2] << 8) | entry_data[3]) : readLittleEndian16(entry_data + 2);
                entry.count = is_big_endian ? 
                    ((uint64_t)entry_data[4] << 56) | ((uint64_t)entry_data[5] << 48) |
                    ((uint64_t)entry_data[6] << 40) | ((uint64_t)entry_data[7] << 32) |
                    ((uint64_t)entry_data[8] << 24) | ((uint64_t)entry_data[9] << 16) |
                    ((uint64_t)entry_data[10] << 8) | entry_data[11] :
                    entry_data[4] | ((uint64_t)entry_data[5] << 8) | ((uint64_t)entry_data[6] << 16) |
                    ((uint64_t)entry_data[7] << 24) | ((uint64_t)entry_data[8] << 32) | 
                    ((uint64_t)entry_data[9] << 40) | ((uint64_t)entry_data[10] << 48) | 
                    ((uint64_t)entry_data[11] << 56);
                entry.value_offset = is_big_endian ? 
                    ((uint64_t)entry_data[12] << 56) | ((uint64_t)entry_data[13] << 48) |
                    ((uint64_t)entry_data[14] << 40) | ((uint64_t)entry_data[15] << 32) |
                    ((uint64_t)entry_data[16] << 24) | ((uint64_t)entry_data[17] << 16) |
                    ((uint64_t)entry_data[18] << 8) | entry_data[19] :
                    entry_data[12] | ((uint64_t)entry_data[13] << 8) | ((uint64_t)entry_data[14] << 16) |
                    ((uint64_t)entry_data[15] << 24) | ((uint64_t)entry_data[16] << 32) | 
                    ((uint64_t)entry_data[17] << 40) | ((uint64_t)entry_data[18] << 48) | 
                    ((uint64_t)entry_data[19] << 56);
            } else {
                entry.tag = is_big_endian ? ((entry_data[0] << 8) | entry_data[1]) : readLittleEndian16(entry_data);
                entry.type = is_big_endian ? ((entry_data[2] << 8) | entry_data[3]) : readLittleEndian16(entry_data + 2);
                entry.count = is_big_endian ? readBigEndian32(entry_data + 4) : readLittleEndian32(entry_data + 4);
                entry.value_offset = is_big_endian ? readBigEndian32(entry_data + 8) : readLittleEndian32(entry_data + 8);
            }
            
            entry.tag_name = getTiffTagName(entry.tag);
            ifd.entries.push_back(entry);
            
            // Decode critical tags
            switch (entry.tag) {
                case 254: if (entry.value_offset & 1) ifd.is_overview = true; break;
                case 256: ifd.image_width = entry.value_offset; break;
                case 257: ifd.image_height = entry.value_offset; break;
                case 277: ifd.samples_per_pixel = entry.value_offset; break;
                case 278: ifd.rows_per_strip = entry.value_offset; break;
                case 322: ifd.tile_width = entry.value_offset; ifd.is_tiled = true; break;
                case 323: ifd.tile_height = entry.value_offset; ifd.is_tiled = true; break;
                case 259: {
                    static const std::map<uint32_t, std::string> compressions = {
                        {1, "None"}, {5, "LZW"}, {7, "JPEG"}, {8, "Deflate"},
                        {32946, "Deflate"}, {50000, "ZSTD"}
                    };
                    auto it = compressions.find(entry.value_offset);
                    ifd.compression = it != compressions.end() ? it->second : "Unknown";
                    break;
                }
            }
            
            offset_in_buffer += entry_size;
        }
        
        // Calculate tile/strip layout
        if (ifd.is_tiled && ifd.tile_width > 0 && ifd.tile_height > 0) {
            ifd.tiles_across = (ifd.image_width + ifd.tile_width - 1) / ifd.tile_width;
            ifd.tiles_down = (ifd.image_height + ifd.tile_height - 1) / ifd.tile_height;
        } else if (ifd.rows_per_strip > 0) {
            ifd.strips_count = (ifd.image_height + ifd.rows_per_strip - 1) / ifd.rows_per_strip;
        }
        
        // Read next IFD offset
        uint8_t* next_offset_data = ifd_data + offset_in_buffer;
        if (is_big_tiff) {
            ifd.next_ifd_offset = is_big_endian ?
                ((uint64_t)next_offset_data[0] << 56) | ((uint64_t)next_offset_data[1] << 48) |
                ((uint64_t)next_offset_data[2] << 40) | ((uint64_t)next_offset_data[3] << 32) |
                ((uint64_t)next_offset_data[4] << 24) | ((uint64_t)next_offset_data[5] << 16) |
                ((uint64_t)next_offset_data[6] << 8) | next_offset_data[7] :
                next_offset_data[0] | ((uint64_t)next_offset_data[1] << 8) | 
                ((uint64_t)next_offset_data[2] << 16) | ((uint64_t)next_offset_data[3] << 24) |
                ((uint64_t)next_offset_data[4] << 32) | ((uint64_t)next_offset_data[5] << 40) |
                ((uint64_t)next_offset_data[6] << 48) | ((uint64_t)next_offset_data[7] << 56);
        } else {
            ifd.next_ifd_offset = is_big_endian ? 
                readBigEndian32(next_offset_data) : readLittleEndian32(next_offset_data);
        }
        
        if (verbose) {
            std::cout << "  Image: " << ifd.image_width << "x" << ifd.image_height;
            if (ifd.is_tiled) {
                std::cout << ", Tiles: " << ifd.tiles_across << "x" << ifd.tiles_down 
                          << " (" << ifd.tile_width << "x" << ifd.tile_height << ")";
            } else {
                std::cout << ", Strips: " << ifd.strips_count;
            }
            std::cout << ", Compression: " << ifd.compression;
            if (ifd.is_overview) std::cout << " [OVERVIEW]";
            std::cout << "\n";
            std::cout << "  Next IFD: " << (ifd.next_ifd_offset ? std::to_string(ifd.next_ifd_offset) : "none") << "\n";
        }
        
        meta.ifds.push_back(ifd);
        if (ifd_level == 0) {
            meta.main_ifd = ifd;
        } else if (ifd.is_overview) {
            meta.overview_ifds.push_back(ifd);
        }
        
        current_ifd_offset = ifd.next_ifd_offset;
        ifd_level++;
    }
    
    // === ACCESS N+: Read tile/strip offset and bytecount arrays for each IFD ===
    for (size_t ifd_idx = 0; ifd_idx < meta.ifds.size(); ifd_idx++) {
        auto& ifd = meta.ifds[ifd_idx];
        
        // Find TileOffsets/StripOffsets and TileByteCounts/StripByteCounts tags
        TiffIFDEntry* offsets_entry = nullptr;
        TiffIFDEntry* bytecounts_entry = nullptr;
        
        for (auto& entry : ifd.entries) {
            if (entry.tag == 324 || entry.tag == 273) {  // TileOffsets or StripOffsets
                offsets_entry = &entry;
            }
            if (entry.tag == 325 || entry.tag == 279) {  // TileByteCounts or StripByteCounts
                bytecounts_entry = &entry;
            }
        }
        
        if (offsets_entry && bytecounts_entry) {
            // Determine if these arrays are stored externally
            size_t type_sizes[] = {0, 1, 1, 2, 4, 8, 1, 1, 2, 4, 8, 4, 8};
            size_t offset_type_size = offsets_entry->type < 13 ? type_sizes[offsets_entry->type] : 4;
            size_t bytecount_type_size = bytecounts_entry->type < 13 ? type_sizes[bytecounts_entry->type] : 4;
            
            size_t offsets_array_size = offsets_entry->count * offset_type_size;
            size_t bytecounts_array_size = bytecounts_entry->count * bytecount_type_size;
            
            size_t inline_size = is_big_tiff ? 8 : 4;
            bool offsets_external = (offsets_array_size > inline_size);
            bool bytecounts_external = (bytecounts_array_size > inline_size);
            
            if (offsets_external && bytecounts_external) {
                meta.total_accesses++;
                int access_num = meta.total_accesses;
                
                // Read both arrays (they're usually contiguous or close)
                uint64_t min_offset = std::min(offsets_entry->value_offset, bytecounts_entry->value_offset);
                uint64_t max_offset = std::max(offsets_entry->value_offset + offsets_array_size,
                                               bytecounts_entry->value_offset + bytecounts_array_size);
                size_t arrays_size = max_offset - min_offset;
                
                std::vector<uint8_t> arrays_buffer(arrays_size);
                FileAccessRecord arrays_access = readFileSegment(filepath, min_offset,
                                                                 arrays_buffer.data(), arrays_size,
                                                                 access_num,
                                                                 "Read tile/strip arrays for IFD " + std::to_string(ifd_idx));
                meta.access_log.push_back(arrays_access);
                meta.total_access_time_ms += arrays_access.time_ms;
                meta.total_metadata_bytes += arrays_size;
                
                if (verbose) {
                    std::cout << "\nAccess " << access_num << ": " << arrays_access.purpose << "\n";
                    std::cout << "  Offset: " << arrays_access.offset << " bytes\n";
                    std::cout << "  Length: " << arrays_access.length << " bytes\n";
                    std::cout << "  Time:   " << std::fixed << std::setprecision(3) << arrays_access.time_ms << " ms\n";
                }
                
                if (arrays_access.success) {
                    // Parse offsets array
                    std::vector<uint64_t> offsets;
                    uint64_t offsets_local = offsets_entry->value_offset - min_offset;
                    for (uint64_t i = 0; i < offsets_entry->count && i < 100000; i++) {
                        uint64_t val = 0;
                        size_t idx = offsets_local + (i * offset_type_size);
                        if (idx + offset_type_size <= arrays_size) {
                            if (offset_type_size == 4) {
                                val = is_big_endian ? 
                                    readBigEndian32(arrays_buffer.data() + idx) :
                                    readLittleEndian32(arrays_buffer.data() + idx);
                            } else if (offset_type_size == 8) {
                                if (is_big_endian) {
                                    for (int j = 0; j < 8; j++) {
                                        val = (val << 8) | arrays_buffer[idx + j];
                                    }
                                } else {
                                    for (int j = 7; j >= 0; j--) {
                                        val = (val << 8) | arrays_buffer[idx + j];
                                    }
                                }
                            }
                            offsets.push_back(val);
                        }
                    }
                    
                    // Parse bytecounts array
                    std::vector<uint64_t> bytecounts;
                    uint64_t bytecounts_local = bytecounts_entry->value_offset - min_offset;
                    for (uint64_t i = 0; i < bytecounts_entry->count && i < 100000; i++) {
                        uint64_t val = 0;
                        size_t idx = bytecounts_local + (i * bytecount_type_size);
                        if (idx + bytecount_type_size <= arrays_size) {
                            if (bytecount_type_size == 4) {
                                val = is_big_endian ?
                                    readBigEndian32(arrays_buffer.data() + idx) :
                                    readLittleEndian32(arrays_buffer.data() + idx);
                            } else if (bytecount_type_size == 8) {
                                if (is_big_endian) {
                                    for (int j = 0; j < 8; j++) {
                                        val = (val << 8) | arrays_buffer[idx + j];
                                    }
                                } else {
                                    for (int j = 7; j >= 0; j--) {
                                        val = (val << 8) | arrays_buffer[idx + j];
                                    }
                                }
                            }
                            bytecounts.push_back(val);
                        }
                    }
                    
                    // Create data location entries
                    for (size_t i = 0; i < offsets.size() && i < bytecounts.size(); i++) {
                        TiffDataLocation loc;
                        loc.index = i;
                        loc.offset = offsets[i];
                        loc.byte_count = bytecounts[i];
                        loc.is_tile = ifd.is_tiled;
                        ifd.data_locations.push_back(loc);
                    }
                    
                    if (verbose) {
                        std::cout << "  Parsed " << ifd.data_locations.size() << " data locations\n";
                        // Show first few and last few
                        int show_count = std::min(3, (int)ifd.data_locations.size());
                        for (int i = 0; i < show_count; i++) {
                            const auto& loc = ifd.data_locations[i];
                            std::cout << "    [" << i << "] offset=" << loc.offset 
                                      << ", bytes=" << loc.byte_count << "\n";
                        }
                        if (ifd.data_locations.size() > 6) {
                            std::cout << "    ... (" << (ifd.data_locations.size() - 6) << " more) ...\n";
                            for (size_t i = ifd.data_locations.size() - 3; i < ifd.data_locations.size(); i++) {
                                const auto& loc = ifd.data_locations[i];
                                std::cout << "    [" << i << "] offset=" << loc.offset 
                                          << ", bytes=" << loc.byte_count << "\n";
                            }
                        } else if (show_count < (int)ifd.data_locations.size()) {
                            for (size_t i = show_count; i < ifd.data_locations.size(); i++) {
                                const auto& loc = ifd.data_locations[i];
                                std::cout << "    [" << i << "] offset=" << loc.offset 
                                          << ", bytes=" << loc.byte_count << "\n";
                            }
                        }
                    }
                }
            }
        }
    }
    
    meta.valid = true;
    
    // === FINAL SUMMARY ===
    if (verbose) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "METADATA READING COMPLETE\n";
        std::cout << std::string(80, '=') << "\n";
        std::cout << "Total file accesses: " << meta.total_accesses << "\n";
        std::cout << "Total metadata bytes: " << meta.total_metadata_bytes << " bytes ("
                  << std::fixed << std::setprecision(2) << (meta.total_metadata_bytes / 1024.0) << " KB)\n";
        std::cout << "Total access time: " << std::fixed << std::setprecision(3) 
                  << meta.total_access_time_ms << " ms\n";
        std::cout << "\nStructure Summary:\n";
        std::cout << "  Total IFDs: " << meta.ifds.size() << "\n";
        std::cout << "  Main image: " << meta.main_ifd.image_width << "x" << meta.main_ifd.image_height << "\n";
        std::cout << "  Overview IFDs: " << meta.overview_ifds.size() << "\n";
        
        uint64_t total_tiles = 0;
        for (const auto& ifd : meta.ifds) {
            total_tiles += ifd.data_locations.size();
        }
        std::cout << "  Total tiles/strips across all levels: " << total_tiles << "\n";
        
        std::cout << "\nAccess Log Summary:\n";
        for (const auto& access : meta.access_log) {
            std::cout << "  [" << access.access_number << "] " << access.purpose 
                      << ": offset=" << access.offset << ", length=" << access.length
                      << ", time=" << std::fixed << std::setprecision(3) << access.time_ms << "ms\n";
        }
        std::cout << std::string(80, '=') << "\n";
    }
    
    return meta;
}

/*
// Read all HEIF metadata (similar structure - showing abbreviated version)

// Complete HEIF metadata reading with detailed access tracking
HeifMetadata readHeifMetadata(const std::string& filepath, bool verbose = true) {
    HeifMetadata meta;
    meta.valid = false;
    meta.total_metadata_bytes = 0;
    meta.total_accesses = 0;
    meta.total_access_time_ms = 0;
    meta.primary_item_id = 0;
    
    if (verbose) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "READING HEIF METADATA: " << filepath << "\n";
        std::cout << std::string(80, '=') << "\n";
    }
    
    // === ACCESS 1: Read ftyp box (first 64 bytes to get complete ftyp) ===
    uint8_t ftyp_buffer[256];
    FileAccessRecord access1 = readFileSegment(filepath, 0, ftyp_buffer, 256, 1, "Read ftyp box");
    meta.access_log.push_back(access1);
    meta.total_accesses++;
    meta.total_access_time_ms += access1.time_ms;
    
    if (!access1.success) {
        meta.error_message = "Failed to read ftyp";
        return meta;
    }
    
    // Parse ftyp box
    uint32_t ftyp_size = readBigEndian32(ftyp_buffer);
    std::string box_type(reinterpret_cast<char*>(ftyp_buffer + 4), 4);
    
    if (box_type != "ftyp") {
        meta.error_message = "First box is not ftyp, found: " + box_type;
        return meta;
    }
    
    meta.ftyp_info.box_size = ftyp_size;
    meta.ftyp_info.major_brand = std::string(reinterpret_cast<char*>(ftyp_buffer + 8), 4);
    meta.ftyp_info.minor_version = readBigEndian32(ftyp_buffer + 12);
    meta.ftyp_info.valid = true;
    
    meta.total_metadata_bytes += ftyp_size;
    
    if (verbose) {
        std::cout << "\nAccess 1: " << access1.purpose << "\n";
        std::cout << "  Offset: " << access1.offset << " bytes\n";
        std::cout << "  Length: " << access1.length << " bytes (read), " << ftyp_size << " bytes (used)\n";
        std::cout << "  Time:   " << std::fixed << std::setprecision(3) << access1.time_ms << " ms\n";
        std::cout << "  Result: Brand=" << meta.ftyp_info.major_brand 
                  << ", Version=" << meta.ftyp_info.minor_version << "\n";
    }
    
    // === ACCESS 2: Scan top-level boxes to find metadata extent ===
    // Read a chunk to scan for box structure (typically metadata is in first 1-2MB)
    uint64_t scan_size = 1048576;  // 1 MB initial scan
    std::vector<uint8_t> scan_buffer(scan_size);
    FileAccessRecord access2 = readFileSegment(filepath, 0, scan_buffer.data(), scan_size, 
                                               2, "Scan for metadata boxes");
    meta.access_log.push_back(access2);
    meta.total_accesses++;
    meta.total_access_time_ms += access2.time_ms;
    
    if (!access2.success) {
        meta.error_message = "Failed to scan boxes";
        return meta;
    }
    
    if (verbose) {
        std::cout << "\nAccess 2: " << access2.purpose << "\n";
        std::cout << "  Offset: " << access2.offset << " bytes\n";
        std::cout << "  Length: " << access2.length << " bytes\n";
        std::cout << "  Time:   " << std::fixed << std::setprecision(3) << access2.time_ms << " ms\n";
    }
    
    // Parse all boxes to find meta and mdat
    uint64_t current_offset = 0;
    bool found_mdat = false;
    uint64_t meta_box_offset = 0;
    uint64_t meta_box_size = 0;
    
    while (current_offset + 8 < scan_size && !found_mdat) {
        uint8_t* box_header = scan_buffer.data() + current_offset;
        
        HeifBox box;
        box.offset = current_offset;
        box.size = readBigEndian32(box_header);
        box.type = std::string(reinterpret_cast<char*>(box_header + 4), 4);
        box.header_size = 8;
        
        if (box.size == 1) {
            // Extended size (64-bit)
            if (current_offset + 16 > scan_size) break;
            box.size = 0;
            for (int i = 0; i < 8; i++) {
                box.size = (box.size << 8) | box_header[8 + i];
            }
            box.header_size = 16;
        } else if (box.size == 0) {
            // Box extends to end of file
            break;
        } else if (box.size < 8) {
            break;
        }
        
        box.data_offset = box.offset + box.header_size;
        box.data_size = box.size - box.header_size;
        
        meta.all_boxes.push_back(box);
        
        if (verbose) {
            std::cout << "  Box: '" << box.type << "' at offset " << box.offset 
                      << ", size " << box.size << " bytes\n";
        }
        
        if (box.type == "meta") {
            meta.meta_box = box;
            meta_box_offset = box.offset;
            meta_box_size = box.size;
        } else if (box.type == "mdat") {
            meta.mdat_box = box;
            found_mdat = true;
            meta.total_metadata_bytes = box.offset;
            if (verbose) {
                std::cout << "  -> mdat found, metadata ends at offset " << box.offset << "\n";
            }
        } else if (box.type == "idat") {
            meta.idat_box = box;
        }
        
        current_offset += box.size;
    }
    
    if (meta_box_size == 0) {
        meta.error_message = "No meta box found";
        return meta;
    }
    
    // === ACCESS 3: Read complete meta box ===
    std::vector<uint8_t> meta_buffer(meta_box_size);
    FileAccessRecord access3 = readFileSegment(filepath, meta_box_offset, meta_buffer.data(), 
                                               meta_box_size, 3, "Read meta box");
    meta.access_log.push_back(access3);
    meta.total_accesses++;
    meta.total_access_time_ms += access3.time_ms;
    
    if (!access3.success) {
        meta.error_message = "Failed to read meta box";
        return meta;
    }
    
    if (verbose) {
        std::cout << "\nAccess 3: " << access3.purpose << "\n";
        std::cout << "  Offset: " << access3.offset << " bytes\n";
        std::cout << "  Length: " << access3.length << " bytes\n";
        std::cout << "  Time:   " << std::fixed << std::setprecision(3) << access3.time_ms << " ms\n";
    }
    
    // Parse meta box children
    uint64_t meta_offset = meta.meta_box.header_size + 4;  // Skip version/flags
    
    while (meta_offset + 8 < meta_box_size) {
        uint8_t* child_header = meta_buffer.data() + meta_offset;
        
        HeifBox child;
        child.offset = meta_box_offset + meta_offset;
        child.size = readBigEndian32(child_header);
        child.type = std::string(reinterpret_cast<char*>(child_header + 4), 4);
        child.header_size = 8;
        
        if (child.size < 8 || meta_offset + child.size > meta_box_size) break;
        
        child.data_offset = child.offset + child.header_size;
        child.data_size = child.size - child.header_size;
        
        meta.meta_box.children.push_back(child);
        
        if (verbose) {
            std::cout << "  Meta child: '" << child.type << "' at offset " << child.offset 
                      << ", size " << child.size << " bytes\n";
        }
        
        // Parse important child boxes
        uint64_t child_data_offset = meta_offset + child.header_size;
        uint8_t* child_data = meta_buffer.data() + child_data_offset;
        
        // Parse iinf (item info)
        if (child.type == "iinf") {
            uint8_t version = child_data[0];
            uint32_t entry_count = 0;
            
            if (version == 0) {
                entry_count = (child_data[4] << 8) | child_data[5];
            } else {
                entry_count = readBigEndian32(child_data + 4);
            }
            
            if (verbose) {
                std::cout << "    -> iinf: " << entry_count << " items\n";
            }
            
            // Parse item info entries
            uint64_t entry_offset = (version == 0) ? 6 : 8;
            for (uint32_t i = 0; i < entry_count && i < 100; i++) {
                if (child_data_offset + entry_offset + 8 > meta_box_size) break;
                
                uint32_t infe_size = readBigEndian32(child_data + entry_offset);
                std::string infe_type(reinterpret_cast<char*>(child_data + entry_offset + 4), 4);
                
                if (infe_type == "infe" && infe_size >= 12) {
                    HeifItemInfo item;
                    uint8_t* infe_data = child_data + entry_offset + 8;
                    uint8_t infe_version = infe_data[0];
                    
                    if (infe_version == 2 || infe_version == 3) {
                        item.item_id = (infe_data[4] << 8) | infe_data[5];
                        item.item_type = std::string(reinterpret_cast<char*>(infe_data + 8), 4);
                        item.is_primary = false;
                        item.is_thumbnail = false;
                        item.width = 0;
                        item.height = 0;
                        
                        meta.items.push_back(item);
                        
                        if (verbose && i < 5) {
                            std::cout << "      Item " << item.item_id << ": type=" << item.item_type << "\n";
                        }
                    }
                    
                    entry_offset += infe_size;
                } else {
                    break;
                }
            }
        }
        
        // Parse iloc (item locations)
        if (child.type == "iloc") {
            uint8_t version = child_data[0];
            uint8_t offset_size = (child_data[4] >> 4) & 0xF;
            uint8_t length_size = child_data[4] & 0xF;
            uint8_t base_offset_size = (child_data[5] >> 4) & 0xF;
            
            uint32_t item_count = 0;
            uint64_t loc_offset = 6;
            
            if (version == 0 || version == 1) {
                item_count = (child_data[6] << 8) | child_data[7];
                loc_offset = 8;
            } else if (version == 2) {
                item_count = readBigEndian32(child_data + 6);
                loc_offset = 10;
            }
            
            if (verbose) {
                std::cout << "    -> iloc: " << item_count << " item locations\n";
                std::cout << "       version=" << (int)version 
                          << ", offset_size=" << (int)offset_size 
                          << ", length_size=" << (int)length_size << "\n";
            }
            
            // Parse each item location
            for (uint32_t i = 0; i < item_count && i < 100; i++) {
                if (child_data_offset + loc_offset + 16 > meta_box_size) break;
                
                HeifItemLocation loc;
                
                // Item ID
                if (version < 2) {
                    loc.item_id = (child_data[loc_offset] << 8) | child_data[loc_offset + 1];
                    loc_offset += 2;
                } else {
                    loc.item_id = readBigEndian32(child_data + loc_offset);
                    loc_offset += 4;
                }
                
                // Construction method (version 1 and 2)
                if (version >= 1) {
                    loc_offset += 2;  // Skip construction_method and data_reference_index
                }
                
                // Base offset
                loc.base_offset = 0;
                if (base_offset_size > 0) {
                    for (int j = 0; j < base_offset_size && j < 8; j++) {
                        loc.base_offset = (loc.base_offset << 8) | child_data[loc_offset++];
                    }
                }
                
                // Extent count
                uint16_t extent_count = (child_data[loc_offset] << 8) | child_data[loc_offset + 1];
                loc_offset += 2;
                
                // Read first extent (simplified - assumes single extent)
                if (extent_count > 0) {
                    // Extent index (version 1 and 2)
                    if (version >= 1) {
                        loc_offset += 8;  // Skip extent_index (usually 0)
                    }
                    
                    // Extent offset
                    loc.extent_offset = 0;
                    for (int j = 0; j < offset_size && j < 8; j++) {
                        loc.extent_offset = (loc.extent_offset << 8) | child_data[loc_offset++];
                    }
                    
                    // Extent length
                    loc.extent_length = 0;
                    for (int j = 0; j < length_size && j < 8; j++) {
                        loc.extent_length = (loc.extent_length << 8) | child_data[loc_offset++];
                    }
                    
                    loc.absolute_offset = loc.base_offset + loc.extent_offset;
                    
                    meta.item_locations.push_back(loc);
                    
                    if (verbose && i < 5) {
                        std::cout << "      Item " << loc.item_id 
                                  << ": offset=" << loc.absolute_offset 
                                  << ", length=" << loc.extent_length << "\n";
                    }
                }
                
                // Skip remaining extents
                for (uint16_t e = 1; e < extent_count; e++) {
                    if (version >= 1) loc_offset += 8;
                    loc_offset += offset_size + length_size;
                }
            }
        }
        
        // Parse pitm (primary item)
        if (child.type == "pitm") {
            uint8_t version = child_data[0];
            if (version == 0) {
                meta.primary_item_id = (child_data[4] << 8) | child_data[5];
            } else {
                meta.primary_item_id = readBigEndian32(child_data + 4);
            }
            
            if (verbose) {
                std::cout << "    -> pitm: Primary item ID = " << meta.primary_item_id << "\n";
            }
            
            // Mark primary item in our list
            for (auto& item : meta.items) {
                if (item.item_id == meta.primary_item_id) {
                    item.is_primary = true;
                }
            }
        }
        
        // Parse iprp (item properties) - contains dimensions
        if (child.type == "iprp") {
            // This is complex - simplified version
            if (verbose) {
                std::cout << "    -> iprp: Item properties container\n";
            }
        }
        
        meta_offset += child.size;
    }
    
    meta.valid = true;
    
    // === FINAL SUMMARY ===
    if (verbose) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "METADATA READING COMPLETE\n";
        std::cout << std::string(80, '=') << "\n";
        std::cout << "Total file accesses: " << meta.total_accesses << "\n";
        std::cout << "Total metadata bytes: " << meta.total_metadata_bytes << " bytes ("
                  << std::fixed << std::setprecision(2) << (meta.total_metadata_bytes / 1024.0) << " KB)\n";
        std::cout << "Total access time: " << std::fixed << std::setprecision(3) 
                  << meta.total_access_time_ms << " ms\n";
        std::cout << "\nStructure Summary:\n";
        std::cout << "  Total boxes: " << meta.all_boxes.size() << "\n";
        std::cout << "  Meta children: " << meta.meta_box.children.size() << "\n";
        std::cout << "  Items: " << meta.items.size() << "\n";
        std::cout << "  Item locations: " << meta.item_locations.size() << "\n";
        std::cout << "  Primary item ID: " << meta.primary_item_id << "\n";
        
        // Show all item locations
        if (!meta.item_locations.empty()) {
            std::cout << "\nAll Item Data Locations:\n";
            for (const auto& loc : meta.item_locations) {
                std::cout << "  Item " << loc.item_id << ": offset=" << loc.absolute_offset
                          << ", length=" << loc.extent_length << " bytes\n";
            }
        }
        
        std::cout << "\nAccess Log Summary:\n";
        for (const auto& access : meta.access_log) {
            std::cout << "  [" << access.access_number << "] " << access.purpose 
                      << ": offset=" << access.offset << ", length=" << access.length
                      << ", time=" << std::fixed << std::setprecision(3) << access.time_ms << "ms\n";
        }
        std::cout << std::string(80, '=') << "\n";
    }
    
    return meta;
}

// Enhanced comparison function
void compareMetadata(const std::string& cog_path, const std::string& heif_path) {
    std::cout << "\n" << std::string(80, '=') << "\n";
    std::cout << "METADATA COMPARISON: COG vs HEIF\n";
    std::cout << std::string(80, '=') << "\n";
    
    auto cog_meta = readGeoTiffMetadata(cog_path, true);
    auto heif_meta = readHeifMetadata(heif_path, true);
    
    std::cout << "\n" << std::string(80, '=') << "\n";
    std::cout << "SIDE-BY-SIDE COMPARISON\n";
    std::cout << std::string(80, '=') << "\n";
    
    std::cout << "\n1. FILE ACCESS PATTERN\n";
    std::cout << std::string(80, '-') << "\n";
    std::cout << std::left << std::setw(40) << "Metric" << std::setw(20) << "COG" << "HEIF\n";
    std::cout << std::string(80, '-') << "\n";
    std::cout << std::left << std::setw(40) << "Total accesses:" 
              << std::setw(20) << cog_meta.total_accesses 
              << heif_meta.total_accesses << "\n";
    std::cout << std::left << std::setw(40) << "Total metadata bytes:" 
              << std::setw(20) << cog_meta.total_metadata_bytes 
              << heif_meta.total_metadata_bytes << "\n";
    std::cout << std::left << std::setw(40) << "Total access time (ms):" 
              << std::setw(20) << std::fixed << std::setprecision(3) << cog_meta.total_access_time_ms 
              << heif_meta.total_access_time_ms << "\n";
    std::cout << std::left << std::setw(40) << "Avg time per access (ms):" 
              << std::setw(20) << (cog_meta.total_access_time_ms / cog_meta.total_accesses)
              << (heif_meta.total_access_time_ms / heif_meta.total_accesses) << "\n";
    
    std::cout << "\n2. DATA STRUCTURE\n";
    std::cout << std::string(80, '-') << "\n";
    std::cout << std::left << std::setw(40) << "Metric" << std::setw(20) << "COG" << "HEIF\n";
    std::cout << std::string(80, '-') << "\n";
    std::cout << std::left << std::setw(40) << "Total IFDs/Items:" 
              << std::setw(20) << cog_meta.ifds.size() 
              << heif_meta.items.size() << "\n";
    std::cout << std::left << std::setw(40) << "Main image IFDs/Primary items:" 
              << std::setw(20) << "1" << "1\n";
    std::cout << std::left << std::setw(40) << "Overview IFDs/Thumbnails:" 
              << std::setw(20) << cog_meta.overview_ifds.size() 
              << (heif_meta.items.size() - 1) << " (approx)\n";
    
    // Count total tiles/data locations
    uint64_t cog_total_tiles = 0;
    for (const auto& ifd : cog_meta.ifds) {
        cog_total_tiles += ifd.data_locations.size();
    }
    
    std::cout << std::left << std::setw(40) << "Total tiles/data locations:" 
              << std::setw(20) << cog_total_tiles 
              << heif_meta.item_locations.size() << "\n";
    
    std::cout << "\n3. METADATA EFFICIENCY\n";
    std::cout << std::string(80, '-') << "\n";
    double cog_bytes_per_access = (double)cog_meta.total_metadata_bytes / cog_meta.total_accesses;
    double heif_bytes_per_access = (double)heif_meta.total_metadata_bytes / heif_meta.total_accesses;
    std::cout << std::left << std::setw(40) << "Bytes per access:" 
              << std::setw(20) << std::fixed << std::setprecision(0) << cog_bytes_per_access 
              << heif_bytes_per_access << "\n";
    
    double cog_metadata_overhead = (double)cog_meta.total_metadata_bytes / BenchmarkUtils::getFileSize(cog_path) * 100;
    double heif_metadata_overhead = (double)heif_meta.total_metadata_bytes / BenchmarkUtils::getFileSize(heif_path) * 100;
    std::cout << std::left << std::setw(40) << "Metadata overhead (% of file):" 
              << std::setw(20) << std::fixed << std::setprecision(3) << cog_metadata_overhead 
              << heif_metadata_overhead << "\n";
    
    std::cout << "\n4. ACCESS PATTERN DETAILS\n";
    std::cout << std::string(80, '-') << "\n";
    std::cout << "\nCOG Access Pattern:\n";
    for (const auto& access : cog_meta.access_log) {
        std::cout << "  [" << access.access_number << "] " << access.purpose 
                  << ": " << access.length << " bytes @ offset " << access.offset
                  << " (" << std::fixed << std::setprecision(3) << access.time_ms << "ms)\n";
    }
    
    std::cout << "\nHEIF Access Pattern:\n";
    for (const auto& access : heif_meta.access_log) {
        std::cout << "  [" << access.access_number << "] " << access.purpose 
                  << ": " << access.length << " bytes @ offset " << access.offset
                  << " (" << std::fixed << std::setprecision(3) << access.time_ms << "ms)\n";
    }
    
    std::cout << "\n5. KEY INSIGHTS\n";
    std::cout << std::string(80, '-') << "\n";
    
    if (cog_meta.total_accesses < heif_meta.total_accesses) {
        std::cout << "✓ COG requires fewer file accesses (" << cog_meta.total_accesses 
                  << " vs " << heif_meta.total_accesses << ")\n";
    } else {
        std::cout << "✓ HEIF requires fewer file accesses (" << heif_meta.total_accesses 
                  << " vs " << cog_meta.total_accesses << ")\n";
    }
    
    if (cog_meta.total_metadata_bytes < heif_meta.total_metadata_bytes) {
        double savings = (1.0 - (double)cog_meta.total_metadata_bytes / heif_meta.total_metadata_bytes) * 100;
        std::cout << "✓ COG has more compact metadata (" << std::fixed << std::setprecision(1) 
                  << savings << "% smaller)\n";
    } else {
        double savings = (1.0 - (double)heif_meta.total_metadata_bytes / cog_meta.total_metadata_bytes) * 100;
        std::cout << "✓ HEIF has more compact metadata (" << std::fixed << std::setprecision(1) 
                  << savings << "% smaller)\n";
    }
    
    if (cog_meta.total_access_time_ms < heif_meta.total_access_time_ms) {
        std::cout << "✓ COG metadata reads faster (" << std::fixed << std::setprecision(3) 
                  << cog_meta.total_access_time_ms << "ms vs " 
                  << heif_meta.total_access_time_ms << "ms)\n";
    } else {
        std::cout << "✓ HEIF metadata reads faster (" << std::fixed << std::setprecision(3) 
                  << heif_meta.total_access_time_ms << "ms vs " 
                  << cog_meta.total_access_time_ms << "ms)\n";
    }
    
    std::cout << "\n" << std::string(80, '=') << "\n";
}

// Batch compare all dataset pairs
void compareAllDatasetPairs(const std::vector<BenchmarkUtils::DatasetPair>& pairs) {
    if (pairs.empty()) {
        std::cout << "No dataset pairs to compare.\n";
        return;
    }
    
    std::cout << "\n" << std::string(80, '=') << "\n";
    std::cout << "COMPARING ALL DATASET PAIRS\n";
    std::cout << std::string(80, '=') << "\n";
    
    for (size_t i = 0; i < pairs.size(); i++) {
        const auto& pair = pairs[i];
        std::cout << "\n\nPair " << (i+1) << "/" << pairs.size() << ": " 
                  << pair.compression << ", tile=" << pair.tile_size 
                  << ", pyramids=" << pair.pyramid_levels << "\n";
        
        compareMetadata(pair.cog_path, pair.geoheif_path);
        
        if (i < pairs.size() - 1) {
            std::cout << "\n" << std::string(80, '=') << "\n";
        }
    }
}

*/

} // namespace FileAnalysis

std::cout << "FileAnalysis namespace loaded (proper multi-access metadata reading with detailed logging)." << std::endl;

##### Examples to read TIFF metadata

In [ ]:
// Read and display all COG metadata with detailed access logging
auto cog_meta = FileAnalysis::readGeoTiffMetadata("benchmark_output/cog_DEFLATE_t256_p3.tif", true);

// Access the detailed information
for (const auto& ifd : cog_meta.ifds) {
    std::cout << "\nIFD Level " << ifd.level << ": " << ifd.image_width << "x" << ifd.image_height << "\n";
    std::cout << "  Data locations: " << ifd.data_locations.size() << "\n";
    for (size_t i = 0; i < std::min(size_t(5), ifd.data_locations.size()); i++) {
        std::cout << "    Tile/Strip " << i << ": offset=" << ifd.data_locations[i].offset
                  << ", bytes=" << ifd.data_locations[i].byte_count << "\n";
    }
}

### Enhanced

##### Enhanced COG Metadata Reading with Complete Tile Details

In [ ]:
namespace FileAnalysis {


// Helper to read tile/strip offset and bytecount arrays
bool readTileArrays(const std::string& filepath,
                   TiffIFD& ifd,
                   bool is_big_endian,
                   bool is_big_tiff,
                   int& access_count,
                   std::vector<FileAccessRecord>& access_log,
                   bool verbose = true) {
    
    // Find TileOffsets/StripOffsets and TileByteCounts/StripByteCounts tags
    TiffIFDEntry* offsets_entry = nullptr;
    TiffIFDEntry* bytecounts_entry = nullptr;
    
    for (auto& entry : ifd.entries) {
        if (entry.tag == 324 || entry.tag == 273) {  // TileOffsets or StripOffsets
            offsets_entry = &entry;
        }
        if (entry.tag == 325 || entry.tag == 279) {  // TileByteCounts or StripByteCounts
            bytecounts_entry = &entry;
        }
    }
    
    if (!offsets_entry || !bytecounts_entry) {
        if (verbose) {
            std::cout << "  [WARN] No tile/strip offset arrays found\n";
        }
        return false;
    }
    
    // Determine array storage (inline vs external)
    size_t type_sizes[] = {0, 1, 1, 2, 4, 8, 1, 1, 2, 4, 8, 4, 8};
    size_t offset_type_size = offsets_entry->type < 13 ? type_sizes[offsets_entry->type] : 4;
    size_t bytecount_type_size = bytecounts_entry->type < 13 ? type_sizes[bytecounts_entry->type] : 4;
    
    size_t offsets_array_size = offsets_entry->count * offset_type_size;
    size_t bytecounts_array_size = bytecounts_entry->count * bytecount_type_size;
    
    size_t inline_size = is_big_tiff ? 8 : 4;
    bool offsets_external = (offsets_array_size > inline_size);
    bool bytecounts_external = (bytecounts_array_size > inline_size);
    
    std::vector<uint64_t> offsets;
    std::vector<uint64_t> bytecounts;
    
    if (offsets_external && bytecounts_external) {
        // Read arrays from file
        access_count++;
        
        uint64_t min_offset = std::min(offsets_entry->value_offset, bytecounts_entry->value_offset);
        uint64_t max_offset = std::max(offsets_entry->value_offset + offsets_array_size,
                                       bytecounts_entry->value_offset + bytecounts_array_size);
        size_t arrays_size = max_offset - min_offset;
        
        std::vector<uint8_t> arrays_buffer(arrays_size);
        FileAccessRecord arrays_access = readFileSegment(filepath, min_offset,
                                                         arrays_buffer.data(), arrays_size,
                                                         access_count,
                                                         "Read tile/strip arrays for IFD " + std::to_string(ifd.level));
        access_log.push_back(arrays_access);
        
        if (!arrays_access.success) {
            return false;
        }
        
        // Parse offsets array
        uint64_t offsets_local = offsets_entry->value_offset - min_offset;
        for (uint64_t i = 0; i < offsets_entry->count; i++) {
            uint64_t val = 0;
            size_t idx = offsets_local + (i * offset_type_size);
            
            if (idx + offset_type_size <= arrays_size) {
                if (offset_type_size == 4) {
                    val = is_big_endian ? 
                        readBigEndian32(arrays_buffer.data() + idx) :
                        readLittleEndian32(arrays_buffer.data() + idx);
                } else if (offset_type_size == 8) {
                    if (is_big_endian) {
                        for (int j = 0; j < 8; j++) {
                            val = (val << 8) | arrays_buffer[idx + j];
                        }
                    } else {
                        for (int j = 7; j >= 0; j--) {
                            val = (val << 8) | arrays_buffer[idx + j];
                        }
                    }
                } else if (offset_type_size == 2) {
                    val = is_big_endian ?
                        ((arrays_buffer[idx] << 8) | arrays_buffer[idx + 1]) :
                        readLittleEndian16(arrays_buffer.data() + idx);
                }
                offsets.push_back(val);
            }
        }
        
        // Parse bytecounts array
        uint64_t bytecounts_local = bytecounts_entry->value_offset - min_offset;
        for (uint64_t i = 0; i < bytecounts_entry->count; i++) {
            uint64_t val = 0;
            size_t idx = bytecounts_local + (i * bytecount_type_size);
            
            if (idx + bytecount_type_size <= arrays_size) {
                if (bytecount_type_size == 4) {
                    val = is_big_endian ?
                        readBigEndian32(arrays_buffer.data() + idx) :
                        readLittleEndian32(arrays_buffer.data() + idx);
                } else if (bytecount_type_size == 8) {
                    if (is_big_endian) {
                        for (int j = 0; j < 8; j++) {
                            val = (val << 8) | arrays_buffer[idx + j];
                        }
                    } else {
                        for (int j = 7; j >= 0; j--) {
                            val = (val << 8) | arrays_buffer[idx + j];
                        }
                    }
                } else if (bytecount_type_size == 2) {
                    val = is_big_endian ?
                        ((arrays_buffer[idx] << 8) | arrays_buffer[idx + 1]) :
                        readLittleEndian16(arrays_buffer.data() + idx);
                }
                bytecounts.push_back(val);
            }
        }
        
        if (verbose) {
            std::cout << "  Parsed " << offsets.size() << " offsets, " 
                      << bytecounts.size() << " byte counts\n";
        }
        
    } else {
        // Handle inline values (rare for real datasets)
        if (!offsets_external) {
            offsets.push_back(offsets_entry->value_offset);
        }
        if (!bytecounts_external) {
            bytecounts.push_back(bytecounts_entry->value_offset);
        }
    }
    
    // Create data location entries with tile positions
    ifd.total_compressed_bytes = 0;
    
    for (size_t i = 0; i < offsets.size() && i < bytecounts.size(); i++) {
        TiffDataLocation loc;
        loc.index = i;
        loc.offset = offsets[i];
        loc.byte_count = bytecounts[i];
        loc.is_tile = ifd.is_tiled;
        loc.compression = ifd.compression;
        
        // Calculate tile position for tiled images
        if (ifd.is_tiled && ifd.tiles_across > 0) {
            loc.tile_x = i % ifd.tiles_across;
            loc.tile_y = i / ifd.tiles_across;
        } else {
            loc.tile_x = 0;
            loc.tile_y = i;  // For strips, y = strip index
        }
        
        ifd.data_locations.push_back(loc);
        ifd.total_compressed_bytes += bytecounts[i];
    }
    
    // Calculate uncompressed size estimate
    if (ifd.is_tiled) {
        uint64_t tile_pixels = static_cast<uint64_t>(ifd.tile_width) * ifd.tile_height;
        ifd.total_uncompressed_bytes = tile_pixels * ifd.samples_per_pixel * 
                                       ifd.data_locations.size();
    } else {
        ifd.total_uncompressed_bytes = static_cast<uint64_t>(ifd.image_width) * 
                                       ifd.image_height * ifd.samples_per_pixel;
    }
    
    if (verbose) {
        std::cout << "  Total compressed: " << (ifd.total_compressed_bytes / 1024.0) << " KB\n";
        std::cout << "  Compression ratio: " << std::fixed << std::setprecision(2)
                  << (static_cast<double>(ifd.total_uncompressed_bytes) / ifd.total_compressed_bytes)
                  << ":1\n";
    }
    
    return true;
}

// Enhanced GeoTIFF metadata reading with complete tile information
GeoTiffMetadata readGeoTiffMetadataComplete(const std::string& filepath, bool verbose = true) {
    GeoTiffMetadata meta;
    meta.valid = false;
    meta.total_metadata_bytes = 0;
    meta.total_accesses = 0;
    meta.total_access_time_ms = 0;
    
    if (verbose) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "READING COMPLETE GEOTIFF METADATA: " << filepath << "\n";
        std::cout << std::string(80, '=') << "\n";
    }
    
    // === ACCESS 1: Read TIFF header ===
    uint8_t header[16];
    FileAccessRecord access1 = readFileSegment(filepath, 0, header, 16, 1, "Read TIFF header");
    meta.access_log.push_back(access1);
    meta.total_accesses++;
    meta.total_access_time_ms += access1.time_ms;
    meta.total_metadata_bytes += access1.length;
    
    if (!access1.success) {
        meta.error_message = "Failed to read header";
        return meta;
    }
    
    // Parse header
    bool is_big_endian = (header[0] == 0x4D && header[1] == 0x4D);
    uint16_t magic = is_big_endian ? ((header[2] << 8) | header[3]) : readLittleEndian16(header + 2);
    bool is_big_tiff = (magic == 43);
    
    meta.header_info.is_big_endian = is_big_endian;
    meta.header_info.is_big_tiff = is_big_tiff;
    meta.header_info.magic_number = magic;
    meta.header_info.valid = (magic == 42 || magic == 43);
    meta.header_info.header_size = is_big_tiff ? 16 : 8;
    
    if (!meta.header_info.valid) {
        meta.error_message = "Invalid TIFF magic number";
        return meta;
    }
    
    // Get first IFD offset
    uint64_t first_ifd_offset;
    if (is_big_tiff) {
        if (is_big_endian) {
            first_ifd_offset = 0;
            for (int i = 8; i < 16; i++) {
                first_ifd_offset = (first_ifd_offset << 8) | header[i];
            }
        } else {
            first_ifd_offset = 0;
            for (int i = 15; i >= 8; i--) {
                first_ifd_offset = (first_ifd_offset << 8) | header[i];
            }
        }
    } else {
        first_ifd_offset = is_big_endian ? readBigEndian32(header + 4) : readLittleEndian32(header + 4);
    }
    meta.header_info.ifd_offset = first_ifd_offset;
    
    if (verbose) {
        std::cout << "\nAccess 1: " << access1.purpose << "\n";
        std::cout << "  Format: " << (is_big_endian ? "Big-endian" : "Little-endian")
                  << ", " << (is_big_tiff ? "BigTIFF" : "Standard TIFF") << "\n";
        std::cout << "  First IFD offset: " << first_ifd_offset << " bytes\n";
    }
    
    // === ACCESS 2+: Read each IFD with complete tile information ===
    uint64_t current_ifd_offset = first_ifd_offset;
    int ifd_level = 0;
    
    while (current_ifd_offset != 0 && ifd_level < 20) {
        meta.total_accesses++;
        int access_num = meta.total_accesses;
        
        // Read IFD
        size_t ifd_read_size = is_big_tiff ? 8192 : 4096;
        std::vector<uint8_t> ifd_buffer(ifd_read_size);
        
        FileAccessRecord ifd_access = readFileSegment(filepath, current_ifd_offset,
                                                      ifd_buffer.data(), ifd_read_size,
                                                      access_num,
                                                      "Read IFD " + std::to_string(ifd_level));
        meta.access_log.push_back(ifd_access);
        meta.total_access_time_ms += ifd_access.time_ms;
        
        if (!ifd_access.success) break;
        
        // Parse IFD
        TiffIFD ifd;
        ifd.offset = current_ifd_offset;
        ifd.level = ifd_level;
        ifd.is_overview = false;
        ifd.image_width = 0;
        ifd.image_height = 0;
        ifd.tile_width = 0;
        ifd.tile_height = 0;
        ifd.rows_per_strip = 0;
        ifd.samples_per_pixel = 1;
        ifd.compression_code = 1;  // Default: no compression
        ifd.compression = "None";
        ifd.predictor = 1;
        ifd.is_tiled = false;
        ifd.total_compressed_bytes = 0;
        ifd.total_uncompressed_bytes = 0;
        
        uint8_t* ifd_data = ifd_buffer.data();
        size_t offset_in_buffer = 0;
        
        // Read entry count
        if (is_big_tiff) {
            if (is_big_endian) {
                ifd.entry_count = 0;
                for (int i = 0; i < 8; i++) {
                    ifd.entry_count = (ifd.entry_count << 8) | ifd_data[i];
                }
            } else {
                ifd.entry_count = 0;
                for (int i = 7; i >= 0; i--) {
                    ifd.entry_count = (ifd.entry_count << 8) | ifd_data[i];
                }
            }
            offset_in_buffer = 8;
        } else {
            ifd.entry_count = is_big_endian ? 
                ((ifd_data[0] << 8) | ifd_data[1]) : readLittleEndian16(ifd_data);
            offset_in_buffer = 2;
        }
        
        size_t entry_size = is_big_tiff ? 20 : 12;
        size_t actual_ifd_size = (is_big_tiff ? 8 : 2) + (ifd.entry_count * entry_size) + (is_big_tiff ? 8 : 4);
        meta.total_metadata_bytes += actual_ifd_size;
        
        if (verbose) {
            std::cout << "\nAccess " << access_num << ": " << ifd_access.purpose << "\n";
            std::cout << "  Entries: " << ifd.entry_count << "\n";
        }
        
        // Parse all entries
        for (uint64_t i = 0; i < ifd.entry_count && i < 1000; i++) {
            uint8_t* entry_data = ifd_data + offset_in_buffer;
            TiffIFDEntry entry;
            
            if (is_big_tiff) {
                entry.tag = is_big_endian ? ((entry_data[0] << 8) | entry_data[1]) : readLittleEndian16(entry_data);
                entry.type = is_big_endian ? ((entry_data[2] << 8) | entry_data[3]) : readLittleEndian16(entry_data + 2);
                entry.count = is_big_endian ? 
                    ((uint64_t)entry_data[4] << 56) | ((uint64_t)entry_data[5] << 48) |
                    ((uint64_t)entry_data[6] << 40) | ((uint64_t)entry_data[7] << 32) |
                    ((uint64_t)entry_data[8] << 24) | ((uint64_t)entry_data[9] << 16) |
                    ((uint64_t)entry_data[10] << 8) | entry_data[11] :
                    entry_data[4] | ((uint64_t)entry_data[5] << 8) | ((uint64_t)entry_data[6] << 16) |
                    ((uint64_t)entry_data[7] << 24) | ((uint64_t)entry_data[8] << 32) | 
                    ((uint64_t)entry_data[9] << 40) | ((uint64_t)entry_data[10] << 48) | 
                    ((uint64_t)entry_data[11] << 56);
                entry.value_offset = is_big_endian ? 
                    ((uint64_t)entry_data[12] << 56) | ((uint64_t)entry_data[13] << 48) |
                    ((uint64_t)entry_data[14] << 40) | ((uint64_t)entry_data[15] << 32) |
                    ((uint64_t)entry_data[16] << 24) | ((uint64_t)entry_data[17] << 16) |
                    ((uint64_t)entry_data[18] << 8) | entry_data[19] :
                    entry_data[12] | ((uint64_t)entry_data[13] << 8) | ((uint64_t)entry_data[14] << 16) |
                    ((uint64_t)entry_data[15] << 24) | ((uint64_t)entry_data[16] << 32) | 
                    ((uint64_t)entry_data[17] << 40) | ((uint64_t)entry_data[18] << 48) | 
                    ((uint64_t)entry_data[19] << 56);
            } else {
                entry.tag = is_big_endian ? ((entry_data[0] << 8) | entry_data[1]) : readLittleEndian16(entry_data);
                entry.type = is_big_endian ? ((entry_data[2] << 8) | entry_data[3]) : readLittleEndian16(entry_data + 2);
                entry.count = is_big_endian ? readBigEndian32(entry_data + 4) : readLittleEndian32(entry_data + 4);
                entry.value_offset = is_big_endian ? readBigEndian32(entry_data + 8) : readLittleEndian32(entry_data + 8);
            }
            
            entry.tag_name = getTiffTagName(entry.tag);
            ifd.entries.push_back(entry);
            
            // Decode critical tags including compression details
            switch (entry.tag) {
                case 254: if (entry.value_offset & 1) ifd.is_overview = true; break;
                case 256: ifd.image_width = entry.value_offset; break;
                case 257: ifd.image_height = entry.value_offset; break;
                case 277: ifd.samples_per_pixel = entry.value_offset; break;
                case 278: ifd.rows_per_strip = entry.value_offset; break;
                case 322: ifd.tile_width = entry.value_offset; ifd.is_tiled = true; break;
                case 323: ifd.tile_height = entry.value_offset; ifd.is_tiled = true; break;
                case 317: ifd.predictor = entry.value_offset; break;  // Predictor
                case 259: {  // Compression
                    ifd.compression_code = entry.value_offset;
                    static const std::map<uint32_t, std::string> compressions = {
                        {1, "None"}, {5, "LZW"}, {7, "JPEG"}, {8, "Deflate"},
                        {32946, "Deflate"}, {50000, "ZSTD"}, {34712, "JPEG2000"}
                    };
                    auto it = compressions.find(entry.value_offset);
                    ifd.compression = it != compressions.end() ? it->second : "Unknown(" + std::to_string(entry.value_offset) + ")";
                    break;
                }
            }
            
            offset_in_buffer += entry_size;
        }
        
        // Calculate tile/strip layout
        if (ifd.is_tiled && ifd.tile_width > 0 && ifd.tile_height > 0) {
            ifd.tiles_across = (ifd.image_width + ifd.tile_width - 1) / ifd.tile_width;
            ifd.tiles_down = (ifd.image_height + ifd.tile_height - 1) / ifd.tile_height;
        } else if (ifd.rows_per_strip > 0) {
            ifd.strips_count = (ifd.image_height + ifd.rows_per_strip - 1) / ifd.rows_per_strip;
        }
        
        // Read next IFD offset
        uint8_t* next_offset_data = ifd_data + offset_in_buffer;
        if (is_big_tiff) {
            ifd.next_ifd_offset = is_big_endian ?
                ((uint64_t)next_offset_data[0] << 56) | ((uint64_t)next_offset_data[1] << 48) |
                ((uint64_t)next_offset_data[2] << 40) | ((uint64_t)next_offset_data[3] << 32) |
                ((uint64_t)next_offset_data[4] << 24) | ((uint64_t)next_offset_data[5] << 16) |
                ((uint64_t)next_offset_data[6] << 8) | next_offset_data[7] :
                next_offset_data[0] | ((uint64_t)next_offset_data[1] << 8) | 
                ((uint64_t)next_offset_data[2] << 16) | ((uint64_t)next_offset_data[3] << 24) |
                ((uint64_t)next_offset_data[4] << 32) | ((uint64_t)next_offset_data[5] << 40) |
                ((uint64_t)next_offset_data[6] << 48) | ((uint64_t)next_offset_data[7] << 56);
        } else {
            ifd.next_ifd_offset = is_big_endian ? 
                readBigEndian32(next_offset_data) : readLittleEndian32(next_offset_data);
        }
        
        if (verbose) {
            std::cout << "  Image: " << ifd.image_width << "x" << ifd.image_height;
            if (ifd.is_tiled) {
                std::cout << ", Tiles: " << ifd.tiles_across << "x" << ifd.tiles_down 
                          << " (" << ifd.tile_width << "x" << ifd.tile_height << ")";
            } else {
                std::cout << ", Strips: " << ifd.strips_count;
            }
            std::cout << "\n  Compression: " << ifd.compression;
            if (ifd.predictor > 1) {
                std::cout << " (Predictor: " << ifd.predictor << ")";
            }
            if (ifd.is_overview) std::cout << " [OVERVIEW]";
            std::cout << "\n";
        }
        
        // Read tile/strip arrays with complete information
        readTileArrays(filepath, ifd, is_big_endian, is_big_tiff, 
                      meta.total_accesses, meta.access_log, verbose);
        
        meta.ifds.push_back(ifd);
        if (ifd_level == 0) {
            meta.main_ifd = ifd;
        } else if (ifd.is_overview) {
            meta.overview_ifds.push_back(ifd);
        }
        
        current_ifd_offset = ifd.next_ifd_offset;
        ifd_level++;
    }
    
    meta.valid = true;
    
    if (verbose) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "METADATA READING COMPLETE\n";
        std::cout << std::string(80, '=') << "\n";
        std::cout << "Total IFDs: " << meta.ifds.size() << "\n";
        std::cout << "Total accesses: " << meta.total_accesses << "\n";
        std::cout << "Total metadata bytes: " << meta.total_metadata_bytes 
                  << " (" << (meta.total_metadata_bytes / 1024.0) << " KB)\n";
        std::cout << "Total time: " << std::fixed << std::setprecision(3) 
                  << meta.total_access_time_ms << " ms\n";
        
        // Summary by level
        for (const auto& ifd : meta.ifds) {
            std::cout << "\nLevel " << ifd.level << " (" 
                      << ifd.image_width << "x" << ifd.image_height << "):\n";
            std::cout << "  Tiles/Strips: " << ifd.data_locations.size() << "\n";
            std::cout << "  Compression: " << ifd.compression << "\n";
            std::cout << "  Compressed size: " << (ifd.total_compressed_bytes / 1024.0) << " KB\n";
            std::cout << "  Compression ratio: " << std::fixed << std::setprecision(2)
                      << (static_cast<double>(ifd.total_uncompressed_bytes) / 
                          std::max(1ULL, ifd.total_compressed_bytes)) << ":1\n";
        }
        std::cout << std::string(80, '=') << "\n";
    }
    
    return meta;
}

} // namespace FileAnalysis

##### Enhanced HEIF Metadata Reading with Complete Tile and Compression Details

In [ ]:
namespace FileAnalysis {

    // Helper function to read big-endian integers
template<typename T>
T readBigEndian(const uint8_t* data) {
    T result = 0;
    for (size_t i = 0; i < sizeof(T); ++i) {
        result = (result << 8) | data[i];
    }
    return result;
}

// Parse HEIF box
HeifBox parseBox(const uint8_t* data, uint64_t size, uint64_t offset) {
    HeifBox box;
    box.offset = offset;
    box.size = readBigEndian<uint32_t>(data);
    box.type = std::string(reinterpret_cast<const char*>(data + 4), 4);
    box.header_size = 8;
    
    if (box.size == 1) {  // Extended size
        box.size = readBigEndian<uint64_t>(data + 8);
        box.header_size = 16;
    }
    
    box.data_offset = offset + box.header_size;
    box.data_size = box.size - box.header_size;
    
    return box;
}

// Parse iinf box
std::vector<HeifItemInfo> parseIinf(const uint8_t* data, uint64_t size) {
    std::vector<HeifItemInfo> items;
    uint8_t version = data[0];
    uint32_t item_count;
    uint64_t offset = 4;
    
    if (version == 0) {
        item_count = readBigEndian<uint16_t>(data + 2);
    } else {
        item_count = readBigEndian<uint32_t>(data + 2);
        offset = 6;
    }
    
    for (uint32_t i = 0; i < item_count; ++i) {
        HeifItemInfo item;
        uint32_t item_size = readBigEndian<uint32_t>(data + offset);
        offset += 4;
        
        if (version >= 1) {
            item.item_id = readBigEndian<uint32_t>(data + offset);
            offset += 4;
        } else {
            item.item_id = readBigEndian<uint16_t>(data + offset);
            offset += 2;
        }
        
        item.item_type = std::string(reinterpret_cast<const char*>(data + offset), 4);
        offset += 4;
        
        // Parse additional properties if needed
        // ...
        
        items.push_back(item);
        offset += item_size - (offset - (offset - item_size));
    }
    
    return items;
}

// Parse iprp box for item properties (including grid info)
void parseIprp(const uint8_t* data, uint64_t size, std::vector<HeifItemInfo>& items) {
    // This is a simplified version. A full implementation would parse ipco and ipma boxes.
    uint64_t offset = 0;
    while (offset < size) {
        HeifBox child_box = parseBox(data + offset, size - offset, offset);
        if (child_box.type == "ipco") {
            // Parse item property container
            // ...
        } else if (child_box.type == "ipma") {
            // Parse item property association
            // ...
        }
        offset += child_box.size;
    }
}


// Enhanced debugging helper
void hexDump(const uint8_t* data, size_t size, size_t offset, const std::string& label) {
    std::cout << label << " (offset " << offset << ", " << size << " bytes):\n  ";
    for (size_t i = 0; i < std::min(size, size_t(32)); ++i) {
        std::cout << std::hex << std::setw(2) << std::setfill('0') << (int)data[i] << " ";
        if ((i + 1) % 16 == 0 && i + 1 < size) std::cout << "\n  ";
    }
    std::cout << std::dec << std::setfill(' ') << "\n";
}

// Corrected variable-size integer reader with bounds checking
uint64_t readBigEndianVar(const uint8_t* data, uint64_t& offset, uint8_t field_size, uint64_t max_size) {
    if (field_size == 0) return 0;
    if (offset + field_size > max_size) {
        std::cout << "  [ERROR] readBigEndianVar: would read beyond buffer (offset=" 
                  << offset << ", field_size=" << (int)field_size 
                  << ", max_size=" << max_size << ")\n";
        return 0;
    }
    
    uint64_t value = 0;
    for (uint8_t i = 0; i < field_size; ++i) {
        value = (value << 8) | data[offset + i];
    }
    
    offset += field_size;
    return value;
}

// Completely rewritten iloc parser with extensive debugging
std::vector<HeifItemLocation> parseIlocDebug(const uint8_t* data, uint64_t size, bool verbose = true) {
    std::vector<HeifItemLocation> locations;
    
    if (size < 8) {
        std::cout << "  [ILOC ERROR] Buffer too small: " << size << " bytes\n";
        return locations;
    }
    
    if (verbose) {
        std::cout << "\n[ILOC PARSER] Starting parse, buffer size: " << size << " bytes\n";
        hexDump(data, std::min(size, uint64_t(64)), 0, "[ILOC] First 64 bytes");
    }
    
    // Parse header (always 8 bytes minimum)
    uint8_t version = data[0];
    uint8_t flags_1 = data[1];
    uint8_t flags_2 = data[2];
    uint8_t flags_3 = data[3];
    
    uint8_t offset_size = (data[4] >> 4) & 0xF;
    uint8_t length_size = data[4] & 0xF;
    uint8_t base_offset_size = (data[5] >> 4) & 0xF;
    uint8_t index_size = 0;
    
    if (version == 1 || version == 2) {
        index_size = data[5] & 0xF;
    }
    
    if (verbose) {
        std::cout << "[ILOC] Header:\n";
        std::cout << "  version: " << (int)version << "\n";
        std::cout << "  flags: " << std::hex << (int)flags_1 << " " << (int)flags_2 << " " << (int)flags_3 << std::dec << "\n";
        std::cout << "  offset_size: " << (int)offset_size << " bytes\n";
        std::cout << "  length_size: " << (int)length_size << " bytes\n";
        std::cout << "  base_offset_size: " << (int)base_offset_size << " bytes\n";
        std::cout << "  index_size: " << (int)index_size << " bytes\n";
    }
    
    // Validate field sizes
    if (offset_size > 8 || length_size > 8 || base_offset_size > 8 || index_size > 8) {
        std::cout << "  [ILOC ERROR] Invalid field sizes detected!\n";
        return locations;
    }
    
    // Read item count
    uint32_t item_count = 0;
    uint64_t offset = 6;
    
    if (version < 2) {
        if (size < 8) {
            std::cout << "  [ILOC ERROR] Buffer too small for item count\n";
            return locations;
        }
        item_count = (data[6] << 8) | data[7];
        offset = 8;
        if (verbose) {
            std::cout << "[ILOC] Item count (v0/v1, 2 bytes): " << item_count << "\n";
            std::cout << "  Raw bytes: " << std::hex << (int)data[6] << " " << (int)data[7] << std::dec << "\n";
        }
    } else {
        if (size < 10) {
            std::cout << "  [ILOC ERROR] Buffer too small for item count (v2)\n";
            return locations;
        }
        item_count = (data[6] << 24) | (data[7] << 16) | (data[8] << 8) | data[9];
        offset = 10;
        if (verbose) {
            std::cout << "[ILOC] Item count (v2, 4 bytes): " << item_count << "\n";
            std::cout << "  Raw bytes: " << std::hex << (int)data[6] << " " << (int)data[7] 
                      << " " << (int)data[8] << " " << (int)data[9] << std::dec << "\n";
        }
    }
    
    // Sanity check item count
    if (item_count == 0) {
        std::cout << "  [ILOC] No items\n";
        return locations;
    }
    if (item_count > 10000) {
        std::cout << "  [ILOC ERROR] Suspicious item count: " << item_count << " (too large)\n";
        return locations;
    }
    
    std::cout << "[ILOC] Processing " << item_count << " items...\n";
    
    // Process each item
    for (uint32_t i = 0; i < item_count; ++i) {
        if (offset >= size) {
            std::cout << "  [ILOC ERROR] Offset " << offset << " exceeds buffer size at item " << i << "\n";
            break;
        }
        
        HeifItemLocation loc;
        uint64_t item_start_offset = offset;
        
        if (verbose && i < 3) {
            std::cout << "\n[ILOC] Item " << i << " at offset " << offset << ":\n";
        }
        
        // Read item ID
        if (version < 2) {
            if (offset + 2 > size) break;
            loc.item_id = (data[offset] << 8) | data[offset + 1];
            if (verbose && i < 3) {
                std::cout << "  item_id (2 bytes): " << loc.item_id 
                          << " [" << std::hex << (int)data[offset] << " " << (int)data[offset+1] << std::dec << "]\n";
            }
            offset += 2;
        } else {
            if (offset + 4 > size) break;
            loc.item_id = (data[offset] << 24) | (data[offset + 1] << 16) | 
                         (data[offset + 2] << 8) | data[offset + 3];
            if (verbose && i < 3) {
                std::cout << "  item_id (4 bytes): " << loc.item_id 
                          << " [" << std::hex << (int)data[offset] << " " << (int)data[offset+1] 
                          << " " << (int)data[offset+2] << " " << (int)data[offset+3] << std::dec << "]\n";
            }
            offset += 4;
        }
        
        // Read construction_method and data_reference_index (version 1 and 2)
        uint16_t construction_method = 0;
        uint16_t data_reference_index = 0;
        if (version >= 1) {
            if (offset + 2 > size) break;
            uint16_t combined = (data[offset] << 8) | data[offset + 1];
            construction_method = (combined >> 12) & 0xF;  // Upper 4 bits (bits 15-12)
            data_reference_index = combined & 0x0FFF;       // Lower 12 bits
            if (verbose && i < 3) {
                std::cout << "  construction_method: " << construction_method 
                          << ", data_reference_index: " << data_reference_index << "\n";
            }
            offset += 2;
        }
        
        // Read base_offset
        loc.base_offset = readBigEndianVar(data, offset, base_offset_size, size);
        if (verbose && i < 3) {
            std::cout << "  base_offset (" << (int)base_offset_size << " bytes): " << loc.base_offset << "\n";
        }
        
        // Read extent_count
        if (offset + 2 > size) break;
        uint16_t extent_count = (data[offset] << 8) | data[offset + 1];
        if (verbose && i < 3) {
            std::cout << "  extent_count: " << extent_count 
                      << " [" << std::hex << (int)data[offset] << " " << (int)data[offset+1] << std::dec << "]\n";
        }
        offset += 2;
        
        // Sanity check extent count
        if (extent_count == 0) {
            if (verbose && i < 3) {
                std::cout << "  [WARN] extent_count is 0\n";
            }
            locations.push_back(loc);
            continue;
        }
        
        if (extent_count > 1000) {
            std::cout << "  [ERROR] Suspicious extent_count: " << extent_count 
                      << " for item " << loc.item_id << "\n";
            break;
        }
        
        // Read extents
        for (uint16_t j = 0; j < extent_count; ++j) {
            if (offset >= size) {
                std::cout << "  [ERROR] Buffer overrun at extent " << j << "\n";
                break;
            }
            
            // Read extent_index (only if version >= 1 and index_size > 0)
            uint64_t extent_index = 0;
            if (version >= 1 && index_size > 0) {
                extent_index = readBigEndianVar(data, offset, index_size, size);
                if (verbose && i < 3 && j < 5) {
                    std::cout << "    extent[" << j << "] index: " << extent_index << "\n";
                }
            }
            
            // Read extent_offset
            uint64_t extent_offset = readBigEndianVar(data, offset, offset_size, size);
            
            // Read extent_length
            uint64_t extent_length = readBigEndianVar(data, offset, length_size, size);
            
            if (verbose && i < 3 && j < 5) {
                std::cout << "    extent[" << j << "] offset=" << extent_offset 
                          << " (" << (int)offset_size << " bytes), "
                          << "length=" << extent_length 
                          << " (" << (int)length_size << " bytes)\n";
            }
            
            // Sanity checks
            if (extent_length == 0) {
                if (verbose && i < 3 && j < 5) {
                    std::cout << "      [WARN] Zero-length extent\n";
                }
            }
            if (extent_length > 1000000000) {  // 1GB
                std::cout << "      [ERROR] Suspicious extent_length: " << extent_length << "\n";
                continue;
            }
            
            loc.extents.emplace_back(extent_offset, extent_length);
        }
        
        if (verbose && i < 3) {
            uint64_t total_bytes = 0;
            for (const auto& ext : loc.extents) {
                total_bytes += ext.second;
            }
            std::cout << "  Total data for item " << loc.item_id << ": " 
                      << total_bytes << " bytes (" << (total_bytes / 1024.0) << " KB)\n";
            std::cout << "  Item consumed " << (offset - item_start_offset) << " bytes\n";
        }
        
        locations.push_back(loc);
    }
    
    std::cout << "[ILOC] Successfully parsed " << locations.size() << " items\n";
    return locations;
}

// Enhanced iloc parsing with proper multi-extent support
std::vector<HeifItemLocation> parseIlocEnhanced(const uint8_t* data, uint64_t size, bool verbose = false) {
    std::vector<HeifItemLocation> locations;
    
    if (size < 8) {
        if (verbose) std::cout << "  [ILOC] Buffer too small\n";
        return locations;
    }
    
    uint8_t version = data[0];
    uint8_t offset_size = (data[4] >> 4) & 0xF;
    uint8_t length_size = data[4] & 0xF;
    uint8_t base_offset_size = (data[5] >> 4) & 0xF;
    uint8_t index_size = (version == 1 || version == 2) ? (data[5] & 0xF) : 0;
    
    if (verbose) {
        std::cout << "  [ILOC] version=" << (int)version 
                  << ", offset_size=" << (int)offset_size 
                  << ", length_size=" << (int)length_size
                  << ", base_offset_size=" << (int)base_offset_size
                  << ", index_size=" << (int)index_size << "\n";
    }
    
    uint32_t item_count;
    uint64_t offset = 6;
    
    if (version < 2) {
        if (size < 8) return locations;
        item_count = readBigEndian<uint16_t>(data + 6);
        offset = 8;
    } else {
        if (size < 10) return locations;
        item_count = readBigEndian<uint32_t>(data + 6);
        offset = 10;
    }
    
    if (verbose) {
        std::cout << "  [ILOC] item_count=" << item_count << "\n";
    }
    
    // Helper lambda to read variable-size integers
    auto readVarInt = [&](uint8_t field_size) -> uint64_t {
        if (field_size == 0) return 0;
        if (offset + field_size > size) return 0;
        
        uint64_t value = 0;
        for (uint8_t i = 0; i < field_size && i < 8; ++i) {
            value = (value << 8) | data[offset++];
        }
        return value;
    };
    
    for (uint32_t i = 0; i < item_count && i < 10000; ++i) {
        if (offset + 8 > size) break;
        
        HeifItemLocation loc;
        
        // Read item ID (2 or 4 bytes depending on version)
        if (version < 2) {
            loc.item_id = readBigEndian<uint16_t>(data + offset);
            offset += 2;
        } else {
            loc.item_id = readBigEndian<uint32_t>(data + offset);
            offset += 4;
        }
        
        // Read construction method (version 1 and 2 only)
        uint16_t construction_method = 0;
        if (version >= 1) {
            if (offset + 2 > size) break;
            construction_method = readBigEndian<uint16_t>(data + offset) & 0x0F;
            offset += 2;  // Skip construction_method (lower 4 bits) and data_reference_index (16 bits)
        }
        
        // Read base offset
        loc.base_offset = readVarInt(base_offset_size);
        
        // Read extent count
        if (offset + 2 > size) break;
        uint16_t extent_count = readBigEndian<uint16_t>(data + offset);
        offset += 2;
        
        if (verbose && i < 5) {
            std::cout << "    Item " << loc.item_id 
                      << ": base_offset=" << loc.base_offset
                      << ", extent_count=" << extent_count 
                      << ", construction_method=" << construction_method << "\n";
        }
        
        // Sanity check: extent count should be reasonable
        if (extent_count > 10000) {
            if (verbose) {
                std::cout << "    [WARN] Suspicious extent_count=" << extent_count 
                          << " for item " << loc.item_id << ", skipping\n";
            }
            break;
        }
        
        // Read each extent
        for (uint16_t j = 0; j < extent_count; ++j) {
            if (offset + index_size + offset_size + length_size > size) {
                if (verbose) {
                    std::cout << "    [WARN] Buffer overrun at extent " << j << "\n";
                }
                break;
            }
            
            // Read extent index (version 1 and 2 only, when index_size > 0)
            uint64_t extent_index = 0;
            if (version >= 1 && index_size > 0) {
                extent_index = readVarInt(index_size);
            }
            
            // Read extent offset and length
            uint64_t extent_offset = readVarInt(offset_size);
            uint64_t extent_length = readVarInt(length_size);
            
            // Sanity checks
            if (extent_length > 1000000000) {  // 1GB per extent is suspicious
                if (verbose) {
                    std::cout << "    [WARN] Suspicious extent_length=" << extent_length 
                              << " for item " << loc.item_id << ", extent " << j << "\n";
                }
                continue;
            }
            
            loc.extents.emplace_back(extent_offset, extent_length);
            
            if (verbose && i < 5 && j < 3) {
                std::cout << "      Extent " << j 
                          << ": offset=" << extent_offset 
                          << ", length=" << extent_length;
                if (extent_index > 0) {
                    std::cout << ", index=" << extent_index;
                }
                std::cout << "\n";
            }
        }
        
        if (verbose && i < 5 && extent_count > 3) {
            std::cout << "      ... (" << (extent_count - 3) << " more extents)\n";
        }
        
        locations.push_back(loc);
    }
    
    return locations;
}

// Update parseMetaBox to use enhanced iloc parsing
void parseMetaBox(const uint8_t* data, uint64_t size, HeifMetadata& meta) {
    uint64_t offset = 4;  // Skip version and flags
    
    while (offset + 8 < size) {
        HeifBox box = parseBox(data + offset, size - offset, offset);
        
        if (box.size == 0 || box.size > size - offset) break;
        
        if (box.type == "iloc") {
            // Use enhanced iloc parser
            meta.item_locations = parseIlocEnhanced(data + box.data_offset, box.data_size, true);
        } else if (box.type == "iinf") {
            meta.items = parseIinf(data + box.data_offset, box.data_size);
        } else if (box.type == "iprp") {
            parseIprp(data + box.data_offset, box.data_size, meta.items);
        } else if (box.type == "pitm") {
            if (box.data_size >= 4) {
                uint8_t version = data[box.data_offset];
                if (version == 0 && box.data_size >= 6) {
                    meta.primary_item_id = readBigEndian<uint16_t>(data + box.data_offset + 4);
                } else if (box.data_size >= 8) {
                    meta.primary_item_id = readBigEndian<uint32_t>(data + box.data_offset + 4);
                }
            }
        }
        
        offset += box.size;
    }
}

void printMetadataSummary(const HeifMetadata& meta) {
    std::cout << "HEIF Metadata Summary:" << std::endl;
    std::cout << "  Valid: " << (meta.valid ? "Yes" : "No") << std::endl;
    std::cout << "  Total parse time: " << meta.total_parse_time_ms << " ms" << std::endl;
    std::cout << "  Primary item ID: " << meta.primary_item_id << std::endl;
    std::cout << "  Number of items: " << meta.items.size() << std::endl;
    std::cout << "  Number of item locations: " << meta.item_locations.size() << std::endl;
    
    // Print details for each item
    for (const auto& item : meta.items) {
        std::cout << "  Item " << item.item_id << ":" << std::endl;
        std::cout << "    Type: " << item.item_type << std::endl;
        std::cout << "    Dimensions: " << item.width << "x" << item.height << std::endl;
        if (item.is_grid) {
            std::cout << "    Grid: " << item.grid_columns << "x" << item.grid_rows << std::endl;
        }
        std::cout << "    Compression: " << item.compression << std::endl;
    }
    
    // Print location information
    for (const auto& loc : meta.item_locations) {
        std::cout << "  Item " << loc.item_id << " location:" << std::endl;
        std::cout << "    Base offset: " << loc.base_offset << std::endl;
        std::cout << "    Extents: " << loc.extents.size() << std::endl;
        for (const auto& extent : loc.extents) {
            std::cout << "      Offset: " << extent.first << ", Length: " << extent.second << std::endl;
        }
    }
}

// Revised readHeifMetadataComplete with corrected iloc parsing
// Updated readHeifMetadataComplete
HeifMetadata readHeifMetadataComplete(const std::string& filepath, bool verbose = true) {
    HeifMetadata meta;
    meta.valid = false;
    
    std::ifstream file(filepath, std::ios::binary);
    if (!file.is_open()) {
        meta.error_message = "Failed to open file";
        return meta;
    }
    
    file.seekg(0, std::ios::end);
    uint64_t file_size = file.tellg();
    file.seekg(0, std::ios::beg);
    
    std::vector<uint8_t> buffer(file_size);
    file.read(reinterpret_cast<char*>(buffer.data()), file_size);
    file.close();
    
    BenchmarkUtils::Timer timer;
    
    if (verbose) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "READING HEIF METADATA: " << filepath << "\n";
        std::cout << "File size: " << (file_size / 1024.0 / 1024.0) << " MB\n";
        std::cout << std::string(80, '=') << "\n";
    }
    
    uint64_t offset = 0;
    uint64_t metadata_end = 0;
    
    // Parse top-level boxes
    while (offset < file_size) {
        if (offset + 8 > file_size) break;
        
        HeifBox box = parseBox(buffer.data() + offset, file_size - offset, offset);
        
        if (box.size == 0 || box.size > file_size - offset) break;
        
        meta.top_level_boxes.push_back(box);
        
        if (verbose) {
            std::cout << "\nTop-level box: '" << box.type << "' at offset " << offset 
                      << ", size " << box.size << " bytes\n";
        }
        
        if (box.type == "ftyp") {
            meta.ftyp_box = box;
        } else if (box.type == "meta") {
            meta.meta_box = box;
            metadata_end = offset + box.size;
            
            // Parse meta box children
            uint64_t meta_offset = box.data_offset - offset;
            uint64_t meta_data_start = box.data_offset;
            
            // Skip version and flags (4 bytes)
            meta_offset += 4;
            
            while (meta_offset + 8 < box.size) {
                HeifBox child = parseBox(buffer.data() + offset + meta_offset, 
                                        box.size - meta_offset, 
                                        offset + meta_offset);
                
                if (child.size == 0 || meta_offset + child.size > box.size) break;
                
                if (verbose) {
                    std::cout << "  Meta child: '" << child.type << "' at offset " 
                              << (offset + meta_offset) << ", size " << child.size << " bytes\n";
                }
                
                if (child.type == "iloc") {
                    uint64_t iloc_data_offset = child.data_offset - offset;
                    meta.item_locations = parseIlocDebug(buffer.data() + offset + meta_offset + child.header_size, 
                                                        child.data_size, 
                                                        verbose);
                } else if (child.type == "iinf") {
                    meta.items = parseIinf(buffer.data() + child.data_offset, child.data_size);
                } else if (child.type == "pitm") {
                    uint64_t pitm_offset = meta_offset + child.header_size;
                    uint8_t pitm_version = buffer[offset + pitm_offset];
                    if (pitm_version == 0) {
                        meta.primary_item_id = (buffer[offset + pitm_offset + 4] << 8) | 
                                              buffer[offset + pitm_offset + 5];
                    } else {
                        meta.primary_item_id = (buffer[offset + pitm_offset + 4] << 24) |
                                              (buffer[offset + pitm_offset + 5] << 16) |
                                              (buffer[offset + pitm_offset + 6] << 8) |
                                              buffer[offset + pitm_offset + 7];
                    }
                    if (verbose) {
                        std::cout << "    Primary item ID: " << meta.primary_item_id << "\n";
                    }
                }
                
                meta_offset += child.size;
            }
        } else if (box.type == "mdat") {
            meta.mdat_box = box;
            if (metadata_end == 0) {
                metadata_end = offset;
            }
        }
        
        offset += box.size;
    }
    
    meta.total_metadata_bytes = metadata_end;
    meta.total_parse_time_ms = timer.elapsed_ms();
    meta.valid = true;
    
    if (verbose) {
        printMetadataSummary(meta);
    }
    
    return meta;
}

} // namespace FileAnalysis

In [ ]:
namespace FileAnalysis {




// Parse iloc box
std::vector<HeifItemLocation> parseIloc(const uint8_t* data, uint64_t size) {
    std::vector<HeifItemLocation> locations;
    uint8_t version = data[0];
    uint8_t offset_size = (data[4] >> 4) & 0xF;
    uint8_t length_size = data[4] & 0xF;
    uint8_t base_offset_size = (data[5] >> 4) & 0xF;
    
    uint32_t item_count;
    uint64_t offset = 8;
    if (version < 2) {
        item_count = readBigEndian<uint16_t>(data + 6);
    } else {
        item_count = readBigEndian<uint32_t>(data + 6);
        offset = 10;
    }
    
    for (uint32_t i = 0; i < item_count; ++i) {
        HeifItemLocation loc;
        if (version < 2) {
            loc.item_id = readBigEndian<uint16_t>(data + offset);
            offset += 2;
        } else {
            loc.item_id = readBigEndian<uint32_t>(data + offset);
            offset += 4;
        }
        
        if (version >= 1) {
            offset += 2;  // Skip construction_method
        }
        
        if (base_offset_size > 0) {
            loc.base_offset = readBigEndian<uint64_t>(data + offset);
            offset += base_offset_size;
        }
        
        uint16_t extent_count = readBigEndian<uint16_t>(data + offset);
        offset += 2;
        
        for (uint16_t j = 0; j < extent_count; ++j) {
            uint64_t extent_offset = 0;
            uint64_t extent_length = 0;
            
            if (version >= 1) {
                offset += 8;  // Skip extent_index
            }
            
            if (offset_size > 0) {
                extent_offset = readBigEndian<uint64_t>(data + offset);
                offset += offset_size;
            }
            
            if (length_size > 0) {
                extent_length = readBigEndian<uint64_t>(data + offset);
                offset += length_size;
            }
            
            loc.extents.emplace_back(extent_offset, extent_length);
        }
        
        locations.push_back(loc);
    }
    
    return locations;
}

void parseMetaBox(const uint8_t* data, uint64_t size, HeifMetadata& meta) {
    uint64_t offset = 4;  // Skip version and flags
    while (offset < size) {
        HeifBox box = parseBox(data + offset, size - offset, offset);
        
        if (box.type == "iloc") {
            meta.item_locations = parseIloc(data + box.data_offset, box.data_size);
        } else if (box.type == "iinf") {
            meta.items = parseIinf(data + box.data_offset, box.data_size);
        } else if (box.type == "iprp") {
            parseIprp(data + box.data_offset, box.data_size, meta.items);
        } else if (box.type == "pitm") {
            meta.primary_item_id = readBigEndian<uint32_t>(data + box.data_offset + 4);
        }
        
        offset += box.size;
    }
}


// Enhanced HEIF metadata reading with complete tile information
HeifMetadata readHeifMetadataComplete(const std::string& filepath, bool verbose = true) {
    HeifMetadata meta;
    meta.valid = false;
    
    std::ifstream file(filepath, std::ios::binary);
    if (!file.is_open()) {
        meta.error_message = "Failed to open file";
        return meta;
    }
    
    file.seekg(0, std::ios::end);
    uint64_t file_size = file.tellg();
    file.seekg(0, std::ios::beg);
    
    std::vector<uint8_t> buffer(file_size);
    file.read(reinterpret_cast<char*>(buffer.data()), file_size);
    file.close();
    
    BenchmarkUtils::Timer timer;
    
    uint64_t offset = 0;
    while (offset < file_size) {
        HeifBox box = parseBox(buffer.data() + offset, file_size - offset, offset);
        meta.top_level_boxes.push_back(box);
        
        if (box.type == "ftyp") {
            meta.ftyp_box = box;
        } else if (box.type == "meta") {
            meta.meta_box = box;
            parseMetaBox(buffer.data() + box.data_offset, box.data_size, meta);
        } else if (box.type == "mdat") {
            meta.mdat_box = box;
        }
        
        offset += box.size;
    }
    
    meta.total_parse_time_ms = timer.elapsed_ms();
    meta.valid = true;
    
    if (verbose) {
        printMetadataSummary(meta);
    }
    
    return meta;
}

} // namespace FileAnalysis

#### Examples

In [ ]:
// Read and display all COG metadata with detailed access logging
auto cog_meta2 = FileAnalysis::readGeoTiffMetadata("benchmark_output/cog_DEFLATE_t256_p3.tif", true);



In [ ]:
auto heif_meta = FileAnalysis::readHeifMetadataComplete("benchmark_output/heif_DEFLATE_t256_p3.heic", true);

# Post - analysis

## Run Complete Benchmarks

In [ ]:
// Check if datasets are ready
if (!g_config_ready) {
    std::cerr << "ERROR: Configuration not complete!\n";
    std::cerr << "Please run: configureAndStart()\n";
} else if (!g_datasets_ready || g_dataset_pairs.empty()) {
    std::cerr << "ERROR: No datasets available for benchmarking!\n";
    std::cerr << "Please create datasets first.\n";
} else {
    // Run benchmarks on all pairs
    std::vector<BenchmarkUtils::PerformanceMetrics> all_metrics;
    
    std::cout << "\n" << std::string(70, '=') << "\n";
    std::cout << "RUNNING BENCHMARKS\n";
    std::cout << std::string(70, '=') << "\n";
    std::cout << "Dataset pairs: " << g_dataset_pairs.size() << "\n";
    std::cout << "Iterations per test: " << g_benchmark_config.benchmark_iterations << "\n";
    
    // Calculate total tests
    int tests_per_pair = 6;  // full_read, subset_read, metadata, first_byte for both formats
    int total_tests = g_dataset_pairs.size() * tests_per_pair;
    std::cout << "Total tests: " << total_tests << "\n";
    std::cout << std::string(70, '=') << "\n\n";
    
    BenchmarkUtils::ProgressBar progress(total_tests, "Overall Progress", 50, g_benchmark_config.show_progress);
    
    for (size_t i = 0; i < g_dataset_pairs.size(); i++) {
        const auto& pair = g_dataset_pairs[i];
        
        std::cout << "\n[" << (i+1) << "/" << g_dataset_pairs.size() << "] ";
        std::cout << pair.compression << " | Tile:" << pair.tile_size 
                  << " | Pyr:" << pair.pyramid_levels << "\n";
        
        // Full read - COG
       BenchmarkUtils::PerformanceMetrics cog_full = Benchmarker::benchmarkCOGRead(
            pair.cog_path, pair.compression, pair.tile_size, pair.pyramid_levels,
            g_benchmark_config.benchmark_iterations, false);
        all_metrics.push_back(cog_full);
        progress.update();
        
        // Full read - HEIF
        BenchmarkUtils::PerformanceMetrics heif_full = Benchmarker::benchmarkHEIFRead(
            pair.geoheif_path, pair.compression, pair.tile_size, pair.pyramid_levels,
            g_benchmark_config.benchmark_iterations, false);
        all_metrics.push_back(heif_full);
        progress.update();
        
        // Subset read (25% of image, centered)
        int subset_x = cog_full.width / 4;
        int subset_y = cog_full.height / 4;
        int subset_w = cog_full.width / 2;
        int subset_h = cog_full.height / 2;
        
        BenchmarkUtils::PerformanceMetrics cog_subset = Benchmarker::benchmarkCOGRead(
            pair.cog_path, pair.compression, pair.tile_size, pair.pyramid_levels,
            g_benchmark_config.benchmark_iterations, true, subset_x, subset_y, subset_w, subset_h);
        all_metrics.push_back(cog_subset);
        progress.update();
        
        BenchmarkUtils::PerformanceMetrics heif_subset = Benchmarker::benchmarkHEIFRead(
            pair.geoheif_path, pair.compression, pair.tile_size, pair.pyramid_levels,
            g_benchmark_config.benchmark_iterations, true, subset_x, subset_y, subset_w, subset_h);
        all_metrics.push_back(heif_subset);
        progress.update();
        
        // Metadata retrieval
        BenchmarkUtils::PerformanceMetrics cog_metadata = Benchmarker::benchmarkMetadataRetrieval(
            pair.cog_path, "COG", pair.compression, pair.tile_size, pair.pyramid_levels,
            g_benchmark_config.benchmark_iterations);
        all_metrics.push_back(cog_metadata);
        
        BenchmarkUtils::PerformanceMetrics heif_metadata = Benchmarker::benchmarkMetadataRetrieval(
            pair.geoheif_path, "GeoHEIF", pair.compression, pair.tile_size, pair.pyramid_levels,
            g_benchmark_config.benchmark_iterations);
        all_metrics.push_back(heif_metadata);
        progress.update();
        
        // First-byte time (cloud optimization)
        if (g_benchmark_config.enable_cloud_optimization) {
            BenchmarkUtils::PerformanceMetrics cog_first_byte = Benchmarker::benchmarkFirstByte(
                pair.cog_path, "COG", pair.compression, pair.tile_size, pair.pyramid_levels);
            all_metrics.push_back(cog_first_byte);
            
            BenchmarkUtils::PerformanceMetrics heif_first_byte = Benchmarker::benchmarkFirstByte(
                pair.geoheif_path, "GeoHEIF", pair.compression, pair.tile_size, pair.pyramid_levels);
            all_metrics.push_back(heif_first_byte);
        }
        progress.update();
    }
    
    progress.finish();
    
    std::cout << "\n" << std::string(70, '=') << "\n";
    std::cout << "BENCHMARKING COMPLETE\n";
    std::cout << "Total metrics collected: " << all_metrics.size() << "\n";
    std::cout << std::string(70, '=') << "\n\n";
    
    // Export results to CSV
    std::filesystem::path output_path(g_output_directory);
    std::filesystem::path csv_file = output_path / "benchmark_results.csv";
    std::string csv_path = csv_file.string();
    
    Visualization::exportMetricsToCSV(all_metrics, csv_path);
    
    // Create visualizations
    Visualization::createGnuplotScript(csv_path, g_output_directory);
}

## Results Summary and Analysis

In [ ]:
// Display comprehensive results summary
// This cell should be run after benchmarking completes

// Read the CSV file to generate summary
std::filesystem::path csv_path = std::filesystem::path(g_output_directory) / "benchmark_results.csv";

if (std::filesystem::exists(csv_path)) {
    std::cout << "\n" << std::string(80, '=') << "\n";
    std::cout << "BENCHMARK RESULTS SUMMARY\n";
    std::cout << std::string(80, '=') << "\n\n";
    
    // Calculate aggregate statistics from metrics
    double cog_total_read = 0, heif_total_read = 0;
    double cog_total_subset = 0, heif_total_subset = 0;
    double cog_total_metadata = 0, heif_total_metadata = 0;
    double cog_total_first_byte = 0, heif_total_first_byte = 0;
    long cog_total_size = 0, heif_total_size = 0;
    
    int cog_read_count = 0, heif_read_count = 0;
    int cog_subset_count = 0, heif_subset_count = 0;
    int cog_metadata_count = 0, heif_metadata_count = 0;
    int cog_first_byte_count = 0, heif_first_byte_count = 0;
    int cog_file_count = 0, heif_file_count = 0;
    
    // Note: In a real implementation, we would parse the CSV here
    // For now, we'll provide a summary format
    
    std::cout << "📊 PERFORMANCE COMPARISON:\n\n";
    std::cout << "Results have been saved to:\n";
    std::cout << "  CSV: " << csv_path.string() << "\n";
    std::cout << "  Gnuplot script: " << (std::filesystem::path(g_output_directory) / "plot_results.gp").string() << "\n";
    std::cout << "\nTo generate visualizations, run:\n";
    std::cout << "  gnuplot \"" << (std::filesystem::path(g_output_directory) / "plot_results.gp").string() << "\"\n";
    
    std::cout << "\n" << std::string(80, '=') << "\n";
    std::cout << "✅ Benchmark complete! Review the CSV file for detailed metrics.\n";
    std::cout << std::string(80, '=') << "\n";
} else {
    std::cout << "⚠ No results file found. Please run the benchmarking cell first.\n";
}

##  Detailed Post-Benchmark Analysis

In [ ]:
// Detailed Post-Benchmark Analysis
if (g_datasets_ready) {
    std::filesystem::path csv_path = std::filesystem::path(g_output_directory) / "benchmark_results.csv";
    
    if (std::filesystem::exists(csv_path)) {
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "DETAILED BENCHMARK ANALYSIS\n";
        std::cout << std::string(80, '=') << "\n\n";
        
        // Parse CSV and calculate statistics
        std::ifstream csv(csv_path);
        std::string line;
        
        // Skip header
        std::getline(csv, line);
        
        struct Stats {
            double sum = 0;
            double min = std::numeric_limits<double>::max();
            double max = 0;
            int count = 0;
            
            void update(double value) {
                sum += value;
                min = std::min(min, value);
                max = std::max(max, value);
                count++;
            }
            
            double mean() const { return count > 0 ? sum / count : 0; }
        };
        
        Stats cog_read, heif_read, cog_size, heif_size;
        
        while (std::getline(csv, line)) {
            std::istringstream iss(line);
            std::string format, operation;
            double read_time, file_size;
            
            // Simple parsing (format,operation,...,read_time,...)
            std::getline(iss, format, ',');
            std::getline(iss, operation, ',');
            
            // Skip to read_time and file_size columns
            std::string token;
            for (int i = 0; i < 6; i++) std::getline(iss, token, ',');
            
            try {
                if (operation == "full_read") {
                    std::getline(iss, token, ','); // file_size
                    std::getline(iss, token, ','); // skip
                    std::getline(iss, token, ','); // skip
                    std::getline(iss, token, ','); // read_time
                    
                    read_time = std::stod(token);
                    
                    if (format == "COG") {
                        cog_read.update(read_time);
                    } else if (format == "GeoHEIF") {
                        heif_read.update(read_time);
                    }
                }
            } catch (...) {
                // Skip invalid lines
            }
        }
        
        csv.close();
        
        std::cout << "📊 PERFORMANCE SUMMARY:\n\n";
        
        if (cog_read.count > 0 && heif_read.count > 0) {
            std::cout << "Full Read Times (ms):\n";
            std::cout << "  COG:\n";
            std::cout << "    Mean:  " << std::fixed << std::setprecision(2) 
                      << cog_read.mean() << " ms\n";
            std::cout << "    Min:   " << cog_read.min << " ms\n";
            std::cout << "    Max:   " << cog_read.max << " ms\n";
            std::cout << "\n  GeoHEIF:\n";
            std::cout << "    Mean:  " << heif_read.mean() << " ms\n";
            std::cout << "    Min:   " << heif_read.min << " ms\n";
            std::cout << "    Max:   " << heif_read.max << " ms\n";
            std::cout << "\n  Performance:\n";
            
            double speedup = cog_read.mean() / heif_read.mean();
            if (speedup > 1.0) {
                std::cout << "    ✅ GeoHEIF is " << std::fixed << std::setprecision(2)
                          << speedup << "x FASTER than COG\n";
            } else {
                std::cout << "    ⚠️  COG is " << std::fixed << std::setprecision(2)
                          << (1.0/speedup) << "x faster than GeoHEIF\n";
            }
        }
        
        std::cout << "\n" << std::string(80, '=') << "\n";
        std::cout << "For complete analysis, see:\n";
        std::cout << "  • " << csv_path.string() << "\n";
        std::cout << "  • Run gnuplot for visualizations\n";
        std::cout << std::string(80, '=') << "\n";
    }
}

In [ ]:
// Verify COG file compression and structure
void verifyCOG(const std::string& cog_path) {
    std::cout << "\n========================================" << std::endl;
    std::cout << "Verifying COG: " << cog_path << std::endl;
    std::cout << "========================================" << std::endl;
    
    GDALDataset* ds = static_cast<GDALDataset*>(GDALOpen(cog_path.c_str(), GA_ReadOnly));
    if (!ds) {
        std::cerr << "Failed to open: " << cog_path << std::endl;
        return;
    }
    
    // Basic info
    std::cout << "Dimensions: " << ds->GetRasterXSize() << "x" << ds->GetRasterYSize() << std::endl;
    std::cout << "Bands: " << ds->GetRasterCount() << std::endl;
    
    // Get first band
    GDALRasterBand* band = ds->GetRasterBand(1);
    if (band) {
        // Block/tile size
        int block_x, block_y;
        band->GetBlockSize(&block_x, &block_y);
        std::cout << "Block/Tile size: " << block_x << "x" << block_y << std::endl;
        
        // Compression
        const char* compression = band->GetMetadataItem("COMPRESSION", "IMAGE_STRUCTURE");
        std::cout << "Compression: " << (compression ? compression : "NONE") << std::endl;
        
        // Overviews/Pyramids
        int overview_count = band->GetOverviewCount();
        std::cout << "Overview count: " << overview_count << std::endl;
        
        for (int i = 0; i < overview_count; i++) {
            GDALRasterBand* overview = band->GetOverview(i);
            if (overview) {
                std::cout << "  Overview " << i << ": " 
                          << overview->GetXSize() << "x" << overview->GetYSize() << std::endl;
            }
        }
        
        // Interleave
        const char* interleave = band->GetMetadataItem("INTERLEAVE", "IMAGE_STRUCTURE");
        std::cout << "Interleave: " << (interleave ? interleave : "UNKNOWN") << std::endl;
    }
    
    // File size
    long file_size = BenchmarkUtils::getFileSize(cog_path);
    std::cout << "File size: " << (file_size / 1024.0 / 1024.0) << " MB" << std::endl;
    
    // Check if it's a valid COG
    char** papszMetadata = ds->GetMetadata("IMAGE_STRUCTURE");
    if (papszMetadata) {
        std::cout << "\nImage Structure Metadata:" << std::endl;
        for (int i = 0; papszMetadata[i] != nullptr; i++) {
            std::cout << "  " << papszMetadata[i] << std::endl;
        }
    }
    
    GDALClose(ds);
    std::cout << "========================================\n" << std::endl;
}

// Test with your files
std::vector<std::string> cog_files = {
    g_output_directory + "/cog_NONE_t256_p3.tif",
    g_output_directory + "/cog_LZW_t256_p3.tif",
    g_output_directory + "/cog_DEFLATE_t256_p3.tif"
};

for (const auto& cog_file : cog_files) {
    if (std::filesystem::exists(cog_file)) {
        verifyCOG(cog_file);
    }
}

In [ ]:
// Compare compression effectiveness
void compareCompression() {
    std::cout << "\n========================================" << std::endl;
    std::cout << "COMPRESSION EFFECTIVENESS COMPARISON" << std::endl;
    std::cout << "========================================\n" << std::endl;
    
    struct FileInfo {
        std::string format;
        std::string compression;
        long size_bytes;
        double size_mb;
    };
    
    std::vector<FileInfo> files;
    
    // Collect file sizes
    std::vector<std::pair<std::string, std::string>> file_list = {
        {"cog_NONE_t256_p3.tif", "COG/NONE"},
        {"cog_LZW_t256_p3.tif", "COG/LZW"},
        {"cog_DEFLATE_t256_p3.tif", "COG/DEFLATE"},
        {"heif_NONE_t256_p3.heic", "HEIF/NONE"},
        {"heif_LZW_t256_p3.heic", "HEIF/LZW"},
        {"heif_DEFLATE_t256_p3.heic", "HEIF/DEFLATE"}
    };
    
    for (const auto& [filename, label] : file_list) {
        std::string full_path = g_output_directory + "/" + filename;
        if (std::filesystem::exists(full_path)) {
            long size = BenchmarkUtils::getFileSize(full_path);
            auto pos = label.find('/');
            files.push_back({
                label.substr(0, pos),
                label.substr(pos + 1),
                size,
                size / 1024.0 / 1024.0
            });
        }
    }
    
    // Print table
    std::cout << std::setw(10) << "Format" 
              << std::setw(12) << "Compression" 
              << std::setw(15) << "Size (MB)" 
              << std::setw(20) << "vs Uncompressed"
              << std::endl;
    std::cout << std::string(57, '-') << std::endl;
    
    // Find uncompressed sizes
    long cog_none_size = 0, heif_none_size = 0;
    for (const auto& f : files) {
        if (f.format == "COG" && f.compression == "NONE") cog_none_size = f.size_bytes;
        if (f.format == "HEIF" && f.compression == "NONE") heif_none_size = f.size_bytes;
    }
    
    for (const auto& f : files) {
        std::cout << std::setw(10) << f.format
                  << std::setw(12) << f.compression
                  << std::setw(15) << std::fixed << std::setprecision(2) << f.size_mb;
        
        long baseline = (f.format == "COG") ? cog_none_size : heif_none_size;
        if (baseline > 0 && f.compression != "NONE") {
            double ratio = 100.0 * f.size_bytes / baseline;
            std::cout << std::setw(18) << ratio << "%";
        } else {
            std::cout << std::setw(20) << "baseline";
        }
        std::cout << std::endl;
    }
    
    std::cout << "\n========================================" << std::endl;
    std::cout << "FINDINGS:" << std::endl;
    std::cout << "========================================" << std::endl;
    
    // Check if LZW == DEFLATE for HEIF
    auto heif_lzw = std::find_if(files.begin(), files.end(), 
        [](const FileInfo& f) { return f.format == "HEIF" && f.compression == "LZW"; });
    auto heif_deflate = std::find_if(files.begin(), files.end(), 
        [](const FileInfo& f) { return f.format == "HEIF" && f.compression == "DEFLATE"; });
    
    if (heif_lzw != files.end() && heif_deflate != files.end()) {
        if (heif_lzw->size_bytes == heif_deflate->size_bytes) {
            std::cout << "⚠️  HEIF LZW and DEFLATE have IDENTICAL sizes!" << std::endl;
            std::cout << "   This means LZW is being mapped to DEFLATE." << std::endl;
        }
    }
    
    // Check if COG LZW is larger than uncompressed
    auto cog_lzw = std::find_if(files.begin(), files.end(), 
        [](const FileInfo& f) { return f.format == "COG" && f.compression == "LZW"; });
    auto cog_none = std::find_if(files.begin(), files.end(), 
        [](const FileInfo& f) { return f.format == "COG" && f.compression == "NONE"; });
    
    if (cog_lzw != files.end() && cog_none != files.end()) {
        if (cog_lzw->size_bytes > cog_none->size_bytes) {
            std::cout << "⚠️  COG LZW is LARGER than uncompressed!" << std::endl;
            std::cout << "   This is unexpected and suggests an issue." << std::endl;
        }
    }
    
    std::cout << "========================================\n" << std::endl;
}

compareCompression();

In [ ]:
// Enhanced COG verification - checks overview compression too
void verifyCOGDetailed(const std::string& cog_path) {
    std::cout << "\n========================================" << std::endl;
    std::cout << "DETAILED COG VERIFICATION: " << std::filesystem::path(cog_path).filename().string() << std::endl;
    std::cout << "========================================" << std::endl;
    
    GDALDataset* ds = static_cast<GDALDataset*>(GDALOpen(cog_path.c_str(), GA_ReadOnly));
    if (!ds) {
        std::cerr << "Failed to open: " << cog_path << std::endl;
        return;
    }
    
    int width = ds->GetRasterXSize();
    int height = ds->GetRasterYSize();
    int bands = ds->GetRasterCount();
    
    std::cout << "\n📊 BASIC INFO:" << std::endl;
    std::cout << "  Dimensions: " << width << "x" << height << std::endl;
    std::cout << "  Bands: " << bands << std::endl;
    
    // Dataset-level compression (THIS IS THE REAL ONE FOR COG)
    char** papszMetadata = ds->GetMetadata("IMAGE_STRUCTURE");
    std::string dataset_compression = "UNKNOWN";
    if (papszMetadata) {
        for (int i = 0; papszMetadata[i] != nullptr; i++) {
            std::string meta(papszMetadata[i]);
            if (meta.find("COMPRESSION=") == 0) {
                dataset_compression = meta.substr(12);
            }
        }
    }
    std::cout << "  Dataset Compression: " << dataset_compression << std::endl;
    
    // Check first band
    GDALRasterBand* band = ds->GetRasterBand(1);
    if (band) {
        int block_x, block_y;
        band->GetBlockSize(&block_x, &block_y);
        std::cout << "  Tile size: " << block_x << "x" << block_y << std::endl;
        
        // Main image band compression
        const char* band_compression = band->GetMetadataItem("COMPRESSION", "IMAGE_STRUCTURE");
        std::cout << "  Main band compression: " << (band_compression ? band_compression : "NONE") << std::endl;
        
        // Check overviews
        int overview_count = band->GetOverviewCount();
        std::cout << "\n📐 OVERVIEWS: " << overview_count << std::endl;
        
        long main_size = 0;
        long overview_total = 0;
        
        for (int i = 0; i < overview_count; i++) {
            GDALRasterBand* overview = band->GetOverview(i);
            if (overview) {
                int ov_width = overview->GetXSize();
                int ov_height = overview->GetYSize();
                
                // Get overview compression
                const char* ov_compression = overview->GetMetadataItem("COMPRESSION", "IMAGE_STRUCTURE");
                
                // Estimate uncompressed size
                long ov_uncompressed = static_cast<long>(ov_width) * ov_height * bands;
                overview_total += ov_uncompressed;
                
                std::cout << "  Overview " << i << ": " << ov_width << "x" << ov_height 
                          << " | Compression: " << (ov_compression ? ov_compression : "NONE") << std::endl;
            }
        }
        
        // Estimate main image uncompressed size
        main_size = static_cast<long>(width) * height * bands;
        
        std::cout << "\n💾 SIZE ANALYSIS:" << std::endl;
        long file_size = BenchmarkUtils::getFileSize(cog_path);
        std::cout << "  File size: " << std::fixed << std::setprecision(2) 
                  << (file_size / 1024.0 / 1024.0) << " MB" << std::endl;
        
        double main_size_mb = main_size / 1024.0 / 1024.0;
        double overview_size_mb = overview_total / 1024.0 / 1024.0;
        double total_uncompressed_mb = main_size_mb + overview_size_mb;
        
        std::cout << "  Main image (uncompressed): " << main_size_mb << " MB" << std::endl;
        std::cout << "  Overviews (uncompressed): " << overview_size_mb << " MB" << std::endl;
        std::cout << "  Total uncompressed: " << total_uncompressed_mb << " MB" << std::endl;
        
        double compression_ratio = total_uncompressed_mb / (file_size / 1024.0 / 1024.0);
        std::cout << "  Compression ratio: " << compression_ratio << ":1" << std::endl;
        
        // Analysis
        std::cout << "\n🔍 ANALYSIS:" << std::endl;
        if (dataset_compression == "NONE" || dataset_compression == "UNKNOWN") {
            std::cout << "  ⚠️  No compression detected" << std::endl;
        } else {
            if (compression_ratio < 1.2) {
                std::cout << "  ⚠️  Low compression ratio - overviews might not be compressed!" << std::endl;
            } else {
                std::cout << "  ✓ Good compression ratio" << std::endl;
            }
        }
    }
    
    GDALClose(ds);
    std::cout << "========================================\n" << std::endl;
}

// Test your files
std::vector<std::string> test_files = {
    "cog_NONE_t256_p3.tif",
    "cog_LZW_t256_p3.tif",
    "cog_DEFLATE_t256_p3.tif"
};

for (const auto& filename : test_files) {
    std::string full_path = g_output_directory + "/" + filename;
    if (std::filesystem::exists(full_path)) {
        verifyCOGDetailed(full_path);
    }
}

In [ ]:
// Check GDAL overview compression support (FIXED)
void checkGDALOverviewSupport() {
    std::cout << "\n========================================" << std::endl;
    std::cout << "GDAL OVERVIEW COMPRESSION CHECK" << std::endl;
    std::cout << "========================================\n" << std::endl;
    
    std::cout << "GDAL Version: " << GDALVersionInfo("RELEASE_NAME") << std::endl;
    std::cout << "Build info: " << GDALVersionInfo("BUILD_INFO") << std::endl << std::endl;
    
    // Check GTiff driver capabilities
    GDALDriver* driver = GetGDALDriverManager()->GetDriverByName("GTiff");
    if (driver) {
        std::cout << "GTiff Driver:" << std::endl;
        std::cout << "  Short name: " << driver->GetDescription() << std::endl;
        std::cout << "  Long name: " << driver->GetMetadataItem(GDAL_DMD_LONGNAME) << std::endl;
        
        const char* metadata = driver->GetMetadataItem(GDAL_DMD_CREATIONOPTIONLIST);
        if (metadata) {
            std::string meta_str(metadata);
            
            // Check for compression options
            if (meta_str.find("COMPRESS") != std::string::npos) {
                std::cout << "  ✓ COMPRESS option available" << std::endl;
            }
            if (meta_str.find("LZW") != std::string::npos) {
                std::cout << "  ✓ LZW compression supported" << std::endl;
            }
            if (meta_str.find("DEFLATE") != std::string::npos) {
                std::cout << "  ✓ DEFLATE compression supported" << std::endl;
            }
        }
    }
    
    // Check COG driver
    GDALDriver* cog_driver = GetGDALDriverManager()->GetDriverByName("COG");
    if (cog_driver) {
        std::cout << "\nCOG Driver:" << std::endl;
        std::cout << "  ✓ Available" << std::endl;
        std::cout << "  Long name: " << cog_driver->GetMetadataItem(GDAL_DMD_LONGNAME) << std::endl;
        
        const char* cog_options = cog_driver->GetMetadataItem(GDAL_DMD_CREATIONOPTIONLIST);
        if (cog_options) {
            std::string options_str(cog_options);
            if (options_str.find("OVERVIEW") != std::string::npos) {
                std::cout << "  ✓ Overview options available" << std::endl;
            }
        }
    } else {
        std::cout << "\nCOG Driver:" << std::endl;
        std::cout << "  ✗ Not available (GDAL < 3.1)" << std::endl;
    }
    
    // Check important config options
    std::cout << "\nGDAL Config Options:" << std::endl;
    const char* compress_ov = CPLGetConfigOption("COMPRESS_OVERVIEW", nullptr);
    std::cout << "  COMPRESS_OVERVIEW: " << (compress_ov ? compress_ov : "NOT SET") << std::endl;
    
    const char* interleave_ov = CPLGetConfigOption("INTERLEAVE_OVERVIEW", nullptr);
    std::cout << "  INTERLEAVE_OVERVIEW: " << (interleave_ov ? interleave_ov : "NOT SET") << std::endl;
    
    std::cout << "========================================\n" << std::endl;
}

checkGDALOverviewSupport();

In [ ]:
// Check libheif capabilities (FIXED)
void checkLibheifCapabilities() {
    std::cout << "\n========================================" << std::endl;
    std::cout << "LIBHEIF CAPABILITIES CHECK" << std::endl;
    std::cout << "========================================\n" << std::endl;
    
    std::cout << "libheif version: " << heif_get_version() << std::endl;
    
    // Check experimental features
#if HEIF_ENABLE_EXPERIMENTAL_FEATURES
    std::cout << "Experimental features: ✅ ENABLED" << std::endl;
    std::cout << "  - heif_context_add_tiled_image: Available" << std::endl;
    std::cout << "  - heif_image_handle_is_unci_image: Available" << std::endl;
    std::cout << "  - heif_image_handle_is_tiled_image: Available" << std::endl;
#else
    std::cout << "Experimental features: ❌ DISABLED" << std::endl;
#endif
    
    // Check available encoders by trying to get them
    std::cout << "\nAvailable encoders:" << std::endl;
    
    // Create a temporary context to check encoders
    HEIFWrappers::HeifContext ctx;
    if (ctx.get()) {
        // Check HEVC
        HEIFWrappers::HeifEncoder encoder;
        heif_error error = heif_context_get_encoder_for_format(ctx.get(), heif_compression_HEVC, encoder.ptr());
        if (error.code == heif_error_Ok && encoder.get()) {
            const char* name = heif_encoder_get_name(encoder.get());
            std::cout << "  ✓ HEVC (" << (name ? name : "unknown") << ")" << std::endl;
        } else {
            std::cout << "  ✗ HEVC (not available)" << std::endl;
        }
        
        // Check AV1
        encoder.set(nullptr);
        error = heif_context_get_encoder_for_format(ctx.get(), heif_compression_AV1, encoder.ptr());
        if (error.code == heif_error_Ok && encoder.get()) {
            const char* name = heif_encoder_get_name(encoder.get());
            std::cout << "  ✓ AV1 (" << (name ? name : "unknown") << ")" << std::endl;
        } else {
            std::cout << "  ✗ AV1 (not available)" << std::endl;
        }
        
        // Check JPEG
        encoder.set(nullptr);
        error = heif_context_get_encoder_for_format(ctx.get(), heif_compression_JPEG, encoder.ptr());
        if (error.code == heif_error_Ok && encoder.get()) {
            const char* name = heif_encoder_get_name(encoder.get());
            std::cout << "  ✓ JPEG (" << (name ? name : "unknown") << ")" << std::endl;
        } else {
            std::cout << "  ✗ JPEG (not available)" << std::endl;
        }
        
        // Check uncompressed
        encoder.set(nullptr);
        error = heif_context_get_encoder_for_format(ctx.get(), heif_compression_uncompressed, encoder.ptr());
        if (error.code == heif_error_Ok && encoder.get()) {
            const char* name = heif_encoder_get_name(encoder.get());
            std::cout << "  ✓ Uncompressed/UNCI (" << (name ? name : "unknown") << ")" << std::endl;
        } else {
            std::cout << "  ✗ Uncompressed/UNCI (not available)" << std::endl;
        }
    }
    
    // Check UNCI compression support
    std::cout << "\nUNCI compression methods:" << std::endl;
    std::cout << "  • NONE (off)" << std::endl;
    std::cout << "  • DEFLATE (zlib-compatible)" << std::endl;
    std::cout << "  • ZLIB" << std::endl;
    std::cout << "  • BROTLI" << std::endl;
    std::cout << "  (Actual availability depends on libheif build)" << std::endl;
    
    std::cout << "\n========================================\n" << std::endl;
}

checkLibheifCapabilities();

## Summary

### Enhanced Features (v2.0):

1. **✅ Comprehensive Compression Handling**
   - Enhanced HEIF compression configuration
   - Support for unci, x265, and AOM encoders
   - Configurable compression parameters

2. **✅ Progress Indicators**
   - Visual progress bars for long operations
   - Real-time feedback during dataset creation
   - Estimated completion tracking

3. **✅ Dataset Validation**
   - Automatic validation of created files
   - Property extraction and verification
   - Error detection and reporting

4. **✅ Enhanced Benchmarking**
   - Metadata retrieval benchmarks
   - First-byte-time measurements (cloud optimization)
   - Comprehensive performance metrics

5. **✅ Improved Results**
   - Detailed CSV export with all metrics
   - Enhanced gnuplot visualizations
   - Results summary and analysis

### Key Improvements:

- **Memory Safety**: RAII wrappers prevent leaks
- **Cross-Platform**: Linux and macOS support
- **Re-runnable**: Namespace isolation prevents conflicts
- **Configurable**: Preset and custom configurations
- **Validated**: Automatic dataset verification
- **Monitored**: Progress tracking throughout

### Usage Workflow:

1. Configure environment (mamba/conda)
2. Load all notebook cells sequentially
3. Run `configureAndStart()` - select preset
4. Create datasets (with progress bar)
5. Validate datasets (automatic)
6. Run benchmarks (with progress tracking)
7. Review results (CSV and plots)

### Output Files:

- `benchmark_output/cog_*.tif` - COG test files
- `benchmark_output/heif_*.heic` - HEIF test files
- `benchmark_output/benchmark_results.csv` - All metrics
- `benchmark_output/plot_results.gp` - Gnuplot script
- `benchmark_output/benchmark_results.png` - Visualizations

### Comments:

This enhanced benchmarking tool provides comprehensive performance comparison
data to support the evaluation of HEIF/GeoHEIF as a modern alternative to
COG/GeoTIFF for geospatial data distribution and cloud-optimized access.
